In [ ]:
# ════════════════════════════════════════════════════════════════════════════
# ALL MODULE-LEVEL IMPORTS consolidated here (cleanup 2026-06-23). Previously
# scattered across cells 6, 15, 18, 20, 24, 70. Function-LOCAL imports are kept
# where they are on purpose (optional / may-fail deps: yfinance, PIL, google.colab).
# ════════════════════════════════════════════════════════════════════════════
import torch
import pandas as pd
import numpy as np
import os, sys, datetime
import datetime as _dt
import torch.nn.functional as F
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from scipy.stats import norm
from ipynb.fs.defs.Process_SPX import process_all, extract

# ════════════════════════════════════════════════════════════════════════════
# float64 is essential: the recursion multiplies large exp() terms together and
# would lose all precision in float32.
# ════════════════════════════════════════════════════════════════════════════
torch.set_default_dtype(torch.float64)





In [ ]:
# ── Vectorized LVG coefficient recursion (transfer-matrix associative scan) ──
# The two-exponential coefficients satisfy a linear recurrence c_{j+1} = M_j c_j
# (left) / c_{j-1} = N_j c_j (right).  Instead of an O(R) Python loop we form all
# 2x2 transfer matrices at once and take their prefix products with a log-depth
# Hillis-Steele scan -> identical values to the loop (verified to ~1e-15) but
# O(log R) batched matmuls, which slashes both the forward cost and the autograd
# backward retrace.  Returns the UNIT coefficients (lam1=lam2=1), as (R,2) tensors
# that index/iterate exactly like the old list-of-pairs.
def _lvg_mat_scan(M):
    """Inclusive prefix matrix product PP[k] = M[k] @ M[k-1] @ ... @ M[0]."""
    n = M.shape[0]; PP = M; shift = 1
    while shift < n:
        PP = torch.cat([PP[:shift], torch.matmul(PP[shift:], PP[:n - shift])], 0)
        shift *= 2
    return PP

def _lvg_scan_coeffs(R1, R2, nus1_a, sigs1, nus2_a, sigs2):
    """nus1_a = [nus1, S0] (len R1+1); nus2_a = [S0, nus2] (len R2+1).
    Returns (C1u, C2u): unit two-exp coefficients, shapes (R1,2) and (R2,2)."""
    dtype = sigs1.dtype
    # LEFT (forward): c0, then c[j+1] = M[j] c[j]
    dL = nus1_a[1:] - nus1_a[:-1]
    c0 = torch.stack([-torch.exp(-dL[0] / sigs1[0]), torch.exp(dL[0] / sigs1[0])])
    if R1 > 1:
        sj = sigs1[:-1]; sjp = sigs1[1:]; d = dL[1:]; r = sjp / sj
        em = torch.exp(-d / sjp); ep = torch.exp(d / sjp)
        M = torch.stack([torch.stack([0.5 * em * (1 + r), 0.5 * em * (1 - r)], -1),
                         torch.stack([0.5 * ep * (1 - r), 0.5 * ep * (1 + r)], -1)], -2)
        C1u = torch.cat([c0[None], torch.matmul(_lvg_mat_scan(M), c0)], 0)
    else:
        C1u = c0[None]
    # RIGHT (backward): c_last, then c[j-1] = N[j] c[j]
    du = nus2_a[-1] - nus2_a[-2]
    cL = torch.stack([torch.exp(du / sigs2[-1]), -torch.exp(-du / sigs2[-1])])
    if R2 > 1:
        sp = sigs2[:-1]; sj2 = sigs2[1:]; rr = sp / sj2; dist = nus2_a[1:-1] - nus2_a[:-2]
        ep = torch.exp(dist / sp); em = torch.exp(-dist / sp)
        NN = torch.stack([torch.stack([0.5 * ep * (1 + rr), 0.5 * ep * (1 - rr)], -1),
                          torch.stack([0.5 * em * (1 - rr), 0.5 * em * (1 + rr)], -1)], -2)
        PPr = _lvg_mat_scan(torch.flip(NN, [0]))
        C2u = torch.cat([torch.matmul(torch.flip(PPr, [0]), cL), cL[None]], 0)
    else:
        C2u = cL[None]
    return C1u, C2u


# ── Smoothness objective: roughness of u = 1/sigma^2 across the knots ────────
# J penalises how non-smooth the local-variance profile is.  With u_i = 1/sigma_i^2
# over the concatenated wings [sigs1 | sigs2] and nu_i the matching knot locations:
#   normalize=False : J = log( sum_i (du_i)^2 )         -- RAW sum.  Refining the mesh
#                     shrinks each du_i, so J drops even when the underlying curve is
#                     unchanged: "free" improvement bought purely by adding knots.
#   normalize=True  : J = log( sum_i (du_i)^2 / dnu_i ) -- mesh-INVARIANT.  (du_i/dnu_i)
#                     is the slope of u and dnu_i the cell width, so the sum is a
#                     midpoint-rule estimate of integral (du/dK)^2 dK : a fixed roughness
#                     functional that does NOT change as knots are added/moved, so J%
#                     reflects genuine smoothing rather than a mesh artifact.
# Toggle via set_J_normalize; run_and_plot / compare_start_dates expose normalize=.
_J_NORMALIZE = False
def set_J_normalize(flag):
    """Select RAW (False) vs mesh-normalised (True) smoothness objective; see above."""
    global _J_NORMALIZE
    _J_NORMALIZE = bool(flag)


def true_reduction_pct(J_init, J_final):
    """TRUE % reduction in the underlying roughness S = exp(J) (NOT the log-objective J).
    The objective is J = log(S) with S = sum (Δu)^2 [/Δν], so the actual roughness ratio
    is S_final/S_init = exp(J_final - J_init) and the true reduction is
        (1 - exp(J_final - J_init)) * 100  %.
    Contrast the headline 'J % improvement' = (J_init - J_final)/|J_init| * 100, which is a
    reduction of the LOG and so compresses big ratios: e.g. halving the roughness (a 50%
    true reduction) is only J_final - J_init = log(0.5) = -0.69, which as a fraction of a
    small |J_init| can read as a huge log-% — the two numbers are NOT comparable."""
    import math
    try:
        return (1.0 - math.exp(float(J_final) - float(J_init))) * 100.0
    except OverflowError:
        return float("-inf")


# ══ Smoothness objective selector (choose with set_J_objective / obj=1|2) ═════════════
# Both penalise jumps of a TRANSFORM of the per-interval local variance.  Recall the LVG
# identity  V(K)/C''(K) = sigma^2  exactly, so the plotted V/C'' curve IS sigma^2.
# Notation: over the concatenated wings [sigs1 | sigs2] let  v_i = sigma_i^2 = (V/C'')_i
# and  u_i = 1/sigma_i^2 = 1/v_i ; nu_i the knot locations, h_i = nu_{i+1}-nu_i the cell
# widths, Delta the forward first difference, S the roughness SUM that gets log-ed.  The
# `normalize` flag (set_J_normalize) divides each squared difference by h_i, turning S
# from a raw mesh-dependent sum into a mesh-invariant integral estimate.  In BOTH cases
# J = log(S), so the S_final / true-reduction metrics describe pure roughness (no offset).
#
#   obj = 1  ORIGINAL — roughness of u = 1/sigma^2            (DEFAULT)
#       S = sum_i (Delta u_i)^2 [ / h_i ] ,     u_i = 1/sigma_i^2
#       Penalises ABSOLUTE jumps in 1/sigma^2 (the model's natural variable).
#
#   obj = 2  RELATIVE / LOG-JUMP objective — roughness of log(sigma^2) = log(V/C'')
#                                                            (RECOMMENDED — most robust)
#       S = sum_i (Delta log v_i)^2 [ / h_i ] , log v_i = log(sigma_i^2) = log(V/C'')_i
#       Penalises the jumps you see on the LOG-scale V/C'' plot — RELATIVE jumps (to 1st
#       order  Delta log v = Delta v / v), weighted EQUALLY across all sigma^2 magnitudes
#       (scale-invariant).  Because it is not dominated by the large-sigma^2 regions, it
#       does NOT stall on dense / tight-band dates the way an absolute-jump objective does,
#       so it is robust across datasets.  (This was 'obj=3'; the old absolute-sigma^2
#       'obj=2' was removed for being magnitude-biased / non-robust under free nus.)
_J_OBJ = 1
def set_J_objective(obj=1):
    """Select the smoothness objective kind: 1=original (1/sigma^2 jumps), 2=relative
    log(V/C'') jumps (recommended, scale-invariant).  See the header for the formulas."""
    global _J_OBJ
    if int(obj) not in (1, 2):
        raise ValueError(f"obj must be 1 or 2; got {obj!r}")
    _J_OBJ = int(obj)

def calculate_J(R1, R2, S0, theta):
    """
    Returns (J, lam1, lam2, c_v1, c_v2):
      J      : log smoothness functional of u = 1/sigma^2 (smaller = smoother).
               RAW  sum (du)^2  or mesh-normalised  sum (du)^2/dnu  per the module
               flag _J_NORMALIZE (toggle with set_J_normalize; run_and_plot and
               compare_start_dates expose it as normalize=True/False).
      lam1/2 : global scales recovered by matching value + derivative at S0.
      c_v1/2 : per-interval two-exponential coefficients (reused by C_K / C2_K).
    """
    # ── Slice the flat parameter vector into the four groups ────────────────
    idx = 0
    nus1 = theta[idx : idx + R1]; idx += R1
    sigs1 = theta[idx : idx + R1]; idx += R1
    nus2 = theta[idx : idx + R2]; idx += R2
    sigs2 = theta[idx : idx + R2]; idx += R2

    # Append/prepend S0 so each side has a closing node at the spot
    nus1c = torch.cat([nus1, torch.tensor([S0])])
    nus2c = torch.cat([torch.tensor([S0]), nus2])

    # ── Unit base coefficients via vectorized transfer-matrix scan ──────────
    # (replaces the O(R) Python forward/backward recursion; identical to ~1e-15)
    c_v1_unit, c_v2_unit = _lvg_scan_coeffs(R1, R2, nus1c, sigs1, nus2c, sigs2)

    # Match value + derivative at S0 to recover the two global scales lam1, lam2
    v1    = c_v1_unit[-1][0] + c_v1_unit[-1][1]
    v2    = c_v2_unit[0][0]  + c_v2_unit[0][1]
    Dk_v1 = (1/sigs1[-1]) * (-c_v1_unit[-1][0] + c_v1_unit[-1][1])
    Dk_v2 = (1/sigs2[0])  * (-c_v2_unit[0][0]  + c_v2_unit[0][1])
    lam1  = v2 / (Dk_v1 * v2 - v1 * Dk_v2)
    lam2  = v1 / (Dk_v1 * v2 - v1 * Dk_v2)

    c_v1 = [(c1 * lam1, c2 * lam1) for (c1, c2) in c_v1_unit]
    c_v2 = [(c1 * lam2, c2 * lam2) for (c1, c2) in c_v2_unit]

    # ── Smoothness objective on u = 1/sigma^2 over all intervals ────────────
    # Differentiable in torch -> the AL L-BFGS gets exact gradients via autograd.
    # No C'' jumps / price weights are needed here, so the old per-knot jump loop
    # (its ~80ms backward was ~70% of the AL closure) is gone entirely.
    # obj kind (1=original | 2=+spacing-reg | 3=relative) is chosen by set_J_objective.
    sigs = torch.cat([sigs1, sigs2])
    nus  = torch.cat([nus1, nus2])                       # interval knot locations
    var  = sigs**2                                       # v = sigma^2 = V/C''
    dnu  = torch.diff(nus).abs().clamp_min(1e-8)         # cell widths h_i (floored)

    if _J_OBJ == 2:
        d = torch.diff(torch.log(var))                   # Delta log(sigma^2) -- relative jumps
    else:
        d = torch.diff(1.0 / var)                        # Delta(1/sigma^2) -- original (obj 1)

    S = torch.sum(d**2 / dnu) if _J_NORMALIZE else torch.sum(d**2)
    J = torch.log(S)

    return J, lam1, lam2, c_v1, c_v2


In [ ]:
# ── Price evaluation: C(K) at one strike, and over a list of strikes ────────
# Given the coefficients (c_v1, c_v2) returned by calculate_J, these evaluate the
# fitted call-price curve. C_K is the single-strike form (kept for reference);
# generate_call_prices is VECTORISED over all strikes for speed.

def C_K(K, R1, R2, S0, theta, c_v1, c_v2):
    idx = 0
    nus1 = theta[idx : idx + R1]; idx += R1
    sigs1 = theta[idx : idx + R1]; idx += R1
    nus2 = theta[idx : idx + R2]; idx += R2
    sigs2 = theta[idx : idx + R2]; idx += R2

    K_val = K if isinstance(K, (int, float)) else K.item()

    if K_val <= S0:
        S0_tensor = torch.tensor([S0], dtype=nus1.dtype, device=nus1.device)
        nus1_full = torch.cat([nus1, S0_tensor])
        index = torch.bucketize(torch.tensor(K_val, device=nus1.device), nus1_full, right=False) - 1
        index = torch.clamp(index, 0, R1 - 1).item()
        dist  = nus1_full[index + 1] - K_val
        V_K   = (c_v1[index][0] * torch.exp(dist / sigs1[index]) +
                 c_v1[index][1] * torch.exp(-dist / sigs1[index]))
        return V_K + (S0 - K_val)          # add intrinsic value for K <= S0
    else:
        S0_tensor = torch.tensor([S0], dtype=nus2.dtype, device=nus2.device)
        nus2_full = torch.cat([S0_tensor, nus2])
        index = torch.bucketize(torch.tensor(K_val, device=nus2.device), nus2_full, right=False) - 1
        index = torch.clamp(index, 0, R2 - 1).item()
        dist  = K_val - nus2_full[index]
        V_K   = (c_v2[index][0] * torch.exp(-dist / sigs2[index]) +
                 c_v2[index][1] * torch.exp(dist / sigs2[index]))
        return V_K

def generate_call_prices(market_strikes, R1, R2, S0, theta, c_v1, c_v2):
    """VECTORISED C(K) over all strikes at once (was a Python loop over C_K).
    Bit-for-bit identical to stacking C_K, but evaluates every strike in a
    handful of tensor ops instead of one Python call per strike, and keeps full
    autograd through theta and the coefficients c_v1/c_v2."""
    idx = 0
    nus1 = theta[idx:idx+R1]; idx += R1
    sigs1 = theta[idx:idx+R1]; idx += R1
    nus2 = theta[idx:idx+R2]; idx += R2
    sigs2 = theta[idx:idx+R2]; idx += R2

    K = market_strikes if torch.is_tensor(market_strikes) else \
        torch.tensor(market_strikes, dtype=nus1.dtype, device=nus1.device)
    K = K.to(dtype=nus1.dtype, device=nus1.device)
    S0_t = torch.tensor([S0], dtype=nus1.dtype, device=nus1.device)

    # Stack the per-interval coefficient tuples into tensors (grad-preserving).
    c10 = torch.stack([c[0] for c in c_v1]); c11 = torch.stack([c[1] for c in c_v1])
    c20 = torch.stack([c[0] for c in c_v2]); c21 = torch.stack([c[1] for c in c_v2])

    out  = torch.zeros_like(K)
    left = K <= S0                       # same split rule as scalar C_K (K_val <= S0)
    right = ~left

    if bool(left.any()):
        Kl = K[left]
        nus1_full = torch.cat([nus1, S0_t])
        il = torch.clamp(torch.bucketize(Kl, nus1_full, right=False) - 1, 0, R1 - 1)
        dl = nus1_full[il + 1] - Kl
        Vl = (c10[il] * torch.exp(dl / sigs1[il]) +
              c11[il] * torch.exp(-dl / sigs1[il]) + (S0 - Kl))   # + intrinsic
        out = out.index_put((left.nonzero(as_tuple=True)[0],), Vl)

    if bool(right.any()):
        Kr = K[right]
        nus2_full = torch.cat([S0_t, nus2])
        ir = torch.clamp(torch.bucketize(Kr, nus2_full, right=False) - 1, 0, R2 - 1)
        dr = Kr - nus2_full[ir]
        Vr = (c20[ir] * torch.exp(-dr / sigs2[ir]) +
              c21[ir] * torch.exp(dr / sigs2[ir]))
        out = out.index_put((right.nonzero(as_tuple=True)[0],), Vr)

    return out

In [ ]:
# ── Analytical second derivative C''(K) ─────────────────────────────────────
# C''(K) is (up to discounting) the risk-neutral density. We use the exact
# analytical form (not finite differences) for the log C''(K) plot, because the
# analytical value is smooth within each interval and strictly positive for a
# well-behaved fit — which is exactly what makes log C''(K) meaningful.

def C2_K(K, R1, R2, S0, theta, c_v1, c_v2):
    idx = 0
    nus1 = theta[idx : idx + R1]; idx += R1
    sigs1 = theta[idx : idx + R1]; idx += R1
    nus2 = theta[idx : idx + R2]; idx += R2
    sigs2 = theta[idx : idx + R2]; idx += R2

    K_val = K if isinstance(K, (int, float)) else K.item()

    if K_val <= S0:
        S0_tensor = torch.tensor([S0], dtype=nus1.dtype, device=nus1.device)
        nus1_full = torch.cat([nus1, S0_tensor])
        index = torch.bucketize(torch.tensor(K_val, device=nus1.device), nus1_full, right=False) - 1
        index = torch.clamp(index, 0, R1 - 1).item()
        dist  = nus1_full[index + 1] - K_val
        return (1.0 / (sigs1[index] ** 2)) * (c_v1[index][0] * torch.exp(dist / sigs1[index]) +
                                              c_v1[index][1] * torch.exp(-dist / sigs1[index]))
    else:
        S0_tensor = torch.tensor([S0], dtype=nus2.dtype, device=nus2.device)
        nus2_full = torch.cat([S0_tensor, nus2])
        index = torch.bucketize(torch.tensor(K_val, device=nus2.device), nus2_full, right=False) - 1
        index = torch.clamp(index, 0, R2 - 1).item()
        dist  = K_val - nus2_full[index]
        return (1.0 / (sigs2[index] ** 2)) * (c_v2[index][0] * torch.exp(-dist / sigs2[index]) +
                                              c_v2[index][1] * torch.exp(dist / sigs2[index]))

def generate_C2K(market_strikes, R1, R2, S0, theta, c_v1, c_v2):
    return torch.stack([C2_K(K, R1, R2, S0, theta, c_v1, c_v2) for K in market_strikes])

In [ ]:
# ── Accessors, bid/ask matrix builder, and the validity checker ─────────────

def obtain_nus(R1, R2, theta):
    nus1 = theta[0 : R1]
    nus2 = theta[2 * R1 : 2 * R1 + R2]
    return nus1, nus2

def obtain_sigs(R1, R2, theta):
    sigs1 = theta[R1 : 2 * R1]
    sigs2 = theta[2 * R1 + R2 : 2 * R1 + 2 * R2]
    return sigs1, sigs2

def obtain_theta(R1, R2, S0, sigs1, sigs2, nus1, nus2):
    # nus do NOT include S0; theta layout = [nus1, sigs1, nus2, sigs2]
    return torch.cat([nus1, sigs1, nus2, sigs2])

# ===== UNUSED (no references found anywhere in notebook, 2026-06-23) — commented out for review; safe to delete =====
# def generate_bid_ask_matrix(R1, R2, S0, theta, eps_bid_vector, eps_ask_vector):
#     """Build an (N x 3) [strike, bid, ask] band around the model's own prices.
#     Handy for constructing self-contained synthetic test problems."""
#     nus1, nus2 = obtain_nus(R1, R2, theta)
#     S0_tensor  = torch.tensor([S0], dtype=nus1.dtype, device=nus1.device)
#     strikes    = torch.cat([nus1, S0_tensor, nus2])
#     _, _, _, c_v1, c_v2 = calculate_J(R1, R2, S0, theta)
#     model_prices = generate_call_prices(strikes, R1, R2, S0, theta, c_v1, c_v2)
#     bid_ask = torch.zeros((R1 + R2 + 1, 3), dtype=theta.dtype, device=theta.device)
#     bid_ask[:, 0] = strikes
#     bid_ask[:, 1] = torch.clamp(model_prices - eps_bid_vector, min=0.0)
#     bid_ask[:, 2] = model_prices + eps_ask_vector
#     return bid_ask


def check_constraints(R1, R2, S0, sigs1, sigs2, nus1, nus2, bid_ask, market_strikes):
    """
    Returns True iff ALL three constraint groups hold. This is the single
    source of truth the optimiser's output must satisfy.
    """
    # 1. Strictly increasing knots + S0 boundaries
    increasing_nus1 = torch.all(torch.diff(nus1) > 0) and (nus1[-1] <= S0)
    increasing_nus2 = torch.all(torch.diff(nus2) > 0) and (nus2[0] >= S0)

    # 2. Strictly positive sigmas
    positive_sigs1 = torch.all(sigs1 > 0)
    positive_sigs2 = torch.all(sigs2 > 0)

    # 3. Inside the bid/ask band at every quoted strike
    market_bids = bid_ask[:, 1]
    market_asks = bid_ask[:, 2]
    theta = obtain_theta(R1, R2, S0, sigs1, sigs2, nus1, nus2)
    _, _, _, c_v1, c_v2 = calculate_J(R1, R2, S0, theta)
    model_prices = generate_call_prices(market_strikes, R1, R2, S0, theta, c_v1, c_v2)
    bid_check = torch.all(model_prices >= market_bids)
    ask_check = torch.all(model_prices <= market_asks)

    is_valid = (increasing_nus1 and increasing_nus2 and
                positive_sigs1 and positive_sigs2 and
                bid_check and ask_check)
    return is_valid.item() if hasattr(is_valid, "item") else bool(is_valid)

In [ ]:

# ════════════════════════════════════════════════════════════════════════════
# REPARAMETERISATION
# Optimise an unconstrained "raw" vector z; map it to a constraint-satisfying
# theta. Ordering / S0-boundaries / sigma>0 then hold for ANY z, with no penalty.
# ════════════════════════════════════════════════════════════════════════════

EPS_REPARAM = 1e-8   # tiny floor so every gap / sigma is strictly (not weakly) positive
NU_FLOOR    = 1e-6   # strict lower floor for the deepest knot:  nus1 > NU_FLOOR > 0

def _inv_softplus(y):
    """Stable inverse of softplus for y > 0:  s^{-1}(y) = y + log(1 - e^{-y}).
    The naive log(expm1(y)) overflows for large y (e.g. a ~1000-wide gap)."""
    return y + torch.log1p(-torch.exp(-y))


def build_theta_from_raw(raw, R1, R2, S0, eps=EPS_REPARAM):
    """
    Map unconstrained raw -> valid theta = [nus1, sigs1, nus2, sigs2].
    Guarantees (for ANY raw):
      * sigs1, sigs2 > 0
      * nus1 strictly increasing, 0 < nus1 and nus1[-1] < S0
      * nus2 strictly increasing, nus2[0]  > S0
    Fully differentiable (softplus, cumsum, flip all support autograd), so
    gradients of J flow back to raw and L-BFGS can optimise in raw-space.
    """
    i = 0
    raw_nus1  = raw[i:i+R1]; i += R1
    raw_sigs1 = raw[i:i+R1]; i += R1
    raw_nus2  = raw[i:i+R2]; i += R2
    raw_sigs2 = raw[i:i+R2]; i += R2

    # Positive scales
    sigs1 = F.softplus(raw_sigs1) + eps
    sigs2 = F.softplus(raw_sigs2) + eps

    # Positive gaps
    g1 = F.softplus(raw_nus1) + eps          # R1 left gaps
    g2 = F.softplus(raw_nus2) + eps          # R2 right gaps

    # Left grid: build DOWNWARD from S0 using suffix sums of the gaps, but BOUND
    # the total left span so the deepest knot stays strictly positive (nus1 > 0).
    #   used = (S0-NU_FLOOR)*tanh( sum(g1) / (S0-NU_FLOOR) )  in (0, S0-NU_FLOOR)
    #   nus1[0] = S0 - used  >  NU_FLOOR > 0   (and still strictly increasing, < S0)
    cap  = S0 - NU_FLOOR
    tot1 = g1.sum()
    used = cap * torch.tanh(tot1 / cap)      # squashed total < cap
    g1   = g1 * (used / tot1)                 # rescale gaps (proportions preserved)
    rev_suffix_sum = torch.flip(torch.cumsum(torch.flip(g1, [0]), 0), [0])
    nus1 = S0 - rev_suffix_sum

    # Right grid: build UPWARD from S0 using prefix sums of the gaps.
    #   nus2[k] = S0 + sum_{j<=k} g2[j]   (strictly increasing, first > S0)
    nus2 = S0 + torch.cumsum(g2, 0)

    return torch.cat([nus1, sigs1, nus2, sigs2])


def raw_from_theta(R1, R2, S0, nus1, sigs1, nus2, sigs2, eps=EPS_REPARAM):
    """
    Inverse map: a FEASIBLE theta -> the raw vector that reproduces it.
    Used once, to convert a known-good starting point into raw-space.
    Inputs must satisfy the ordering/boundary/positivity constraints.
    """
    # Left gaps: interior diffs, plus the closing gap to S0 at the top.
    g1 = torch.cat([nus1[1:] - nus1[:-1], (S0 - nus1[-1]).reshape(1)])
    # Right gaps: the opening gap from S0, plus interior diffs.
    g2 = torch.cat([(nus2[0] - S0).reshape(1), nus2[1:] - nus2[:-1]])

    # Left: invert the tanh-bounded total span used in build_theta_from_raw.
    #   sum(g1) = S0 - nus1[0] = used ;  recover the pre-squash total and gaps.
    cap         = S0 - NU_FLOOR
    used_target = torch.clamp(g1.sum(), max=cap * (1 - 1e-9))   # < cap  (nus1[0] > NU_FLOOR)
    tot_raw     = cap * torch.atanh(used_target / cap)
    g1_raw      = g1 * (tot_raw / g1.sum())                     # un-squash to pre-scale gaps

    # Clamp away from 0 so inv_softplus is finite, then invert softplus(.)+eps.
    raw_nus1  = _inv_softplus(torch.clamp(g1_raw - eps, min=1e-12))
    raw_nus2  = _inv_softplus(torch.clamp(g2     - eps, min=1e-12))
    raw_sigs1 = _inv_softplus(torch.clamp(sigs1 - eps, min=1e-12))
    raw_sigs2 = _inv_softplus(torch.clamp(sigs2 - eps, min=1e-12))
    return torch.cat([raw_nus1, raw_sigs1, raw_nus2, raw_sigs2])


# Quick self-test: the round trip must be exact and the output must be valid.
# ===== UNUSED (no references found anywhere in notebook, 2026-06-23) — commented out for review; safe to delete =====
# def _reparam_self_test(R1, R2, S0, nus1, sigs1, nus2, sigs2):
#     raw   = raw_from_theta(R1, R2, S0, nus1, sigs1, nus2, sigs2)
#     theta = build_theta_from_raw(raw, R1, R2, S0)
#     n1, n2 = obtain_nus(R1, R2, theta)
#     s1, s2 = obtain_sigs(R1, R2, theta)
#     err = (theta - obtain_theta(R1, R2, S0, sigs1, sigs2, nus1, nus2)).abs().max().item()
#     ok  = (torch.all(n1 > 0) and torch.all(torch.diff(n1) > 0) and n1[-1] < S0 and
#            torch.all(torch.diff(n2) > 0) and n2[0] > S0 and
#            torch.all(s1 > 0) and torch.all(s2 > 0))
#     print(f"[reparam self-test] round-trip max-abs-err = {err:.2e} | constraints hold = {bool(ok)}")
#     return bool(ok) and err < 1e-6

# ── Sigma-only reparametrisation: nus are FIXED, only sigmas are optimised ──
# BUDGET reparam: rather than flooring each sigma (which over-smooths and blocks
# the sharp local bends needed to thread the band), we bound only the CUMULATIVE
# rate  S = sum_j gap_j / sigma_j  per side via a tanh squash to a cap.  Then the
# recursion growth exp(S) <= exp(cap) for ANY raw vector -> no overflow ever, yet
# an individual sigma_j may be arbitrarily small (sharp curvature) as long as the
# budget holds.  gap_j are the FIXED knot gaps (incl. the closing gap to S0), in
# exactly the order calculate_J traverses them.

EPS_RATE        = 1e-30    # floor on each per-interval rate; must be BELOW the
                           # smallest real rate gap/sigma so a near-zero knot gap
                           # (collapsed knot, e.g. nus2[0]->S0) is represented
                           # exactly instead of floored into a spurious tiny sigma
SIG_BUDGET_CAP  = 250.0    # max cumulative sum_j gap_j/sigma_j per side
                           # exp(250)~3.7e108; two-sided product ~1.4e217 << 1.8e308

def _sigonly_gaps(nus1_t, nus2_t, S0):
    """Fixed per-interval gaps in calculate_J's traversal order.
    Left  : [diff(nus1), S0 - nus1[-1]]   (R1 gaps, closing at S0)
    Right : [nus2[0] - S0, diff(nus2)]    (R2 gaps, opening from S0)"""
    gaps1 = torch.cat([nus1_t[1:] - nus1_t[:-1], (S0 - nus1_t[-1]).reshape(1)])
    gaps2 = torch.cat([(nus2_t[0] - S0).reshape(1), nus2_t[1:] - nus2_t[:-1]])
    return gaps1, gaps2

def _sigmas_from_raw_budget(raw, gaps, cap, eps_rate=EPS_RATE):
    """raw -> sigmas with sum(gap/sigma) < cap guaranteed (tanh-squashed budget)."""
    r    = F.softplus(raw) + eps_rate              # demanded per-interval rate > 0
    T    = r.sum()
    rate = r * (cap * torch.tanh(T / cap) / T)     # sum(rate) = cap*tanh(T/cap) < cap
    return gaps / rate                             # sigma_j = gap_j / rate_j

def _raw_from_sigmas_budget(sigmas, gaps, cap, eps_rate=EPS_RATE):
    """Inverse of _sigmas_from_raw_budget.  Requires sum(gap/sigma) < cap."""
    rate   = gaps / sigmas
    T_post = rate.sum()
    T      = cap * torch.atanh(torch.clamp(T_post / cap, max=1 - 1e-12))
    r      = rate * (T / T_post)
    return _inv_softplus(torch.clamp(r - eps_rate, min=1e-300))


def raw_sigonly_from_sigs(sigs1, sigs2, gaps1, gaps2,
                          cap_left=SIG_BUDGET_CAP, cap_right=SIG_BUDGET_CAP):
    """Map positive sigmas -> unconstrained raw (R1+R2 elements), budget-consistent.
    The caps/gaps must match build_theta_sigonly_from_raw so the round-trip is exact."""
    raw1 = _raw_from_sigmas_budget(sigs1, gaps1, cap_left)
    raw2 = _raw_from_sigmas_budget(sigs2, gaps2, cap_right)
    return torch.cat([raw1, raw2])


def build_theta_sigonly_from_raw(raw_sigs, R1, R2, nus1_t, nus2_t, S0,
                                 cap_left=SIG_BUDGET_CAP, cap_right=SIG_BUDGET_CAP):
    """Map unconstrained raw_sigs (R1+R2) -> theta = [nus1_t, sigs1, nus2_t, sigs2]
    with FIXED nus and budget-bounded sigmas.

    For ANY raw, sum_j gap_j/sigma_j < cap on each side, so the calculate_J
    recursion (and the Wronskian) stays finite throughout optimization -- while
    individual sigmas remain free to get very small for sharp local curvature.
    Fully differentiable (softplus, tanh, sum), so gradients of J flow to raw."""
    gaps1, gaps2 = _sigonly_gaps(nus1_t, nus2_t, S0)
    sigs1 = _sigmas_from_raw_budget(raw_sigs[:R1], gaps1, cap_left)
    sigs2 = _sigmas_from_raw_budget(raw_sigs[R1:], gaps2, cap_right)
    return torch.cat([nus1_t, sigs1, nus2_t, sigs2])

# ── Sigma + L + Kbar reparametrisation ──────────────────────────────────────
# Like the sigma-only budget reparam, but the TWO extreme boundary knots are
# also free: L = nus1[0] (deepest left knot) and Kbar = nus2[-1] (furthest right
# knot).  ALL interior nus stay FIXED.  L and Kbar are squashed into open
# intervals via a sigmoid so ordering (0 <= L < nus1[1] and nus2[-2] < Kbar)
# holds for ANY raw, and the budget cap on sum_j gap_j/sigma_j keeps the
# calculate_J recursion finite.  raw layout: [raw_sigs (R1+R2), raw_L, raw_Kbar].

def _bounded(raw, lo, hi):
    """Map an unconstrained scalar raw -> (lo, hi) via a sigmoid (differentiable)."""
    return lo + (hi - lo) * torch.sigmoid(raw)

def _bounded_inv(val, lo, hi, eps=1e-6):
    """Inverse of _bounded.  frac is clipped to (eps, 1-eps) so a value sitting
    exactly on a bound (e.g. L_init=0 at lo=0) still inverts to a finite raw;
    the warm start is then exact to ~eps*(hi-lo) (negligible)."""
    import math
    frac = (float(val) - lo) / (hi - lo)
    frac = min(max(frac, eps), 1.0 - eps)
    return torch.tensor(math.log(frac / (1.0 - frac)), dtype=torch.float64)

def sigonly_LKbar_bounds(nus1_t, nus2_t, S0, kbar_up_factor=2.0, edge_frac=1e-3):
    """Open intervals for the movable boundary knots L and Kbar.
      L    in (0,            nus1[1] - margin)                  (kept >= 0, < nus1[1])
      Kbar in (nus2[-2]+margin,  S0 + kbar_up_factor*(Kbar_init - S0))
    margins are a tiny fraction (edge_frac) of the adjacent gap so ordering is strict."""
    n1 = nus1_t.detach(); n2 = nus2_t.detach()
    Kbar_init = float(n2[-1])
    m1 = edge_frac * float(n1[1] - n1[0] + 1e-12)
    m2 = edge_frac * float(n2[-1] - n2[-2] + 1e-12)
    lo_L, hi_L = 0.0, float(n1[1]) - m1
    lo_K = float(n2[-2]) + m2
    hi_K = float(S0) + kbar_up_factor * (Kbar_init - float(S0))
    return lo_L, hi_L, lo_K, hi_K

def build_theta_sigonly_LKbar_from_raw(raw, R1, R2, nus1_fixed, nus2_fixed, S0,
                                       lo_L, hi_L, lo_K, hi_K,
                                       cap_left=SIG_BUDGET_CAP, cap_right=SIG_BUDGET_CAP):
    """Map raw=[raw_sigs(R1+R2), raw_L, raw_Kbar] -> theta=[nus1,sigs1,nus2,sigs2]
    with interior nus FIXED, L=nus1[0] and Kbar=nus2[-1] free (sigmoid-bounded),
    and budget-bounded sigmas.  The boundary gaps (gaps1[0], gaps2[-1]) move with
    L/Kbar, so sigmas there rescale accordingly.  Fully differentiable, so
    gradients of J reach raw_sigs, raw_L and raw_Kbar alike."""
    raw_sigs = raw[:R1 + R2]
    L    = _bounded(raw[R1 + R2],     lo_L, hi_L)
    Kbar = _bounded(raw[R1 + R2 + 1], lo_K, hi_K)
    nus1 = torch.cat([L.reshape(1),     nus1_fixed[1:]])
    nus2 = torch.cat([nus2_fixed[:-1],  Kbar.reshape(1)])
    gaps1, gaps2 = _sigonly_gaps(nus1, nus2, S0)
    sigs1 = _sigmas_from_raw_budget(raw_sigs[:R1], gaps1, cap_left)
    sigs2 = _sigmas_from_raw_budget(raw_sigs[R1:], gaps2, cap_right)
    return torch.cat([nus1, sigs1, nus2, sigs2])


In [ ]:
# ════════════════════════════════════════════════════════════════════════════
# AUGMENTED-LAGRANGIAN INNER SOLVE  (the work of one outer iteration)
# ----------------------------------------------------------------------------
# GOAL: minimise the smoothness objective J(theta) SUBJECT TO every fitted call
# price staying inside the quoted bid/ask band:   bid_k <= C_k(theta) <= ask_k.
#
# Per-strike band violations (zero when the price is inside the band):
#       g_bid_k = max(bid_k - C_k, 0)      # amount the price sits BELOW the bid
#       g_ask_k = max(C_k - ask_k, 0)      # amount the price sits ABOVE the ask
#
# The Augmented Lagrangian replaces the hard constraint with an UNCONSTRAINED
# objective combining a linear (multiplier) term and a quadratic (penalty) term:
#
#   L_rho(theta; lam) = J(theta)
#                     + sum_k [ lam_bid_k * g_bid_k + lam_ask_k * g_ask_k ]   (linear)
#                     + (rho/2) * sum_k [ g_bid_k^2 + g_ask_k^2 ]             (quadratic)
#
#   - the LINEAR term carries Lagrange multipliers lam>=0 that approximate the true
#     constraint forces, so the band can be met EXACTLY at a FINITE rho (a pure
#     penalty method would need rho -> infinity, which is badly conditioned);
#   - the QUADRATIC term (penalty rho) convexifies/conditions the subproblem and
#     drives down whatever violation the multipliers have not yet removed.
#
# ONE OUTER ITERATION (this file does (1) and (2); the next cell does (3)):
#   (1) INNER SOLVE  — minimise L_rho over the parameters with L-BFGS;
#   (2) DUAL ASCENT  — lam_bid <- max(lam_bid + rho*g_bid, 0),  likewise lam_ask;
#   (3) PENALTY UPDATE — the outer loop raises rho if the band is still violated.
#
# WHY OPTIMISE `raw` AND NOT theta DIRECTLY:  we optimise an UNCONSTRAINED vector
# `raw` and map raw -> theta with reparam_fn (default build_theta_from_raw), which
# BAKES IN ordering (nus increasing), boundaries (0 < nus1 < S0 < nus2) and
# positivity (sigs > 0) for ANY raw whatsoever.  So L-BFGS can roam all of R^n with
# no constraints of its own, and the ONLY constraint the AL must handle is the
# bid/ask band.  (Pass a sigma-only reparam_fn to freeze the nus and move sigmas.)
# ════════════════════════════════════════════════════════════════════════════

def band_violation(raw, R1, R2, S0, bid_ask, loss_fn, generate_call_prices,
                   reparam_fn=None):
    """Max bid/ask violation (0 => inside the band).  All ordering/positivity
    constraints are guaranteed by reparam_fn, so this is the only check needed."""
    if reparam_fn is None:
        reparam_fn = lambda r: build_theta_from_raw(r, R1, R2, S0)
    market_strikes = bid_ask[:, 0]; market_bids = bid_ask[:, 1]; market_asks = bid_ask[:, 2]
    with torch.no_grad():
        theta = reparam_fn(raw)
        J, _, _, c1, c2 = loss_fn(R1, R2, S0, theta)
        if torch.isnan(J) or torch.isinf(J):
            return float("inf"), float("inf")
        C = generate_call_prices(market_strikes, R1, R2, S0, theta, c1, c2)
        mv = max(torch.clamp(market_bids - C, min=0).max().item(),
                 torch.clamp(C - market_asks, min=0).max().item())
        return mv, J.item()


def augmented_lagrangian_step(raw, lam_bid, lam_ask, rho, R1, R2, S0,
                              bid_ask, loss_fn, generate_call_prices,
                              lbfgs_max_iter=8, reparam_fn=None):
    """
    Minimise L_rho(z; lambda) = J + lam.viol + (rho/2)||viol||^2 over `raw`
    with L-BFGS (strong-Wolfe line search), then dual-ascent the multipliers.
    Returns (raw, lam_bid, lam_ask, J_true).
    """
    if reparam_fn is None:
        reparam_fn = lambda r: build_theta_from_raw(r, R1, R2, S0)
    market_strikes = bid_ask[:, 0]; market_bids = bid_ask[:, 1]; market_asks = bid_ask[:, 2]

    raw = raw.detach().clone().requires_grad_(True)
    optimizer = torch.optim.LBFGS(
        [raw], max_iter=lbfgs_max_iter, line_search_fn="strong_wolfe",
        tolerance_grad=1e-5, tolerance_change=1e-7)
    # tolerance_grad=1e-5 not 1e-13: solving AL subproblem to machine precision is wasted
    # work -- each outer step only needs a rough minimisation.  The outer dual-ascent loop
    # provides convergence; tight inner tolerance just causes 8x more line-search evals.

    def closure():
        optimizer.zero_grad()
        theta = reparam_fn(raw)
        J, _, _, c_v1, c_v2 = loss_fn(R1, R2, S0, theta)
        if torch.isnan(J) or torch.isinf(J):
            return torch.tensor(1e9, dtype=raw.dtype, requires_grad=True)
        C = generate_call_prices(market_strikes, R1, R2, S0, theta, c_v1, c_v2)
        bid_viol = torch.clamp(market_bids - C, min=0.0)
        ask_viol = torch.clamp(C - market_asks, min=0.0)
        lag = torch.dot(lam_bid, bid_viol) + torch.dot(lam_ask, ask_viol)
        pen = (rho / 2) * (torch.sum(bid_viol**2) + torch.sum(ask_viol**2))
        L = J + lag + pen        # the augmented Lagrangian L_rho(theta; lam)
        # Guard the FULL Lagrangian, not just J: with the new objective J stays finite
        # even when sigma -> 0, but the call-price recursion (C) can overflow -> nan L.
        # Reject such steps so L-BFGS backtracks instead of accepting a nan step (which
        # poisons raw and trips the outer non-finite halt).
        if not torch.isfinite(L):
            return torch.tensor(1e9, dtype=raw.dtype, requires_grad=True)
        L.backward()
        return L

    optimizer.step(closure)

    with torch.no_grad():
        theta = reparam_fn(raw)
        J_val, _, _, c_v1, c_v2 = loss_fn(R1, R2, S0, theta)
        C = generate_call_prices(market_strikes, R1, R2, S0, theta, c_v1, c_v2)
        # DUAL ASCENT (step 2): nudge each multiplier by rho * (signed violation),
        # clamped to >= 0 since these are one-sided (inequality) constraints.
        lam_bid = torch.clamp(lam_bid + rho * (market_bids - C), min=0.0)
        lam_ask = torch.clamp(lam_ask + rho * (C - market_asks), min=0.0)
        J_out = J_val.item() if torch.isfinite(J_val) else float("inf")

    return raw.detach(), lam_bid, lam_ask, J_out


In [ ]:
# ════════════════════════════════════════════════════════════════════════════
# SEARCH HEURISTICS (all in RAW space => candidates are always ordered/positive).
# These keep the algorithm moving when the inner solve stalls on the bid/ask wall.
# reparam_fn: same reparametrisation used by the caller (default: build_theta_from_raw).
# ════════════════════════════════════════════════════════════════════════════

def sample_feasible_neighborhood(raw_ref, R1, R2, S0, bid_ask, loss_fn,
                                 generate_call_prices, n_samples=100,
                                 radius=1.0, tol_constraint=1e-6,
                                 reparam_fn=None, verbose=True):
    """Multi-scale random search around raw_ref. Returns the best band-feasible
    raw found with strictly lower J, or None if nothing beats the reference."""
    mv_ref, J_ref = band_violation(raw_ref, R1, R2, S0, bid_ask, loss_fn,
                                   generate_call_prices, reparam_fn=reparam_fn)
    best = {"raw": None, "J": J_ref if mv_ref < tol_constraint else float("inf")}

    radii = [radius, radius * 0.1, radius * 0.01, radius * 0.001, radius * 10]
    per   = max(1, n_samples // len(radii))
    no_improve_streak = 0
    for r in radii:
        feasible_count = 0
        best_before_radius = best["J"]
        for _ in range(per):
            with torch.no_grad():
                raw_cand = raw_ref.detach() + torch.randn_like(raw_ref) * r
            mv, Jc = band_violation(raw_cand, R1, R2, S0, bid_ask, loss_fn,
                                    generate_call_prices, reparam_fn=reparam_fn)
            if mv < tol_constraint:
                feasible_count += 1
                if Jc < best["J"]:
                    best["J"], best["raw"] = Jc, raw_cand.clone()
                    no_improve_streak = 0
        if verbose:
            print(f"    [radius={r:.4g}] feasible {feasible_count}/{per} | best_J={best['J']:.10g}")
        # Early exit: if no improvement in last 2 radii, the search is exhausted (narrow basin)
        if best["J"] >= best_before_radius * 0.9999:  # <0.01% improvement
            no_improve_streak += 1
            if no_improve_streak >= 2:
                if verbose:
                    print(f"    [early exit] no improvement in last {no_improve_streak} radii, stopping search")
                break
    return best["raw"]


def find_feasible_perturbation(raw, R1, R2, S0, bid_ask, loss_fn,
                               generate_call_prices, n_attempts=100, radius=1e-2,
                               tol_constraint=1e-6, reparam_fn=None, verbose=True):
    """Gradient-orthogonal boundary search: move perpendicular to the augmented
    gradient (which points into the constraint wall), i.e. ALONG the boundary."""
    if reparam_fn is None:
        reparam_fn = lambda r: build_theta_from_raw(r, R1, R2, S0)
    market_strikes = bid_ask[:, 0]; market_bids = bid_ask[:, 1]; market_asks = bid_ask[:, 2]
    _, J_curr = band_violation(raw, R1, R2, S0, bid_ask, loss_fn, generate_call_prices,
                               reparam_fn=reparam_fn)
    best_raw, best_J = None, J_curr

    raw_g = raw.detach().clone().requires_grad_(True)
    theta_g = reparam_fn(raw_g)
    Jg, _, _, c1g, c2g = loss_fn(R1, R2, S0, theta_g)
    Cg = generate_call_prices(market_strikes, R1, R2, S0, theta_g, c1g, c2g)
    aug = Jg + 1e3 * (torch.sum(torch.clamp(market_bids - Cg, min=0)**2) +
                      torch.sum(torch.clamp(Cg - market_asks, min=0)**2))
    aug.backward()
    grad_dir = raw_g.grad.detach()
    grad_dir = grad_dir / (grad_dir.norm() + 1e-12)

    for attempt in range(n_attempts):
        with torch.no_grad():
            noise = torch.randn_like(raw)
            noise = noise - torch.dot(noise, grad_dir) * grad_dir
            noise = noise / (noise.norm() + 1e-12)
            scale = radius * (0.5 ** (attempt // 10))
            raw_cand = raw.detach() + noise * scale
        mv, Jc = band_violation(raw_cand, R1, R2, S0, bid_ask, loss_fn,
                                generate_call_prices, reparam_fn=reparam_fn)
        if mv < tol_constraint and Jc < best_J * 1.01:
            best_J, best_raw = Jc, raw_cand.clone()
            if verbose:
                print(f"    [perturb {attempt+1:03d}] feasible seed J={best_J:.10g}")
    return best_raw


def global_minimum_probe(raw_best, R1, R2, S0, bid_ask, loss_fn, generate_call_prices,
                         n_restarts=6, scatter=2.0, inner_iters=40,
                         tol_constraint=1e-6, reparam_fn=None, verbose=True):
    """
    Basin-hopping test for global vs. local. Scatter several random restarts in
    raw space, run a short Augmented-Lagrangian solve on each, keep the best
    band-feasible result. Returns a strictly-better feasible raw, or None.
    """
    n = bid_ask.shape[0]
    mv0, J0 = band_violation(raw_best, R1, R2, S0, bid_ask, loss_fn,
                             generate_call_prices, reparam_fn=reparam_fn)
    base_J = J0 if mv0 < tol_constraint else float("inf")
    best = {"raw": None, "J": base_J}
    if verbose:
        print(f"  [global probe] baseline feasible J = {base_J:.10g} "
              f"| launching {n_restarts} restarts (scatter={scatter})")

    for k in range(n_restarts):
        with torch.no_grad():
            raw_k = raw_best.detach() + torch.randn_like(raw_best) * scatter
        lam_b = torch.zeros(n); lam_a = torch.zeros(n); rho = 1.0
        for _ in range(3):
            raw_k, lam_b, lam_a, _ = augmented_lagrangian_step(
                raw_k, lam_b, lam_a, rho, R1, R2, S0, bid_ask,
                loss_fn, generate_call_prices, lbfgs_max_iter=inner_iters,
                reparam_fn=reparam_fn)
            rho *= 5.0
        mv, Jk = band_violation(raw_k, R1, R2, S0, bid_ask, loss_fn,
                                generate_call_prices, reparam_fn=reparam_fn)
        tag = "feasible" if mv < tol_constraint else f"infeasible(mv={mv:.1e})"
        if verbose:
            print(f"    restart {k+1}/{n_restarts}: J={Jk:.10g} [{tag}]")
        if mv < tol_constraint and Jk < best["J"] - 1e-12:
            best["J"], best["raw"] = Jk, raw_k.clone()

    if best["raw"] is not None and verbose:
        print(f"  [global probe] FOUND better basin: {base_J:.10g} -> {best['J']:.10g}")
    elif verbose:
        print(f"  [global probe] no better basin — current minimum survived the test")
    return best["raw"]


def feasibility_restoration(raw, R1, R2, S0, bid_ask, loss_fn, generate_call_prices,
                            n_steps=6, lbfgs_iters=40, tol_constraint=1e-6,
                            reparam_fn=None, verbose=True):
    """PURE-FEASIBILITY pass: minimise ONLY the squared bid/ask band violation (J is
    ignored) with L-BFGS, starting FROM the stuck infeasible iterate.

    Used by the dead-state guard: when rho has saturated and J is frozen while the
    iterate is still parked just outside the band (a violation floor ABOVE the
    feas-polish stall window, e.g. ~2e-3), the ordinary rewind discards this DEEP
    (low-J) point and restarts from the shallow feasible seed -- which simply re-runs
    the identical dead trajectory.  Instead we try to DRAG this deep point across the
    band by descending the violation alone, so its low J is captured feasible.
    Returns a band-feasible raw close to the input, or None if feasibility not reached."""
    if reparam_fn is None:
        reparam_fn = lambda r: build_theta_from_raw(r, R1, R2, S0)
    market_strikes = bid_ask[:, 0]; market_bids = bid_ask[:, 1]; market_asks = bid_ask[:, 2]

    raw_v = raw.detach().clone().requires_grad_(True)

    def closure():
        opt.zero_grad()
        theta = reparam_fn(raw_v)
        J, _, _, c1, c2 = loss_fn(R1, R2, S0, theta)
        C = generate_call_prices(market_strikes, R1, R2, S0, theta, c1, c2)
        bid_viol = torch.clamp(market_bids - C, min=0.0)
        ask_viol = torch.clamp(C - market_asks, min=0.0)
        F = torch.sum(bid_viol ** 2) + torch.sum(ask_viol ** 2)   # violation ONLY (no J)
        if not torch.isfinite(F):
            return torch.tensor(1e9, dtype=raw_v.dtype, requires_grad=True)
        F.backward()
        return F

    for k in range(n_steps):
        opt = torch.optim.LBFGS([raw_v], max_iter=lbfgs_iters,
                                line_search_fn="strong_wolfe",
                                tolerance_grad=1e-9, tolerance_change=1e-12)
        opt.step(closure)
        mv, Jc = band_violation(raw_v, R1, R2, S0, bid_ask, loss_fn,
                                generate_call_prices, reparam_fn=reparam_fn)
        if mv < tol_constraint:
            if verbose:
                print(f"    [feas-restore] reached feasibility in {k+1} step(s): "
                      f"J={Jc:.6g} (viol={mv:.2e})")
            return raw_v.detach()
    if verbose:
        mv, Jc = band_violation(raw_v, R1, R2, S0, bid_ask, loss_fn,
                                generate_call_prices, reparam_fn=reparam_fn)
        print(f"    [feas-restore] could not reach feasibility (best viol={mv:.2e}, J={Jc:.6g})")
    return None

In [ ]:
# ════════════════════════════════════════════════════════════════════════════
# MAIN DRIVER — outer Augmented-Lagrangian loop over the reparameterised `raw`.
# ----------------------------------------------------------------------------
# Each OUTER ITERATION calls augmented_lagrangian_step (prev cell) which does one
# L-BFGS minimisation of L_rho followed by a dual-ascent update of the multipliers.
# THIS driver wraps that inner step with the machinery that makes it robust:
#
#   1. PENALTY (rho) SCHEDULE — start at rho_init; raise rho by rho_scale (cap rho_max)
#      ONLY when an infeasible iteration's violation failed to shrink by factor
#      rho_increase_eta (textbook AL gate) -- while it IS shrinking, hold rho and let the
#      multipliers do the work.  This keeps the subproblem conditioned and stops rho
#      saturating at rho_max (which freezes L-BFGS).  A DEAD-STATE guard (dead_patience)
#      additionally rewinds+resets if rho ever saturates with J frozen while infeasible.
#
#   2. NEVER HALT ON INFEASIBILITY — a band violation is normal mid-search, not a
#      failure.  We keep iterating; we only ever RETURN a point that passes EVERY
#      check_constraints condition (see record_if_best / the feasible best-tracker).
#
#   3. STALL ESCAPE — if progress flatlines near the band boundary we first try a
#      FEASIBILITY-POLISH (#2): when the stalled iterate is much DEEPER (lower J) than the
#      best feasible point but only slightly infeasible, a few high-rho, extra-inner-iter
#      steps push it across the band so the deep point is captured feasible (instead of
#      being abandoned -- the main instability fix).  Only if that fails do we fall back to
#      sampling feasible neighbours and shrinking the radius, re-seeding from the best
#      feasible point.  The inner L-BFGS budget per outer is lbfgs_max_iter (#3, default 16).
#
#   4. BASIN-HOP PROBE (global vs local) — when J looks converged (converge_patience
#      hits) we optionally scatter probe_restarts starts (global_minimum_probe) to
#      check no deeper basin exists; if one does we jump to it.
#
#   5. BEST-FEASIBLE TRACKING — every iteration's feasible candidates are scored and
#      the lowest-J FEASIBLE theta seen is what gets returned (not the last iterate).
#
# Hard constraints (nus ordering / 0<nus1<S0<nus2 / sigs>0) are GUARANTEED by
# reparam_fn, so the loop only ever has to police the bid/ask band.
#
# KEY ARGS:
#   rho_init/rho_scale/rho_max : penalty schedule (see 1).
#   max_outer                  : outer-iteration budget (default 100; raise for big grids).
#   stop_on_convergence=False  : run the full budget; set True to stop once converged.
#   reparam_fn                 : raw->theta map (default build_theta_from_raw = joint
#                                sig+nu).  A sigma-only reparam freezes the nus.
#   raw_init                   : starting raw vector (default from raw_from_theta);
#                                sigma-only mode sets this explicitly.
# RETURNS: the lowest-J theta satisfying every check_constraints condition.
# ════════════════════════════════════════════════════════════════════════════

def optimize_augmented_lagrangian(R1, R2, S0, sigs1, sigs2, nus1, nus2,
                                  loss_fn, generate_call_prices, bid_ask,
                                  rho_init=1.0, rho_scale=2.0, rho_max=1e8,
                                  max_outer=100,
                                  tol_J_change=1e-4, tol_constraint=1e-6,
                                  n_samples=20, search_radius=2.0,
                                  converge_patience=2, probe_restarts=0,
                                  probe_scatter=2.0,
                                  stop_on_convergence=False,
                                  reparam_fn=None, raw_init=None,
                                  max_recover=3,
                                  rho_increase_eta=0.5, dead_patience=4,
                                  lbfgs_max_iter=10,
                                  feas_polish=True, feas_polish_thresh=5e-3,
                                  feas_polish_margin=0.1, feas_polish_steps=8,
                                  feas_polish_iters=30, feas_polish_rho_floor=1e4,
                                  feas_restore=True, feas_restore_steps=6,
                                  feas_restore_iters=40,
                                  verbose=True):

    n = bid_ask.shape[0]
    market_strikes = bid_ask[:, 0]

    # ── Default reparametrisation: joint sig+nu ──────────────────────────────
    if reparam_fn is None:
        reparam_fn = lambda r: build_theta_from_raw(r, R1, R2, S0)

    # ── Initialise raw ───────────────────────────────────────────────────────
    if raw_init is not None:
        raw = raw_init.clone().detach()
    else:
        raw = raw_from_theta(R1, R2, S0,
                             nus1.detach(), sigs1.detach(),
                             nus2.detach(), sigs2.detach())

    # ── Phase 0: hunt for a better feasible seed before optimising ──────────
    if verbose:
        print("=" * 64)
        print("Phase 0: neighbourhood sampling for a good feasible start")
        print("=" * 64)
    seed = sample_feasible_neighborhood(raw, R1, R2, S0, bid_ask, loss_fn,
                                        generate_call_prices, n_samples=n_samples,
                                        radius=search_radius, tol_constraint=tol_constraint,
                                        reparam_fn=reparam_fn, verbose=verbose)
    if seed is not None:
        raw = seed.clone()
        if verbose: print("Starting from improved feasible seed.\n")
    else:
        if verbose: print("No better seed found; starting from the initial point.\n")

    lam_bid = torch.zeros(n); lam_ask = torch.zeros(n); rho = rho_init
    best = {"theta": None, "raw": None, "J": float("inf")}
    J_prev = float("inf"); converge_streak = 0; current_radius = search_radius
    n_restarts = 0; n_probes = 0; n_recover = 0
    prev_viol = float("inf")   # #3: previous outer's band violation (gates rho growth)
    dead_streak = 0            # #4: consecutive rho-saturated + frozen + infeasible iters

    def record_if_best(raw_now):
        """Update `best` only with band-feasible points (hard constraints always hold)."""
        mv, J = band_violation(raw_now, R1, R2, S0, bid_ask, loss_fn,
                               generate_call_prices, reparam_fn=reparam_fn)
        if mv < tol_constraint and J < best["J"]:
            best["J"]     = J
            best["raw"]   = raw_now.detach().clone()
            best["theta"] = reparam_fn(best["raw"]).detach()
            return True, mv, J
        return False, mv, J

    # Capture the (feasible) starting point / Phase-0 seed as the initial best, so a
    # feasible solution is never lost even if every later AL iterate is infeasible.
    record_if_best(raw)

    for outer in range(max_outer):
        if verbose:
            print(f"\n-- Outer {outer+1:03d}/{max_outer} | rho={rho:.3g} | radius={current_radius:.3g} "
                  f"| restarts={n_restarts} | probes={n_probes} --")

        raw, lam_bid, lam_ask, J_curr = augmented_lagrangian_step(
            raw, lam_bid, lam_ask, rho, R1, R2, S0, bid_ask,
            loss_fn, generate_call_prices,
            lbfgs_max_iter=lbfgs_max_iter, reparam_fn=reparam_fn)

        improved, max_viol, J_true = record_if_best(raw)
        is_feasible = max_viol < tol_constraint
        J_change = abs(J_prev - J_curr)
        if verbose:
            star = "  * new best feasible" if improved else ""
            print(f"  J={J_true:.12g} | max_viol={max_viol:.2e} | feasible={is_feasible} "
                  f"| dJ={J_change:.2e}{star}")

        # RECOVER-AND-CONTINUE: a non-finite iterate means the inner solve
        # overflowed (high rho pushed sigma -> 0 -> LVG recursion blew up).  Rather
        # than abandon the whole run, rewind to the last feasible point, reset the
        # duals and drop rho back so the next step is gentle, shrink the search
        # radius, and keep using the remaining outer budget.  Only halt once the
        # recovery budget (max_recover) is exhausted (or no feasible point exists).
        if not (np.isfinite(J_true) and np.isfinite(max_viol)):
            n_recover += 1
            if best["raw"] is not None and n_recover <= max_recover:
                raw = best["raw"].clone().detach()
                lam_bid = torch.zeros(n); lam_ask = torch.zeros(n)
                rho = rho_init
                current_radius = max(current_radius * 0.5, 0.1)
                J_prev = float("inf"); prev_viol = float("inf"); dead_streak = 0
                if verbose:
                    print(f"  non-finite J/violation -> recovering (#{n_recover}/{max_recover}) "
                          f"from best feasible J={best['J']:.6g}; rho reset to {rho:.3g}, "
                          f"radius -> {current_radius:.3g}")
                continue
            if verbose:
                print("  non-finite J/violation -> halting (recovery budget exhausted "
                      "or no feasible point); keeping best feasible so far")
            break

        if not is_feasible:
            # #3 VIOLATION-GATED rho: raise rho ONLY when the band violation failed to
            # shrink by at least factor rho_increase_eta vs the previous outer.  While the
            # violation IS shrinking the multipliers are doing the job, so HOLD rho -- this
            # keeps the subproblem well-conditioned and stops rho running away to rho_max
            # (which is what froze L-BFGS in the unstable run).  Multipliers update every
            # step regardless, so feasibility is still driven.  No extra evals.
            if max_viol > rho_increase_eta * prev_viol:
                rho = min(rho * rho_scale, rho_max)
                if verbose: print(f"  band-infeasible (viol not shrinking) -> rho -> {rho:.3g}")
            elif verbose:
                print(f"  band-infeasible (viol shrinking) -> hold rho={rho:.3g}")
        prev_viol = max_viol if np.isfinite(max_viol) else prev_viol

        # #4 DEAD-STATE (rho-saturation) ESCAPE: if rho has hit rho_max and J is frozen
        # while STILL infeasible, the penalty subproblem is uninvertible and L-BFGS makes
        # zero progress; the ordinary stall-search may not fire (the residual violation can
        # sit just above its threshold).  Detect it and force the same rewind-to-best +
        # rho-reset + perturbation the stall path uses, so the remaining budget is spent
        # productively.  A scalar check per outer -> runtime unchanged; never fires on
        # dates that converge normally.
        if (rho >= rho_max) and (not is_feasible) and (J_change < tol_J_change):
            dead_streak += 1
        else:
            dead_streak = 0
        if dead_streak >= dead_patience:
            if verbose:
                print(f"  [dead-state] rho saturated + J frozen + infeasible for "
                      f"{dead_streak} iters")
            # OPTION-2 FEASIBILITY RESTORATION (runs ONLY inside this dead-state branch, so
            # dates that converge normally never reach it -> their results are unchanged).
            # The stuck iterate is DEEP (low J) but parked just outside the band at a
            # violation floor (e.g. ~2e-3) ABOVE the feas-polish stall window, so the polish
            # never fired; the default rewind then throws this deep point away and restarts
            # from the shallow feasible seed -- which just re-runs the identical dead
            # trajectory (the instability seen on ~1/30 short-tenor dates).  Instead we first
            # try to DRAG this deep point across the band by minimising the band violation
            # ALONE (J ignored); if it lands feasible with lower J it is captured as best.
            if feas_restore:
                restored = feasibility_restoration(
                    raw.detach(), R1, R2, S0, bid_ask, loss_fn, generate_call_prices,
                    n_steps=feas_restore_steps, lbfgs_iters=feas_restore_iters,
                    tol_constraint=tol_constraint, reparam_fn=reparam_fn, verbose=verbose)
                if restored is not None:
                    r_improved, r_mv, r_J = record_if_best(restored)
                    if verbose:
                        _tag = ("captured new best feasible" if r_improved
                                else "feasible but not better than current best")
                        print(f"    [feas-restore] {_tag}: J={r_J:.6g} (viol={r_mv:.2e})")
                    if r_improved:
                        raw = restored.clone().detach()
                        lam_bid = torch.zeros(n); lam_ask = torch.zeros(n); rho = rho_init
                        J_prev = float("inf"); prev_viol = float("inf"); dead_streak = 0
                        n_restarts += 1
                        continue
            # restoration disabled / failed / not better -> original rewind-and-perturb path
            if verbose:
                print(f"  [dead-state] -> rewind to best feasible, reset rho, perturb")
            start = best["raw"] if best["raw"] is not None else raw.detach()
            new_raw = sample_feasible_neighborhood(
                start, R1, R2, S0, bid_ask, loss_fn, generate_call_prices,
                n_samples=n_samples, radius=current_radius,
                tol_constraint=tol_constraint, reparam_fn=reparam_fn, verbose=verbose)
            raw = (new_raw if new_raw is not None else start).clone().detach()
            lam_bid = torch.zeros(n); lam_ask = torch.zeros(n); rho = rho_init
            J_prev = float("inf"); prev_viol = float("inf"); dead_streak = 0
            n_restarts += 1
            record_if_best(raw)
            continue

        # Check for convergence: feasible for >=3 iterations + J stable
        if is_feasible:
            converge_streak += 1
            if converge_streak >= 3 and J_change < tol_J_change:
                if verbose:
                    print(f"  CONVERGED: feasible for {converge_streak} iter + dJ={J_change:.2e} < {tol_J_change}")
                break
        else:
            converge_streak = 0

        stalled = (J_change < tol_J_change * 1000) and (max_viol < tol_constraint * 100)
        if stalled and not improved:
            # #2 FEASIBILITY-POLISH: the smoothest feasible curve sits right on the band
            # edge, so a deep iterate can stall just BARELY infeasible (e.g. viol~1e-4) while
            # the best feasible point so far is much shallower.  The default stall-search
            # would ABANDON this deep point and rewind to the shallow feasible one -- the
            # main cause of run-to-run instability.  Instead, first try to CAPTURE the deep
            # point: a few high-rho, extra-inner-iteration steps FROM THE CURRENT raw to push
            # its small residual violation below tol.  Only if that fails do we fall through
            # to the ordinary restart-search.  Fires rarely (stall events only) -> cheap.
            if (feas_polish and np.isfinite(J_true) and np.isfinite(max_viol)
                    and J_true < best["J"] - feas_polish_margin
                    and tol_constraint <= max_viol < feas_polish_thresh):
                if verbose:
                    print(f"  [feas-polish] deep (J={J_true:.6g} < best {best['J']:.6g}) & "
                          f"near-feasible (viol={max_viol:.2e}) -> pushing toward feasibility")
                polish_raw = raw.detach().clone()
                polish_rho = min(max(rho, feas_polish_rho_floor), rho_max)
                captured = False
                for _ in range(feas_polish_steps):
                    polish_raw, lam_bid, lam_ask, _ = augmented_lagrangian_step(
                        polish_raw, lam_bid, lam_ask, polish_rho, R1, R2, S0, bid_ask,
                        loss_fn, generate_call_prices,
                        lbfgs_max_iter=max(lbfgs_max_iter, feas_polish_iters),
                        reparam_fn=reparam_fn)
                    pol_improved, pol_mv, pol_J = record_if_best(polish_raw)
                    polish_rho = min(polish_rho * rho_scale, rho_max)
                    if pol_improved:
                        captured = True
                        if verbose:
                            print(f"    [feas-polish] CAPTURED feasible J={pol_J:.6g} (viol={pol_mv:.2e})")
                        break
                if captured:
                    raw = polish_raw.detach(); rho = polish_rho
                    J_prev = float("inf"); prev_viol = float("inf"); dead_streak = 0
                    continue
                elif verbose:
                    print("    [feas-polish] could not reach feasibility -> normal restart-search")
            if verbose: print("  stalled near boundary -> searching")
            start = best["raw"] if best["raw"] is not None else raw.detach()
            new_raw = sample_feasible_neighborhood(
                start, R1, R2, S0, bid_ask, loss_fn, generate_call_prices,
                n_samples=n_samples, radius=current_radius,
                tol_constraint=tol_constraint, reparam_fn=reparam_fn, verbose=verbose)
            if new_raw is not None and abs(new_raw.sum().item() - start.sum().item()) > 1e-9:
                current_radius *= 1.5
            else:
                current_radius *= 0.5
                new_raw = find_feasible_perturbation(
                    start, R1, R2, S0, bid_ask, loss_fn, generate_call_prices,
                    n_attempts=100, radius=current_radius,
                    tol_constraint=tol_constraint, reparam_fn=reparam_fn, verbose=verbose)
            if new_raw is not None:
                raw = new_raw.clone().detach()
                lam_bid = torch.zeros(n); lam_ask = torch.zeros(n); rho = rho_init
                J_prev = float("inf"); prev_viol = float("inf"); dead_streak = 0
                n_restarts += 1
                record_if_best(raw)
                continue

        if J_change < tol_J_change:
            converge_streak += 1
        else:
            converge_streak = 0

        if converge_streak >= converge_patience:
            if verbose: print(f"  J looks converged ({converge_streak}x) -> global-vs-local probe")
            n_probes += 1
            start = best["raw"] if best["raw"] is not None else raw.detach()
            jumped = global_minimum_probe(
                start, R1, R2, S0, bid_ask, loss_fn, generate_call_prices,
                n_restarts=probe_restarts, scatter=probe_scatter,
                tol_constraint=tol_constraint, reparam_fn=reparam_fn, verbose=verbose)
            converge_streak = 0
            if jumped is not None:
                raw = jumped.clone().detach()
                lam_bid = torch.zeros(n); lam_ask = torch.zeros(n); rho = rho_init
                J_prev = float("inf"); prev_viol = float("inf"); dead_streak = 0
                record_if_best(raw)
                continue
            elif stop_on_convergence:
                if verbose: print("  minimum survived the probe and stop_on_convergence=True -> stop")
                break

        J_prev = J_curr

    # ── Finalise: return the best FEASIBLE theta ─────────────────────────────
    if best["theta"] is not None:
        theta_final = best["theta"].clone().detach()
        mv_f, J_f = band_violation(best["raw"], R1, R2, S0, bid_ask, loss_fn,
                                   generate_call_prices, reparam_fn=reparam_fn)
    else:
        theta_final = reparam_fn(raw).detach()
        mv_f, J_f = band_violation(raw, R1, R2, S0, bid_ask, loss_fn,
                                   generate_call_prices, reparam_fn=reparam_fn)
        if verbose: print("\n[WARNING] No band-feasible point found; returning last iterate.")

    if verbose:
        print(f"\nFinal J:               {J_f:.12g}")
        print(f"Final max band viol:   {mv_f:.2e}")
        print(f"All constraints met:   {mv_f < tol_constraint}")
    return theta_final.requires_grad_(True)


In [ ]:

# ── small helpers ───────────────────────────────────────────────────────────
def extract_nus_sigs(theta, R1, R2):
    idx = 0
    nus1  = theta[idx:idx+R1].detach().cpu().numpy(); idx += R1
    sigs1 = theta[idx:idx+R1].detach().cpu().numpy(); idx += R1
    nus2  = theta[idx:idx+R2].detach().cpu().numpy(); idx += R2
    sigs2 = theta[idx:idx+R2].detach().cpu().numpy(); idx += R2
    return nus1, sigs1, nus2, sigs2

def compute_second_derivative(C_K_vals, strikes):
    """Numerical C''(K) via central differences (used in plot E alongside analytics)."""
    strikes_np = strikes.detach().cpu().numpy()
    C_np       = C_K_vals.detach().cpu().numpy()
    return np.gradient(np.gradient(C_np, strikes_np), strikes_np)

# ── implied volatility (Black model — same logic as Hwk_prob5b_sol.ipynb) ──
def blackPremium(discountFactor, forward, strike, timeToExpiration, sigma):
    """Black call formula: discountFactor * (F*N(d1) - K*N(d2))."""
    with np.errstate(divide='ignore'):
        terminalVolatility = sigma * np.sqrt(timeToExpiration)
        logMoneyness = np.log(np.divide(forward, strike))
        d1 = (0.0 if logMoneyness == 0.0
              else np.divide(logMoneyness, terminalVolatility)) + 0.5 * terminalVolatility
        d2 = (0.0 if logMoneyness == 0.0
              else np.divide(logMoneyness, terminalVolatility)) - 0.5 * terminalVolatility
        return discountFactor * (forward * norm.cdf(d1) - strike * norm.cdf(d2))

def blackVolatility(discountFactor, forward, strike, timeToExpiration, premium):
    """Invert Black formula via bisection (same structure as Hwk_prob5b_sol.ipynb)."""
    def getPremium(sigma):
        return blackPremium(discountFactor, forward, strike, timeToExpiration, sigma)
    if np.isnan(premium):
        return np.nan
    if premium <= getPremium(0.0):   # at or below intrinsic — zero time value
        return 0.0
    if premium >= getPremium(np.inf):  # above discounted forward — infeasible
        return np.inf
    low = 0.0; high = 1.0
    while getPremium(high) < premium:  # double until bracketed
        high = 2.0 * high
    while high - low > 1.0e-8:        # bisect
        mid = 0.5 * (low + high)
        if getPremium(mid) >= premium:
            high = mid
        else:
            low = mid
    return 0.5 * (low + high)

# ===== UNUSED (no references found anywhere in notebook, 2026-06-23) — commented out for review; safe to delete =====
# def compute_iv_surface(C_vals, strikes_np, S0, T=1.0, r=0.0):
#     """Compute implied vol for each strike using the Black model."""
#     discountFactor = np.exp(-r * T)
#     forward = S0 * np.exp(r * T)
#     return np.array([blackVolatility(discountFactor, forward, float(K), T, float(C_vals[i]))
#                      for i, K in enumerate(strikes_np)])

def plot_implied_vols(strikes_np, C_init_np, C_final_np, S0,
                      bids_np=None, asks_np=None, T=1.0, save=True,
                      iv_floor=1e-3, iv_cap=3.0, spread_fence=1.5):
    """Implied-volatility smile, Hwk_prob5b_sol.ipynb logic, in two panels.

    Prices from Process_SPX are in the normalised 'S0 measure' (mfac=S0/(D*F)
    divides out the discount factor, Khat=K*S0/F rescales strikes), so the
    correct Black inversion uses discountFactor=1.0 and forward=S0 -- exactly
    Hwk 5b's blackVolatility(1.0, S, K, T, premium).  x-axis is log-moneyness
    log(K/S0); bid/ask IV are scatter points (Hwk 5b style).

    LEFT panel  : the full valid chain (every finite, in-(floor,cap) IV).
    RIGHT panel : a clean, Hwk-5b-like near-money window.  Deep-ITM calls are
                  priced almost at intrinsic, so their IV is ill-conditioned and
                  the bid-ask IV spread blows up there.  The clean window is the
                  log-moneyness span of the points whose bid-ask IV spread is at
                  or below the MEDIAN spread (a parameter-free split).  This is
                  purely data-driven -- no hard-coded strike/moneyness limits --
                  so it adapts to any chain.
    """
    # log-moneyness x-axis (Hwk 5b: x = [log(k/S) for k in K])
    x = np.log(strikes_np / S0)

    # model IV via Black with df=1, forward=S0 (Hwk 5b convention)
    iv_init  = np.array([blackVolatility(1.0, S0, float(k), T, float(C_init_np[i]))
                         for i, k in enumerate(strikes_np)])
    iv_final = np.array([blackVolatility(1.0, S0, float(k), T, float(C_final_np[i]))
                         for i, k in enumerate(strikes_np)])

    # market bid/ask IV (computed once, reused by both panels)
    iv_bid = iv_ask = None
    if bids_np is not None and asks_np is not None:
        iv_bid = np.array([blackVolatility(1.0, S0, float(k), T, float(bids_np[i]))
                           for i, k in enumerate(strikes_np)])
        iv_ask = np.array([blackVolatility(1.0, S0, float(k), T, float(asks_np[i]))
                           for i, k in enumerate(strikes_np)])

    # a point is meaningful only if its IV is finite and not collapsed/explosive
    def _good(v):
        return np.isfinite(v) & (v > iv_floor) & (v < iv_cap)

    # data-driven clean window from the bid-ask IV spread.
    # Deep-ITM calls are ill-conditioned: a fixed dollar bid-ask maps to a huge
    # IV spread because vega -> 0 there.  Near ATM vega is large so the IV spread
    # is small.  We keep the strikes whose bid-ask IV spread is at or below the
    # MEDIAN spread (a parameter-free split, no hard-coded moneyness limits), then
    # take their log-moneyness span as the window.  This reproduces a Hwk-5b-like
    # near-money view and adapts to any chain.
    if iv_bid is not None:
        both = _good(iv_bid) & _good(iv_ask)
    else:
        both = _good(iv_final)
    clean_keep = np.zeros_like(x, dtype=bool)
    if both.sum() >= 4 and iv_bid is not None:
        spr = (iv_ask - iv_bid)[both]
        tight = both.copy()
        tight[both] = spr <= np.median(spr)
        if tight.any():
            xlo, xhi = x[tight].min(), x[tight].max()
            clean_keep = (x >= xlo) & (x <= xhi)
    if not clean_keep.any():
        # fallback: no two-sided quotes -> use the span of valid model IVs
        clean_keep = both.copy() if both.any() else np.ones_like(x, dtype=bool)

    def _panel(ax, keep, title):
        # market bid/ask IV as scatter (Hwk 5b: plt.scatter(x_mrkt, IV_a/IV_b))
        xs, ys = [], []
        if iv_bid is not None:
            ma = _good(iv_ask) & keep; mb = _good(iv_bid) & keep
            ax.scatter(x[ma], iv_ask[ma], s=22, color="firebrick", marker="^", label="Market ask IV")
            ax.scatter(x[mb], iv_bid[mb], s=22, color="green",     marker="v", label="Market bid IV")
            xs += [x[ma], x[mb]]; ys += [iv_ask[ma], iv_bid[mb]]
        # model smile as connected lines (Hwk 5b: plt.plot(x, IV))
        vi = _good(iv_init) & keep; vf = _good(iv_final) & keep
        ax.plot(x[vi], iv_init[vi], "-", color="steelblue", lw=1.2, label="Initial IV (arb-free)")
        ax.plot(x[vf], iv_final[vf], "-", color="tomato",   lw=1.6, label="Final IV (LVG fit)")
        xs += [x[vi], x[vf]]; ys += [iv_init[vi], iv_final[vf]]
        # ATM marker at log-moneyness 0
        ax.axvline(0.0, color="gray", ls="--", lw=0.8, label="ATM (K=S0)")
        # auto-zoom to the plotted points
        ax_x = np.concatenate([a for a in xs if a.size]) if any(a.size for a in xs) else np.array([])
        ax_y = np.concatenate([a for a in ys if a.size]) if any(a.size for a in ys) else np.array([])
        if ax_x.size:
            xpad = 0.03 * (ax_x.max() - ax_x.min() + 1e-9)
            ypad = 0.08 * (ax_y.max() - ax_y.min() + 1e-9)
            ax.set_xlim(ax_x.min() - xpad, ax_x.max() + xpad)
            ax.set_ylim(max(0.0, ax_y.min() - ypad), ax_y.max() + ypad)
        ax.set_xlabel("log-moneyness  log(K / S0)")
        ax.set_ylabel("Implied Vol")
        ax.set_title(title)
        ax.legend(fontsize=8); ax.grid(alpha=0.3)

    fig, (ax_full, ax_clean) = plt.subplots(1, 2, figsize=(16, 6))
    _panel(ax_full,  np.ones_like(x, dtype=bool), "Implied Volatility Smile")
    _panel(ax_clean, clean_keep,                  "Implied Volatility Smile")

    #   (Hwk 5b: df=1, forward=S0)
    plt.suptitle(f"Implied Volatility | T={T}, S0={S0}",
                 fontsize=12, fontweight="bold")
    stamp = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
    plt.tight_layout()
    if save:
        plt.savefig(f"implied_volatility_{stamp}.png", dpi=150, bbox_inches="tight"); plt.show()
    return fig


# ── diagnostic (b): per-knot decomposition of J ─────────────────────────────
def j_per_knot(R1, R2, S0, theta):
    """Read-only replica of calculate_J's per-knot terms (calculate_J unchanged).

    Reuses the scaled coefficients (c_v1, c_v2) that calculate_J returns, then
    replicates only the jump / weight / spread-spacing-factor assembly so the
    contributions sum EXACTLY to J.  Returns (knot_locations, contributions).
    """
    idx = 0
    nus1 = theta[idx:idx+R1]; idx += R1
    sigs1 = theta[idx:idx+R1]; idx += R1
    nus2 = theta[idx:idx+R2]; idx += R2
    sigs2 = theta[idx:idx+R2]; idx += R2
    # pull the scaled two-exponential coefficients straight from calculate_J
    _, _, _, c_v1, c_v2 = calculate_J(R1, R2, S0, theta)
    nus1 = torch.cat([nus1, torch.tensor([S0])])
    nus2 = torch.cat([torch.tensor([S0]), nus2])

    def eval_V_K(K_val):
        if K_val <= S0:
            i = torch.clamp(torch.bucketize(K_val, nus1, right=False) - 1, 0, R1 - 1).item()
            d = nus1[i+1] - K_val
            V = c_v1[i][0]*torch.exp(d/sigs1[i]) + c_v1[i][1]*torch.exp(-d/sigs1[i])
            return V + (S0 - K_val)
        i = torch.clamp(torch.bucketize(K_val, nus2, right=False) - 1, 0, R2 - 1).item()
        d = K_val - nus2[i]
        return c_v2[i][0]*torch.exp(-d/sigs2[i]) + c_v2[i][1]*torch.exp(d/sigs2[i])

    jumps, weights, knot_ks = [], [], []
    with torch.no_grad():
        # left interior knots
        for j in range(R1 - 1):
            vL = (1/sigs1[j]**2)*(c_v1[j][0] + c_v1[j][1])
            d  = nus1[j+2] - nus1[j+1]
            vR = (1/sigs1[j+1]**2)*(c_v1[j+1][0]*torch.exp(d/sigs1[j+1]) +
                                    c_v1[j+1][1]*torch.exp(-d/sigs1[j+1]))
            jumps.append((vR - vL).item())
            weights.append(float(torch.clamp(eval_V_K(nus1[j+1]), min=1e-4)))
            knot_ks.append(nus1[j+1].item())
        # the S0 seam
        vL0 = (1/sigs1[-1]**2)*(c_v1[-1][0] + c_v1[-1][1])
        vR0 = (1/sigs2[0]**2)*(c_v2[0][0] + c_v2[0][1])
        jumps.append((vR0 - vL0).item())
        weights.append(float(torch.clamp(eval_V_K(torch.tensor(float(S0), dtype=theta.dtype)), min=1e-4)))
        knot_ks.append(float(S0))
        # right interior knots
        for j in range(R2 - 1):
            vR = (1/sigs2[j+1]**2)*(c_v2[j+1][0] + c_v2[j+1][1])
            d  = nus2[j+1] - nus2[j]
            vL = (1/sigs2[j]**2)*(c_v2[j][0]*torch.exp(-d/sigs2[j]) +
                                  c_v2[j][1]*torch.exp(d/sigs2[j]))
            jumps.append((vR - vL).item())
            weights.append(float(torch.clamp(eval_V_K(nus2[j+1]), min=1e-4)))
            knot_ks.append(nus2[j+1].item())

    kk = np.asarray(knot_ks, float)
    jumps = np.asarray(jumps, float); weights = np.asarray(weights, float)
    # Per-knot C''-jump contribution (time-value weighted).  NOTE: this is the
    # C'' roughness decomposition used for the diagnostic plot only; the optimised
    # objective J is now the 1/sigma^2 roughness (see calculate_J), not this sum.
    contrib = jumps**2 / weights
    return kk, contrib


def plot_j_per_knot(R1i, R2i, theta_i, R1f, R2f, theta_f, S0, top_n=20, save=False):
    """Diagnostic (b): where J comes from, per knot, initial vs final."""
    kk_i, ci = j_per_knot(R1i, R2i, S0, theta_i)
    kk_f, cf = j_per_knot(R1f, R2f, S0, theta_f)
    Ji, Jf = float(ci.sum()), float(cf.sum())
    xi, xf = np.log(kk_i / S0), np.log(kk_f / S0)

    fig, (axL, axR) = plt.subplots(1, 2, figsize=(16, 5))

    # LEFT: each knot's contribution vs its log-moneyness (log y)
    # set the log floor from the smallest POSITIVE contribution so near-zero
    # knots don't drag the axis down to ~1e-300
    allc = np.concatenate([ci, cf])
    pos  = allc[allc > 0]
    floor = (pos.min() * 0.5) if pos.size else 1e-12
    ymax  = (pos.max() * 2.0) if pos.size else 1.0
    axL.vlines(xi, floor, np.clip(ci, floor, None), color="steelblue", alpha=0.4, lw=1)
    axL.plot(xi, np.clip(ci, floor, None), "o", color="steelblue", ms=3, label=f"initial (J={Ji:.4g})")
    axL.vlines(xf, floor, np.clip(cf, floor, None), color="tomato", alpha=0.4, lw=1)
    axL.plot(xf, np.clip(cf, floor, None), "s", color="tomato", ms=3, label=f"final (J={Jf:.4g})")
    axL.axvline(0.0, color="gray", ls="--", lw=0.8, label="ATM")
    if pos.size:
        axL.set_yscale("log"); axL.set_ylim(floor, ymax)
    axL.set_xlabel("knot log-moneyness  log(K / S0)")
    axL.set_ylabel("J contribution  (log scale)")
    axL.set_title("Per-knot J contribution vs location")
    axL.legend(fontsize=8); axL.grid(alpha=0.3, which="both")

    # RIGHT: the top-N knots dominating the FINAL J, with cumulative share
    n = min(top_n, cf.size)
    order = np.argsort(cf)[::-1][:n]
    vals, locs = cf[order], kk_f[order]
    bx = np.arange(n)
    axR.bar(bx, vals, color="tomato", alpha=0.75)
    axR.set_xticks(bx); axR.set_xticklabels([f"{l:.0f}" for l in locs], rotation=90, fontsize=6)
    axR.set_xlabel("knot location Khat (sorted by contribution)")
    axR.set_ylabel("J contribution")
    axR.set_title(f"Top {n} knots dominating final J")
    axR.grid(alpha=0.3, axis="y")
    ax2 = axR.twinx()
    cum = (np.cumsum(vals) / Jf * 100.0) if Jf > 0 else np.zeros(n)
    ax2.plot(bx, cum, "k.-", lw=1, ms=4)
    ax2.set_ylabel("cumulative % of final J"); ax2.set_ylim(0, 105)

    fig.suptitle(f"Per-knot J decomposition   (init J={Ji:.4g}  ->  final J={Jf:.4g})",
                 fontsize=12, fontweight="bold")
    fig.tight_layout()
    if save:
        stamp = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
        fig.savefig(f"j_per_knot_{stamp}.png", dpi=150, bbox_inches="tight")
    return fig


# ── diagnostic (a): optimizer convergence traces ────────────────────────────
def plot_convergence(run_log_text, tol_constraint=1e-6, save=False):
    """Diagnostic (a): J, max band violation, and penalty rho vs outer iteration.

    Parsed from the captured optimizer log (the optimizer itself is untouched)."""
    import re
    def _f(x):
        try: return float(x)
        except Exception: return np.nan
    heads = re.findall(r'-- Outer\s+(\d+)/\d+\s*\|\s*rho=([0-9.eE+\-]+|nan|inf)', run_log_text)
    jl    = re.findall(r'J=([0-9.eE+\-]+|nan|inf)\s*\|\s*max_viol=([0-9.eE+\-]+|nan|inf)', run_log_text)
    n = min(len(heads), len(jl))

    fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))
    if n == 0:
        for ax in axes:
            ax.text(0.5, 0.5, "no per-iteration\nlog parsed", ha="center", va="center")
            ax.set_xticks([]); ax.set_yticks([])
        fig.suptitle("Optimizer convergence (no iteration lines found in log)",
                     fontsize=12, fontweight="bold")
        fig.tight_layout()
        return fig

    it   = np.array([int(heads[i][0]) for i in range(n)])
    rho  = np.array([_f(heads[i][1]) for i in range(n)])
    Jh   = np.array([_f(jl[i][0]) for i in range(n)])
    viol = np.array([_f(jl[i][1]) for i in range(n)])

    # J vs outer
    ax = axes[0]
    ax.plot(it, Jh, "o-", color="tomato", ms=4)
    finite = Jh[np.isfinite(Jh)]
    if finite.size and np.all(finite > 0):
        ax.set_yscale("log")
    ax.set_xlabel("outer iteration"); ax.set_ylabel("J (objective)")
    ax.set_title("J vs iteration"); ax.grid(alpha=0.3, which="both")

    # max band violation vs outer
    ax = axes[1]
    ax.plot(it, np.clip(viol, 1e-16, None), "s-", color="steelblue", ms=4, label="max band viol")
    ax.axhline(tol_constraint, color="green", ls="--", lw=1, label=f"tol={tol_constraint:g}")
    ax.set_yscale("log"); ax.set_xlabel("outer iteration"); ax.set_ylabel("max band violation")
    ax.set_title("Feasibility vs iteration"); ax.legend(fontsize=8); ax.grid(alpha=0.3, which="both")

    # penalty rho vs outer
    ax = axes[2]
    ax.plot(it, rho, "d-", color="purple", ms=4)
    if np.all(rho[np.isfinite(rho)] > 0):
        ax.set_yscale("log")
    ax.set_xlabel("outer iteration"); ax.set_ylabel("penalty rho")
    ax.set_title("Penalty rho vs iteration"); ax.grid(alpha=0.3, which="both")

    fig.suptitle("Optimizer convergence traces", fontsize=12, fontweight="bold")
    fig.tight_layout()
    if save:
        stamp = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
        fig.savefig(f"convergence_{stamp}.png", dpi=150, bbox_inches="tight")
    return fig


# ── Yahoo-Finance spot S0 (drop-in alternative to the data-fitted ch["S0"]) ──
def yahoo_S0(date, ticker="^SPX"):
    """Underlying spot close from Yahoo Finance for a quote date.

    Use as a one-liner override of the options-fitted spot, e.g.
        S0 = yahoo_S0(20240829)
    then pass that S0 into build_theta / run_and_plot.

    date  : int or str 'YYYYMMDD' (e.g. 20240829).
    ticker: Yahoo symbol of the underlying ('^SPX' for the S&P 500 index).
    Returns a float close; raises ValueError if Yahoo has no data that day.

    NOTE: a Yahoo close is taken at a different instant than the option-quote
    snapshot and ignores the borrow/dividend wedge, so it can be mildly
    inconsistent with the put-call-parity forward used by Process_SPX's
    normalisation.  Prefer ch["S0"] (fitted from the options) when available;
    use this for single-expiry slices where S0 cannot be fitted endogenously.
    """
    import yfinance as yf
    from datetime import datetime, timedelta
    d = str(int(date)) if not isinstance(date, str) else date
    start = datetime.strptime(d, "%Y%m%d")
    end   = start + timedelta(days=1)
    px = yf.download(ticker, start=start.strftime("%Y-%m-%d"),
                     end=end.strftime("%Y-%m-%d"), progress=False, auto_adjust=False)
    if px is None or len(px) == 0:
        raise ValueError(f"Yahoo returned no data for {ticker} on {d}")
    close = px["Close"]
    val = close.iloc[0]
    # handle multi-column frames (close.iloc[0] may itself be a Series)
    return float(val.iloc[0]) if hasattr(val, "iloc") else float(val)



In [ ]:
def verify_solution(theta, R1, R2, S0, bid_ask, tol_constraint=1e-6, verbose=True):
    """Report each constraint group of check_constraints separately."""
    market_strikes = bid_ask[:, 0]
    nus1, nus2 = obtain_nus(R1, R2, theta)
    sigs1, sigs2 = obtain_sigs(R1, R2, theta)
    inc1 = bool(torch.all(torch.diff(nus1) > 0) and nus1[-1] <= S0)
    inc2 = bool(torch.all(torch.diff(nus2) > 0) and nus2[0] >= S0)
    pos1 = bool(torch.all(sigs1 > 0)); pos2 = bool(torch.all(sigs2 > 0))
    _, _, _, c1, c2 = calculate_J(R1, R2, S0, theta)
    C = generate_call_prices(market_strikes, R1, R2, S0, theta, c1, c2)
    mv = max(torch.clamp(bid_ask[:,1]-C, min=0).max().item(),
             torch.clamp(C-bid_ask[:,2], min=0).max().item())
    band = mv < tol_constraint
    allok = inc1 and inc2 and pos1 and pos2 and band
    if verbose:
        print("\n" + "=" * 64)
        print("CONSTRAINT VERIFICATION (every condition in check_constraints)")
        print("=" * 64)
        print(f"  nus1 strictly increasing & <= S0 : {inc1}")
        print(f"  nus2 strictly increasing & >= S0 : {inc2}")
        print(f"  sigs1 > 0                        : {pos1}")
        print(f"  sigs2 > 0                        : {pos2}")
        print(f"  inside bid/ask band (max viol)   : {band}  (max_viol={mv:.2e})")
        print(f"  --> check_constraints PASSES          : {allok}")
    return allok


def run_and_plot(R1, R2, S0, sigs1, sigs2, nus1, nus2,
                 loss_fn, generate_call_prices, bid_ask,
                 arb_prices=None, arb_strikes=None,
                 optimize_mode="sig_nu",
                 R1_new=None, R2_new=None, Kbar_sigonly=None, uniform_nus=False,
                 R1_mult=None, R2_mult=None,
                 normalize=False, obj=1,
                 T=1.0, r=0.0, start_date=None, expiry_date=None, build_log=None,
                 out_dir=".",
                 **optimizer_kwargs):
    """One-call wrapper: initial diagnostics -> optimise -> final diagnostics ->
    constraint check -> full plot suite (A-G + IV). Returns the final theta.

    optimize_mode: 'sig_nu'   -- optimise both sigmas and nus (default / original)
                   'sig_only' -- fix equally-spaced nus, optimise sigmas only
                   'sig_only_LKbar' -- fix nus except for the two extreme knots (L, Kbar), optimise sigmas only

    R1_new, R2_new: knot counts for the sigma-only grid (default: 2*R1, 2*R2).
    Kbar_sigonly:   Kbar used by build_sigma_only_start (default: max market strike).

    arb_prices / arb_strikes: arb-free call prices from Process_SPX (within the
      bid-ask band by construction).  When supplied, price/band plots show these
      as the 'initial' curve.  Smoothing/C'' diagnostics always use the LVG theta.
    """
    market_strikes = bid_ask[:, 0]; market_bids = bid_ask[:, 1]; market_asks = bid_ask[:, 2]
    # The single smoothness objective is calculate_J (log roughness of 1/sigma^2).
    # `normalize` picks RAW vs mesh-invariant; `obj` picks the objective KIND
    # (1=original | 2=+knot-spacing reg | 3=relative/log-jump; see set_J_objective).
    # Set both on the module flags so the optimiser, band checks and headline J agree.
    set_J_objective(obj)
    set_J_normalize(normalize)
    import io as _io, sys as _sys, time as _time, re as _re
    # heal a leftover run-log tee left behind by a previously-interrupted run
    while getattr(_sys.stdout, '_is_runlog_tee', False):
        _sys.stdout = _sys.stdout.ws[0]
    _run_buf = _io.StringIO()
    class _Tee:
        _is_runlog_tee = True
        def __init__(self, *ws): self.ws = ws
        def write(self, x):
            for w in self.ws:
                w.write(x)
                # flush every write so progress streams live in VS Code / Jupyter
                try: w.flush()
                except Exception: pass
        def flush(self):
            for w in self.ws:
                try: w.flush()
                except Exception: pass
    _stdout_save = _sys.stdout
    _sys.stdout = _Tee(_stdout_save, _run_buf)

    # ── 1. Initial diagnostics (always from build_theta output: R1/R2) ───────
    theta_init = obtain_theta(R1, R2, S0, sigs1, sigs2, nus1, nus2).clone().detach()
    with torch.no_grad():
        J_init, _, _, c1_i, c2_i = loss_fn(R1, R2, S0, theta_init)
        C_init_lgv = generate_call_prices(market_strikes, R1, R2, S0, theta_init, c1_i, c2_i)
        C2_init_an = generate_C2K(market_strikes, R1, R2, S0, theta_init, c1_i, c2_i)
    print("=" * 64); print(f"INITIAL J = {J_init.item():.12g}"); print("=" * 64)

    # ── 2. Optimize ──────────────────────────────────────────────────────────
    _t0 = _time.time()
    if optimize_mode == "sig_nu":
        # sig_nu ALSO densifies: insert new nus (R1_new/R2_new via R1_mult/R2_mult), then
        # run the JOINT AL with ALL of the (denser) nus + sigmas free.  uniform_nus picks
        # uniform-width insertion vs the proportional split.  When R1_new/R2_new are None
        # (or == R1/R2) this is a no-op and we optimise the original build_theta grid.
        _verbose = optimizer_kwargs.get("verbose", True)
        if R1_new is None and R2_new is None:
            R1_final, R2_final = R1, R2
            sigs1_sn, sigs2_sn, nus1_sn, nus2_sn = sigs1, sigs2, nus1, nus2
        else:
            _R1_new = R1_new if R1_new is not None else R1
            _R2_new = R2_new if R2_new is not None else R2
            _Kbar = Kbar_sigonly if Kbar_sigonly is not None else float(market_strikes.max().item())
            sigs1_sn, sigs2_sn, nus1_sn, nus2_sn = build_sigma_only_start(
                bid_ask, S0, sigs1, sigs2, nus1, nus2,
                _R1_new, _R2_new, Kbar=_Kbar, uniform=uniform_nus, verbose=_verbose)
            # uniform insertion gives a data-dependent count -> use the ACTUAL lengths
            R1_final, R2_final = len(sigs1_sn), len(sigs2_sn)
        theta_final = optimize_augmented_lagrangian(
            R1_final, R2_final, S0, sigs1_sn, sigs2_sn, nus1_sn, nus2_sn,
            loss_fn, generate_call_prices, bid_ask, **optimizer_kwargs)
    else:  # sig_only or sig_only_LKbar  (both densify nus, then optimise sigmas)
        _R1_new = R1_new if R1_new is not None else R1 * 2
        _R2_new = R2_new if R2_new is not None else R2 * 2
        _Kbar = Kbar_sigonly if Kbar_sigonly is not None else float(market_strikes.max().item())
        _verbose = optimizer_kwargs.get("verbose", True)
        sigs1_so, sigs2_so, nus1_so, nus2_so = build_sigma_only_start(
            bid_ask, S0, sigs1, sigs2, nus1, nus2,
            _R1_new, _R2_new, Kbar=_Kbar, uniform=uniform_nus, verbose=_verbose)
        # uniform insertion gives a data-dependent count -> use the ACTUAL lengths
        R1_final, R2_final = len(sigs1_so), len(sigs2_so)
        if optimize_mode == "sig_only_LKbar":
            # also free the two extreme boundary knots L=nus1[0], Kbar=nus2[-1]
            theta_final = optimize_sigma_only_LKbar(
                _R1_new, _R2_new, S0, sigs1_so, sigs2_so, nus1_so, nus2_so,
                loss_fn, generate_call_prices, bid_ask, **optimizer_kwargs)
        else:
            theta_final = optimize_sigma_only(
                _R1_new, _R2_new, S0, sigs1_so, sigs2_so, nus1_so, nus2_so,
                loss_fn, generate_call_prices, bid_ask, **optimizer_kwargs)
    _run_time = _time.time() - _t0

    # ── 3. Final diagnostics + constraint verification ───────────────────────
    with torch.no_grad():
        J_final, _, _, c1_f, c2_f = loss_fn(R1_final, R2_final, S0, theta_final)
        C_final    = generate_call_prices(market_strikes, R1_final, R2_final, S0, theta_final, c1_f, c2_f)
        C2_final_an = generate_C2K(market_strikes, R1_final, R2_final, S0, theta_final, c1_f, c2_f)
    print(f"\nFINAL J   = {J_final.item():.12g}")
    pct_improve = ((J_init.item() - J_final.item()) / abs(J_init.item()) * 100.0
                   if J_init.item() != 0 else float("nan"))
    true_reduce = true_reduction_pct(J_init.item(), J_final.item())
    print(f"J IMPROVEMENT:  {pct_improve:.2f}% (log-J)   "
          f"(J_init={J_init.item():.6g} -> J_final={J_final.item():.6g})")
    print(f"TRUE ROUGHNESS REDUCTION:  {true_reduce:.2f}%   "
          f"(S=exp(J): {np.exp(J_init.item()):.6g} -> {np.exp(J_final.item()):.6g})")
    S_final     = float(np.exp(J_final.item()))          # absolute final roughness Sum (Du)^2[/Dnu]
    rms_final   = float(np.exp(J_final.item() / 2.0))    # sqrt(S_final): L2 magnitude of the jumps
    print(f"ABSOLUTE FINAL ROUGHNESS:  S_final={S_final:.6g}   sqrt(S_final)={rms_final:.6g}")
    verify_solution(theta_final, R1_final, R2_final, S0, bid_ask)

    # ── 4. Numpy arrays for plotting ─────────────────────────────────────────
    nus1_i, sigs1_i, nus2_i, sigs2_i = extract_nus_sigs(theta_init,  R1,       R2)
    nus1_f, sigs1_f, nus2_f, sigs2_f = extract_nus_sigs(theta_final, R1_final, R2_final)
    strikes_np = market_strikes.detach().cpu().numpy()
    bids_np    = market_bids.detach().cpu().numpy()
    asks_np    = market_asks.detach().cpu().numpy()
    C_final_np = C_final.detach().cpu().numpy()
    nus_all_i  = np.concatenate([nus1_i, nus2_i]); sigs_all_i = np.concatenate([sigs1_i, sigs2_i])
    nus_all_f  = np.concatenate([nus1_f, nus2_f]); sigs_all_f = np.concatenate([sigs1_f, sigs2_f])

    if arb_prices is not None:
        arb_s_np = (arb_strikes.detach().cpu().numpy() if torch.is_tensor(arb_strikes)
                    else np.asarray(arb_strikes, dtype=float))
        arb_p_np = (arb_prices.detach().cpu().numpy()  if torch.is_tensor(arb_prices)
                    else np.asarray(arb_prices,  dtype=float))
        C_init_np  = np.interp(strikes_np, arb_s_np, arb_p_np)
        init_label = "Arb-free C(K)"
    else:
        C_init_np  = C_init_lgv.detach().cpu().numpy()
        init_label = "Initial C(K)"

    d2C_init   = compute_second_derivative(
        torch.tensor(C_init_np, dtype=theta_init.dtype), market_strikes)
    d2C_final  = compute_second_derivative(C_final, market_strikes)
    logC_init  = np.log(np.clip(C_init_np,  1e-12, None))
    logC_final = np.log(np.clip(C_final_np, 1e-12, None))
    c2_init_np  = C2_init_an.detach().cpu().numpy()
    c2_final_np = C2_final_an.detach().cpu().numpy()

    mode_label = {"sig_nu": f"Sig+Nu | R1={R1}, R2={R2}",
                  "sig_only": f"Sigma-only | R1={R1_final}, R2={R2_final}",
                  "sig_only_LKbar": f"Sigma + L/Kbar | R1={R1_final}, R2={R2_final}"}[optimize_mode]

    # consistent suptitle padding across figures of different heights: reserve a
    # fixed ABSOLUTE band (in inches) for the title so the gap looks identical on
    # the tall summary figure and the short quote/smoothing figures alike.
    def _padded_suptitle(f, text, pad_in=0.5, fontsize=14):
        import warnings as _warnings
        h = f.get_size_inches()[1]
        frac = pad_in / h
        with _warnings.catch_warnings():
            _warnings.simplefilter("ignore", UserWarning)
            f.tight_layout(rect=[0, 0, 1, 1 - frac])
        f.suptitle(text, fontsize=fontsize, fontweight="bold", y=1 - 0.25 / h)

    # ── 5. Plot suite A-G ────────────────────────────────────────────────────
    fig = plt.figure(figsize=(18, 34))
    gs  = gridspec.GridSpec(5, 2, figure=fig, hspace=0.40, wspace=0.30)

    ax = fig.add_subplot(gs[0, 0])
    ax.plot(nus_all_i, sigs_all_i, "o-", color="steelblue", ms=4, label=f"Initial (R1={R1}, R2={R2})")
    ax.plot(nus_all_f, sigs_all_f, "s-", color="tomato",    ms=4, label=f"Final (R1={R1_final}, R2={R2_final})")
    ax.axvline(S0, color="gray", ls="--", lw=0.8, label=f"S0={S0}")
    ax.set_xlabel("ν (knot location)"); ax.set_ylabel("σ"); ax.set_title("A: σ vs ν")
    ax.legend(fontsize=8); ax.grid(alpha=0.3)

    ax = fig.add_subplot(gs[0, 1])
    ax.plot(sigs_all_i, "o-", color="steelblue", ms=4, label=f"Initial (R1={R1}, R2={R2})")
    ax.plot(sigs_all_f, "s-", color="tomato",    ms=4, label=f"Final (R1={R1_final}, R2={R2_final})")
    ax.axvline(R1_final - 0.5, color="gray", ls="--", lw=0.8, label=f"S0 boundary (idx {R1_final})")
    ax.set_xlabel("Index"); ax.set_ylabel("σ"); ax.set_title("B: σ vs Index")
    ax.legend(fontsize=8); ax.grid(alpha=0.3)

    ax = fig.add_subplot(gs[1, 0])
    ax.fill_between(strikes_np, bids_np, asks_np, alpha=0.25, color="green", label="Bid-Ask spread")
    ax.plot(strikes_np, bids_np,    "g--", lw=0.8, label="Bid")
    ax.plot(strikes_np, asks_np,    "g:",  lw=0.8, label="Ask")
    ax.plot(strikes_np, C_init_np,  "o-", color="steelblue", ms=3, label=init_label)
    ax.plot(strikes_np, C_final_np, "s-", color="tomato",    ms=3, label="Final C(K)")
    ax.axvline(S0, color="gray", ls="--", lw=0.8)
    ax.set_xlabel("Strike K"); ax.set_ylabel("C(K)"); ax.set_title("C: Call Prices C(K)")
    ax.legend(fontsize=8); ax.grid(alpha=0.3)

    ax = fig.add_subplot(gs[1, 1])
    log_b = np.log(np.clip(bids_np, 1e-12, None)); log_a = np.log(np.clip(asks_np, 1e-12, None))
    ax.fill_between(strikes_np, log_b, log_a, alpha=0.25, color="green", label="log Bid-Ask")
    ax.plot(strikes_np, logC_init,  "o-", color="steelblue", ms=3, label=f"Initial log C(K)  [{init_label}]")
    ax.plot(strikes_np, logC_final, "s-", color="tomato",    ms=3, label="Final log C(K)")
    ax.axvline(S0, color="gray", ls="--", lw=0.8)
    ax.set_xlabel("Strike K"); ax.set_ylabel("log C(K)"); ax.set_title("D: log C(K)")
    ax.legend(fontsize=8); ax.grid(alpha=0.3)

    ax = fig.add_subplot(gs[2, 0])
    ax.plot(strikes_np, d2C_init,  "o-", color="steelblue", ms=3, label=f"Initial C''(K)  [{init_label}]")
    ax.plot(strikes_np, d2C_final, "s-", color="tomato",    ms=3, label="Final C''(K)")
    ax.axhline(0, color="black", lw=0.6, ls="--"); ax.axvline(S0, color="gray", ls="--", lw=0.8)
    ax.set_xlabel("Strike K"); ax.set_ylabel("C''(K)"); ax.set_title("E: Second Derivative C''(K)")
    ax.legend(); ax.grid(alpha=0.3)

    ax = fig.add_subplot(gs[2, 1])
    pos_i = c2_init_np  > 0
    pos_f = c2_final_np > 0
    ax.plot(strikes_np[pos_i], np.log(c2_init_np[pos_i]),  "o-", color="steelblue", ms=3,
            label="Initial log C''(K)  [LVG]")
    ax.plot(strikes_np[pos_f], np.log(c2_final_np[pos_f]), "s-", color="tomato",    ms=3,
            label="Final log C''(K)")
    if np.any(~pos_f):
        # floor for the C''<=0 markers; guard the all-nonpositive case (pos_f empty)
        _yfloor = float(np.log(c2_final_np[pos_f]).min()) if pos_f.any() else 0.0
        ax.scatter(strikes_np[~pos_f], np.full((~pos_f).sum(), _yfloor),
                   marker="x", color="black", zorder=5, label="C''<=0 (final)")
    ax.axvline(S0, color="gray", ls="--", lw=0.8)
    ax.set_xlabel("Strike K"); ax.set_ylabel("log C''(K)")
    ax.set_title("G: log C''(K)  (analytical, LVG model)")
    ax.legend(fontsize=8); ax.grid(alpha=0.3)

    ax = fig.add_subplot(gs[3, 0])
    ax.bar(["Initial J", "Final J"], [J_init.item(), J_final.item()],
           color=["steelblue", "tomato"], width=0.4)
    ax.set_ylabel("J")
    ax.set_title(f"F: J  {J_init.item():.4e} -> {J_final.item():.4e}   ")
    ax.text(0.5, 0.92, f"log-J {pct_improve:.2f}%  |  true {true_reduce:.2f}%", transform=ax.transAxes,
            ha="center", fontsize=11, fontweight="bold", color="darkgreen")
    ax.grid(alpha=0.3, axis="y")
    for i, v in enumerate([J_init.item(), J_final.item()]):
        ax.text(i, v * 1.01, f"{v:.3e}", ha="center", fontsize=9)

    ax = fig.add_subplot(gs[3, 1])
    ax.plot(nus_all_i, sigs_all_i, "o-", color="steelblue", ms=4, label=f"Initial (R1={R1}, R2={R2})")
    ax.plot(nus_all_f, sigs_all_f, "s-", color="tomato",    ms=4, label=f"Final (R1={R1_final}, R2={R2_final})")
    ax.axvline(S0, color="gray", ls="--", lw=0.8, label=f"S0={S0}")
    if len(nus_all_f) > 5:
        x_mid = np.concatenate([nus_all_i[2:-2], nus_all_f[2:-2]])
        y_mid = np.concatenate([sigs_all_i[2:-2], sigs_all_f[2:-2]])
        xpad = 0.02 * (x_mid.max() - x_mid.min() + 1e-9)
        ypad = 0.08 * (y_mid.max() - y_mid.min() + 1e-9)
        ax.set_xlim(x_mid.min() - xpad, x_mid.max() + xpad)
        ax.set_ylim(y_mid.min() - ypad, y_mid.max() + ypad)
    ax.set_xlabel("ν (knot location)"); ax.set_ylabel("σ")
    ax.set_title("A2: σ vs ν   (zoomed)")
    ax.legend(fontsize=8); ax.grid(alpha=0.3)

    # ── Panels (A) fit-quality vs T and (B) ATM local-variance vs T (this date) ──
    # For a single date these carry ONE point each (this T); across-date trends live
    # in the compare image.  ATM sigma^2 = V/C'' at K=S0 (log-moneyness x=0): since the
    # intrinsic value is 0 at K=S0, V(S0)=C(S0), so sigma^2(S0)=C(S0)/C''(S0).
    with torch.no_grad():
        _kS0 = torch.tensor([float(S0)], dtype=theta_init.dtype)
        _Ci  = generate_call_prices(_kS0, R1, R2, S0, theta_init, c1_i, c2_i)
        _C2i = generate_C2K(_kS0, R1, R2, S0, theta_init, c1_i, c2_i)
        _Cf  = generate_call_prices(_kS0, R1_final, R2_final, S0, theta_final, c1_f, c2_f)
        _C2f = generate_C2K(_kS0, R1_final, R2_final, S0, theta_final, c1_f, c2_f)
    def _atm_sig2(C, C2):
        C = float(np.asarray(C).reshape(-1)[0]); C2 = float(np.asarray(C2).reshape(-1)[0])
        return C / C2 if C2 > 1e-30 else float("nan")    # V(S0)=C(S0); sigma^2 = V/C''
    sig2_S0_init = _atm_sig2(_Ci, _C2i)
    sig2_S0_fin  = _atm_sig2(_Cf, _C2f)

    # (A) fit quality vs time-to-expiry
    ax = fig.add_subplot(gs[4, 0])
    ax.scatter([T], [true_reduce], s=110, color="tomato", zorder=3)
    ax.annotate(f"T={T:.3f} yr\ntrue reduction = {true_reduce:.2f}%\n"
                f"log-J = {pct_improve:.2f}%\nS_final = {S_final:.4g}",
                (T, true_reduce), fontsize=9, textcoords="offset points", xytext=(10, -6))
    ax.set_xlabel("time to expiry  T  (years)")
    ax.set_ylabel("TRUE roughness reduction (%)")
    ax.set_title("(A) fit quality vs time-to-expiry  (this date)")
    ax.grid(alpha=0.3)

    # (B) ATM local-variance term-structure point: sigma^2(S0) before vs after
    ax = fig.add_subplot(gs[4, 1])
    ax.scatter([T], [sig2_S0_init], s=110, facecolors="none", edgecolors="steelblue",
               linewidths=2, label="before")
    ax.scatter([T], [sig2_S0_fin], s=110, color="tomato", label="after")
    if np.isfinite(sig2_S0_init) and np.isfinite(sig2_S0_fin):
        ax.plot([T, T], [sig2_S0_init, sig2_S0_fin], color="gray", ls=":", lw=1)
    ax.set_yscale("log")
    ax.set_xlabel("time to expiry  T  (years)")
    ax.set_ylabel("ATM  V/C''  = σ²(S0)   (log)")
    ax.set_title("(B) ATM local variance vs T  (this date)")
    ax.annotate(f"before = {sig2_S0_init:.4g}\nafter = {sig2_S0_fin:.4g}",
                (T, sig2_S0_fin), fontsize=9, textcoords="offset points", xytext=(10, 0))
    ax.grid(alpha=0.3, which="both"); ax.legend(fontsize=8)

    stamp = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
    os.makedirs(out_dir, exist_ok=True)
    _op = lambda nm: os.path.join(out_dir, nm)    # route every saved file into out_dir
    _mult_lbl = f" | mult={R1_mult}x{R2_mult}" if (R1_mult is not None or R2_mult is not None) else ""
    _padded_suptitle(fig, f"Optimization Summary | {mode_label} | obj={obj}{_mult_lbl} | S0={S0} | start={start_date} | expiry={expiry_date}")

    # ── 6b. SMOOTHING DIAGNOSTICS ─────────────────────────────────────────────
    K_fine = np.linspace(strikes_np.min(), strikes_np.max(), 600)
    K_fine_t = torch.tensor(K_fine, dtype=theta_final.dtype)
    with torch.no_grad():
        c2f_init  = generate_C2K(K_fine_t, R1,       R2,       S0, theta_init,  c1_i, c2_i).cpu().numpy()
        c2f_final = generate_C2K(K_fine_t, R1_final, R2_final, S0, theta_final, c1_f, c2_f).cpu().numpy()
        Cf_init   = generate_call_prices(K_fine_t, R1,       R2,       S0, theta_init,  c1_i, c2_i).cpu().numpy()
        Cf_final  = generate_call_prices(K_fine_t, R1_final, R2_final, S0, theta_final, c1_f, c2_f).cpu().numpy()
    intrinsic_fine = np.maximum(S0 - K_fine, 0.0)
    Vf_init  = Cf_init  - intrinsic_fine
    Vf_final = Cf_final - intrinsic_fine

    def knot_jumps(theta, c1, c2, nus1_k, nus2_k, R1_, R2_, delta=1e-3):
        knots = np.unique(np.concatenate([nus1_k, [S0], nus2_k]))
        knots = knots[(knots > strikes_np.min()) & (knots < strikes_np.max())]
        with torch.no_grad():
            jl = generate_C2K(torch.tensor(knots - delta, dtype=theta.dtype),
                               R1_, R2_, S0, theta, c1, c2).cpu().numpy()
            jr = generate_C2K(torch.tensor(knots + delta, dtype=theta.dtype),
                               R1_, R2_, S0, theta, c1, c2).cpu().numpy()
        return knots, np.abs(jr - jl)

    kn_i, jmp_i = knot_jumps(theta_init,  c1_i, c2_i, nus1_i, nus2_i, R1,       R2)
    kn_f, jmp_f = knot_jumps(theta_final, c1_f, c2_f, nus1_f, nus2_f, R1_final, R2_final)

    figS, axS = plt.subplots(3, 2, figsize=(16, 16))
    ax = axS[0, 0]
    ax.plot(K_fine, c2f_init,  "-", color="steelblue", lw=1.6, label="Initial C''(K)  [LVG]")
    ax.plot(K_fine, c2f_final, "-", color="tomato",    lw=1.6, label="Final C''(K)")
    ax.axvline(S0, color="gray", ls="--", lw=0.8); ax.axhline(0, color="k", lw=0.6)
    ax.set_title("Analytical C''(K)")
    ax.set_xlabel("Strike K"); ax.set_ylabel("C''(K)"); ax.legend(fontsize=8); ax.grid(alpha=0.3)

    ax = axS[0, 1]
    pi = c2f_init > 0; pf = c2f_final > 0
    ax.semilogy(K_fine[pi], c2f_init[pi],  "-", color="steelblue", lw=1.4, label="Initial  [LVG]")
    ax.semilogy(K_fine[pf], c2f_final[pf], "-", color="tomato",    lw=1.4, label="Final")
    ax.axvline(S0, color="gray", ls="--", lw=0.8)
    ax.set_title("Analytical C''(K), log scale")
    ax.set_xlabel("Strike K"); ax.set_ylabel("C''(K)  (log)"); ax.legend(fontsize=8); ax.grid(alpha=0.3, which="both")

    ax = axS[1, 0]
    ax.vlines(kn_i, 0, jmp_i, color="steelblue", alpha=0.6, lw=2, label="Initial |ΔC''|  [LVG]")
    ax.vlines(kn_f + 0.0, 0, jmp_f, color="tomato", alpha=0.6, lw=2, label="Final |ΔC''|")
    ax.plot(kn_i, jmp_i, "o", color="steelblue", ms=3); ax.plot(kn_f, jmp_f, "s", color="tomato", ms=3)
    ax.axvline(S0, color="gray", ls="--", lw=0.8)
    ax.set_title("Per-knot jump |C''(ν+) − C''(ν−)|")
    ax.set_xlabel("knot ν"); ax.set_ylabel("|jump in C''|"); ax.legend(fontsize=8); ax.grid(alpha=0.3)

    ax = axS[1, 1]
    si = np.sort(jmp_i**2)[::-1]; sf = np.sort(jmp_f**2)[::-1]
    ax.plot(np.arange(si.size), si, "o-", color="steelblue", ms=3, label=f"Initial  Σjump²={si.sum():.2e}  [LVG]")
    ax.plot(np.arange(sf.size), sf, "s-", color="tomato",    ms=3, label=f"Final    Σjump²={sf.sum():.2e}")
    ax.set_yscale("log")
    ax.set_title("Squared C'' jumps, sorted")
    ax.set_xlabel("knot rank"); ax.set_ylabel("jump²  (log)"); ax.legend(fontsize=8); ax.grid(alpha=0.3, which="both")

    ax = axS[2, 0]
    mi = c2f_init > 0; mf = c2f_final > 0
    ax.semilogy(K_fine[mi], np.clip(Vf_init[mi]  / c2f_init[mi],  1e-30, None), "-", color="steelblue", lw=1.4, label="Initial V/C''  [LVG]")
    ax.semilogy(K_fine[mf], np.clip(Vf_final[mf] / c2f_final[mf], 1e-30, None), "-", color="tomato",    lw=1.4, label="Final V/C''")
    ax.axvline(S0, color="gray", ls="--", lw=0.8)
    ax.set_title("Analytical  V(K) / C''(K)")
    ax.set_xlabel("Strike K"); ax.set_ylabel("V / C''  (log)"); ax.legend(fontsize=8); ax.grid(alpha=0.3, which="both")

    ax = axS[2, 1]
    ni = (c2f_init > 0) & (Vf_init > 0); nf = (c2f_final > 0) & (Vf_final > 0)
    ax.semilogy(K_fine[ni], np.clip(c2f_init[ni]  / Vf_init[ni],  1e-30, None), "-", color="steelblue", lw=1.4, label="Initial C''/V  [LVG]")
    ax.semilogy(K_fine[nf], np.clip(c2f_final[nf] / Vf_final[nf], 1e-30, None), "-", color="tomato",    lw=1.4, label="Final C''/V")
    ax.axvline(S0, color="gray", ls="--", lw=0.8)
    ax.set_title("Analytical  C''(K) / V(K)")
    ax.set_xlabel("Strike K"); ax.set_ylabel("C'' / V  (log)"); ax.legend(fontsize=8); ax.grid(alpha=0.3, which="both")
    _padded_suptitle(figS, "Smoothing diagnostics")

    # ── 5b. OBSERVED market bid/ask ───────────────────────────────────────────
    figQ, axQ = plt.subplots(1, 2, figsize=(16, 6))
    mb = bids_np > 0
    ax = axQ[0]
    ax.scatter(strikes_np, bids_np, s=22, color="green",     marker="v", label="Market bid")
    ax.scatter(strikes_np, asks_np, s=22, color="firebrick", marker="^", label="Market ask")
    ax.plot(strikes_np, C_init_np,  "o-", color="steelblue", ms=3, label=init_label)
    ax.plot(strikes_np, C_final_np, "s-", color="tomato",    ms=3, label="Final C(K)")
    ax.axvline(S0, color="gray", ls="--", lw=0.8, label=f"S0={S0}")
    ax.set_xlabel("Strike K"); ax.set_ylabel("C(K)")
    ax.set_title("Observed market bid/ask  C(K)")
    ax.legend(fontsize=8); ax.grid(alpha=0.3)

    ax = axQ[1]
    log_b = np.log(np.clip(bids_np, 1e-12, None))
    log_a = np.log(np.clip(asks_np, 1e-12, None))
    ax.scatter(strikes_np[mb], log_b[mb], s=22, color="green",     marker="v", label="Market bid")
    ax.scatter(strikes_np,     log_a,     s=22, color="firebrick", marker="^", label="Market ask")
    ax.plot(strikes_np, logC_init,  "o-", color="steelblue", ms=3, label=f"Initial log C(K)  [{init_label}]")
    ax.plot(strikes_np, logC_final, "s-", color="tomato",    ms=3, label="Final log C(K)")
    ax.set_yscale("log"); ax.axvline(S0, color="gray", ls="--", lw=0.8)
    ax.set_ylabel("C(K)   (log scale)")
    ax.set_title("Observed market bid/ask  log C(K)")
    ax.legend(fontsize=8); ax.grid(alpha=0.3, which="both")
    _padded_suptitle(figQ, "Observed market quotes")
    # (figure `figQ` collected into diagnostics_{stamp}.png below)

    # ── 6. Implied-volatility smile ───────────────────────────────────────────
    # Hwk 5b IV smile: pass prices (df=1, forward=S0 done inside), x-axis = log-moneyness
    fig_iv = plot_implied_vols(strikes_np, C_init_np, C_final_np, S0,
                               bids_np=bids_np, asks_np=asks_np, T=T, save=False)

    # ── 6b. Optimizer convergence (a) + per-knot J decomposition (b) ──────────
    # convergence traces parsed from the captured optimizer log (_run_buf)
    fig_conv = plot_convergence(_run_buf.getvalue(), save=False)
    # per-knot J contribution, initial vs final theta
    fig_jk   = plot_j_per_knot(R1, R2, theta_init, R1_final, R2_final, theta_final, S0, save=False)

    # ── 6c. Merge ALL panels into ONE diagnostics image ──────────────────────
    try:
        import io as _io2
        from PIL import Image as _PILImage
        def _fig_png(f):
            _b = _io2.BytesIO(); f.savefig(_b, format='png', dpi=130, bbox_inches='tight')
            _b.seek(0); return _PILImage.open(_b).convert('RGB')
        _imgs = [_fig_png(fig), _fig_png(figS), _fig_png(figQ), _fig_png(fig_iv), _fig_png(fig_conv), _fig_png(fig_jk)]
        _W = max(im.width for im in _imgs)
        _scaled = [im if im.width == _W else im.resize((_W, round(im.height*_W/im.width))) for im in _imgs]
        _H = sum(im.height for im in _scaled)
        _canvas = _PILImage.new('RGB', (_W, _H), 'white')
        _y = 0
        for im in _scaled:
            _canvas.paste(im, (0, _y)); _y += im.height
        _canvas.save(_op(f'diagnostics_{stamp}.png'))
        print(f'Saved {_op(f"diagnostics_{stamp}.png")}  ({_W}x{_H})')
    except Exception as _e:
        for _f, _nm in ((fig,'optimization_summary'),(figS,'smoothing_diagnostics'),
                        (figQ,'observed_quotes'),(fig_iv,'implied_volatility')):
            _f.savefig(_op(f'{_nm}_{stamp}.png'), dpi=150, bbox_inches='tight')
        print(f'[diagnostics] PIL merge failed ({_e}); saved 4 separate PNGs as fallback')
    for _f in (fig, figS, figQ, fig_iv):
        plt.close(_f)

    # Render the merged image inline so it shows in VS Code / Jupyter
    # (figures were closed above; we display the saved PNG instead).
    try:
        import os as _os
        from IPython.display import Image as _IPImage, display as _ipdisplay
        _diag = _op(f'diagnostics_{stamp}.png')
        if _os.path.exists(_diag):
            _ipdisplay(_IPImage(filename=_diag))
        else:
            # PIL-merge fallback path saved 4 separate PNGs; show each
            for _nm in ('optimization_summary', 'smoothing_diagnostics',
                        'observed_quotes', 'implied_volatility'):
                _fp = _op(f'{_nm}_{stamp}.png')
                if _os.path.exists(_fp):
                    _ipdisplay(_IPImage(filename=_fp))
    except Exception:
        pass

    # ── 7. Print parameters and export CSV ───────────────────────────────────
    np.set_printoptions(threshold=np.inf, linewidth=200, precision=12, suppress=False)
    print("\n" + "=" * 64)
    print("FINAL FITTED PARAMETERS")
    print("=" * 64)
    print("final_nus1  =", np.array2string(nus1_f,  separator=", "))
    print("final_sigs1 =", np.array2string(sigs1_f, separator=", "))
    print("final_nus2  =", np.array2string(nus2_f,  separator=", "))
    print("final_sigs2 =", np.array2string(sigs2_f, separator=", "))

    spread_np = asks_np - bids_np
    param_init_df = pd.DataFrame({
        "record_type": "param_init",
        "side": (["L"] * R1) + (["R"] * R2),
        "idx":  list(range(R1)) + list(range(R2)),
        "nu": nus_all_i, "sigma": sigs_all_i,
    })
    param_final_df = pd.DataFrame({
        "record_type": "param_final",
        "side": (["L"] * R1_final) + (["R"] * R2_final),
        "idx":  list(range(R1_final)) + list(range(R2_final)),
        "nu": nus_all_f, "sigma": sigs_all_f,
        "spread_at_knot": np.interp(nus_all_f, strikes_np, spread_np),
    })
    in_band = (C_final_np >= bids_np - 1e-6) & (C_final_np <= asks_np + 1e-6)
    market_df = pd.DataFrame({
        "record_type": "market", "strike": strikes_np,
        "bid": bids_np, "ask": asks_np, "spread": spread_np,
        "C_initial": C_init_np, "C_final": C_final_np, "in_band": in_band,
    })
    _outers = [int(m) for m in _re.findall(r"Outer\s+(\d+)/", _run_buf.getvalue())]
    _n_outer = max(_outers) if _outers else 0
    meta_df = pd.DataFrame({
        "record_type": "meta",
        "key":   ["R1_init", "R2_init", "R1_final", "R2_final", "S0",
                  "J_initial", "J_final", "J_pct_improvement", "true_roughness_reduction_pct",
                  "S_final", "sqrt_S_final",
                  "objective", "normalize",
                  "init_price_source", "optimize_mode",
                  "dataset_start_date", "dataset_expiry_date",
                  "run_time_sec", "n_outer_iterations"],
        "value": [R1, R2, R1_final, R2_final, S0,
                  float(J_init.item()), float(J_final.item()), float(pct_improve), float(true_reduce),
              float(S_final), float(rms_final),
                  int(obj), bool(normalize),
                  init_label, optimize_mode,
                  start_date, expiry_date,
                  float(_run_time), _n_outer],
    })
    out_df = pd.concat([meta_df, param_init_df, param_final_df, market_df], ignore_index=True)

    csv_name = _op(f"fit_results_{stamp}_{optimize_mode}.csv")
    out_df.to_csv(csv_name, index=False)
    print(f"\nSaved fit results CSV -> {os.path.abspath(csv_name)}")

    # google.colab auto-download removed (cleanup 2026-06-23); on local Jupyter
    # we only show a download link -- identical to the previous local behaviour.
    try:
        from IPython.display import display, FileLink
        display(FileLink(csv_name))
    except Exception:
        pass

    # ── 8. Write run log (build_theta prints + this run's full stdout) ───────
    try:
        _blines = build_log if build_log is not None else globals().get('BUILD_THETA_LOG', [])
        _logname = _op(f'run_log_{stamp}.txt')
        with open(_logname, 'w') as _lf:
            _lf.write(f'start_date={start_date}  expiry_date={expiry_date}\n')
            _lf.write(f'optimize_mode={optimize_mode}  R1={R1} R2={R2} -> R1_final={R1_final} R2_final={R2_final}\n')
            _lf.write(f'run_time_sec={_run_time:.2f}  n_outer_iterations={_n_outer}\n')
            _lf.write(f'J_initial={J_init.item():.10g}  J_final={J_final.item():.10g}  improvement={pct_improve:.4f}% (log-J)  true_reduction={true_reduce:.4f}%\n')
            _lf.write('='*72 + '\n[build_theta log]\n')
            _lf.write('\n'.join(str(x) for x in _blines) + '\n')
            _lf.write('='*72 + '\n[run_and_plot log]\n')
            _lf.write(_run_buf.getvalue())
        print(f'Saved {_logname} -> {os.path.abspath(_logname)}')
    except Exception as _e:
        print(f'[run_log] failed: {_e}')
    finally:
        _sys.stdout = _stdout_save

    return theta_final


In [ ]:
def load_market_data(quotes_path, arb_free_path):
    """Original loader for the real CSVs (kept for convenience)."""
    quotes_df = pd.read_csv(quotes_path, header=None)
    strikes = quotes_df.iloc[:, 0].values
    market_asks = quotes_df.iloc[:, 1].values
    market_bids = quotes_df.iloc[:, 2].values
    bid_ask = torch.tensor(np.column_stack((strikes, market_bids, market_asks)),
                           dtype=torch.float64)
    arb_df = pd.read_csv(arb_free_path, header=None)
    market_prices = torch.tensor(arb_df.iloc[:, 0].values, dtype=torch.float64)
    arb_strikes   = torch.tensor(arb_df.iloc[:, 1].values, dtype=torch.float64)
    return bid_ask, market_prices, arb_strikes

In [ ]:

# ════════════════════════════════════════════════════════════════════════════
# build_theta
# ════════════════════════════════════════════════════════════════════════════
# Every production guard that protects against NaN / overflow / non-monotone
# output is present but COMMENTED OUT, tagged  # >>> FIX x.  Each fix block says
# exactly which HW5b line(s) to comment out and what the fix prevents.  To get a
# working build_theta for long-dated / all-strike datasets, uncomment the FIX
# block and comment the HW5b line it replaces (noted in each block).
#
# Interface glue (not in HW5b, required to return optimiser tensors) is tagged
# # [interface].  Prints go through _blog so run_and_plot can fold them into
# run_log_{stamp}.txt (passed explicitly via build_log=BUILD_THETA_LOG).

BUILD_THETA_LOG = []          # [interface] last build_theta() run's printed lines
BUILD_THETA_DEBUG = True       # [interface] extra gap / non-finite diagnostics
def _blog(msg):               # [interface] print + remember
    print(msg, flush=True)
    BUILD_THETA_LOG.append(str(msg))


def Atilde(sigma,A,B,w,K):
    return 0.5*(A-sigma*B)*np.exp(-(w-K)/sigma) + 0.5*(A+sigma*B)*np.exp((w-K)/sigma)

def Btilde(sigma,A,B,w,K):
    return -(0.5/sigma)*(A-sigma*B)*np.exp(-(w-K)/sigma) + (0.5/sigma)*(A+sigma*B)*np.exp((w-K)/sigma)

def findSigmaHat(A,B,w,K1,K2,V):
    my_eps = 1.0e-8
    def SigmaHat_obj(sigma):
        return Atilde(sigma,A,B,w,K1) + Btilde(sigma,A,B,w,K1)*(K2-w) - V
    low = my_eps
    high = 1.0
    # >>> FIX A (expansion overflow / non-termination) ------------------------
    # HW5b doubles `high` with no bound.  If the objective never crosses 0
    # (deep-OTM / long-dated pieces) high -> inf, np.exp overflows to inf->nan,
    # and `nan > 0` is False so the loop exits with high=nan -> sigma=nan ->
    # nan theta -> nan J.  To enable: comment the next HW5b line and uncomment
    # the g-capped version below it.
    g = 0
    while (SigmaHat_obj(high) > 0) and g < 100:
        high = 2.0*high; g += 1
    # g = 0
    # while (SigmaHat_obj(high) > 0) and g < 100:
    #     high = 2.0*high; g += 1
    # ------------------------------------------------------------------------
    # >>> FIX B (bisection non-termination on nan) ---------------------------
    # If SigmaHat_obj(mid) is nan, neither branch fires deterministically.  To
    # enable: comment the next HW5b while-block and uncomment the guarded one.
    k = 0
    while (high - low > my_eps) and k < 200:
        mid = 0.5*(low+high); fm = SigmaHat_obj(mid)
        if not np.isfinite(fm): break
        if fm >= 0: low = mid
        else: high = mid
        k += 1
    # k = 0
    # while (high - low > my_eps) and k < 200:
    #     mid = 0.5*(low+high); fm = SigmaHat_obj(mid)
    #     if not np.isfinite(fm): break
    #     if fm >= 0: low = mid
    #     else: high = mid
    #     k += 1
    # ------------------------------------------------------------------------
    return 0.5 * (low + high)

def SigmaTilde(sigma,A,B,w,K1,K2,V):
    my_eps = 1.0e-8
    At = Atilde(sigma,A,B,w,K1)
    Bt = Btilde(sigma,A,B,w,K1)
    def SigmaTilde_obj(sigma_t):
        return 0.5*(At+sigma_t*Bt)*np.exp((K2-w)/sigma_t) + 0.5*(At-sigma_t*Bt)*np.exp(-(K2-w)/sigma_t) - V
    low = my_eps
    high = 1
    # >>> FIX A (same as findSigmaHat) ---------------------------------------
    g = 0
    while (SigmaTilde_obj(high) > 0) and g < 100:
        high = 2.0*high; g += 1
    # g = 0
    # while (SigmaTilde_obj(high) > 0) and g < 100:
    #     high = 2.0*high; g += 1
    # ------------------------------------------------------------------------
    # >>> FIX B (same as findSigmaHat) ---------------------------------------
    k = 0
    while (high - low > my_eps) and k < 200:
        mid = 0.5*(low+high); fm = SigmaTilde_obj(mid)
        if not np.isfinite(fm): break
        if fm >= 0: low = mid
        else: high = mid
        k += 1
    # k = 0
    # while (high - low > my_eps) and k < 200:
    #     mid = 0.5*(low+high); fm = SigmaTilde_obj(mid)
    #     if not np.isfinite(fm): break
    #     if fm >= 0: low = mid
    #     else: high = mid
    #     k += 1
    # ------------------------------------------------------------------------
    return 0.5 * (low + high)

def FindSigma(A,B,w,K1,K2,V,B1,sigma_h):
    my_eps = 1.0e-8
    def FindSigma_obj(sigma):
        sigma_t = SigmaTilde(sigma,A,B,w,K1,K2,V)
        At = Atilde(sigma,A,B,w,K1)
        Bt = Btilde(sigma,A,B,w,K1)
        return B1 - (0.5/sigma_t)*(At + sigma_t*Bt)*np.exp((K2-w)/sigma_t) + (0.5/sigma_t)*(At - sigma_t*Bt)*np.exp(-(K2-w)/sigma_t)
    low = sigma_h
    high = sigma_h+1
    # >>> FIX A (same as findSigmaHat) ---------------------------------------
    g = 0
    while (FindSigma_obj(high) > 0) and g < 100:
        high = 2.0*high; g += 1
    # g = 0
    # while (FindSigma_obj(high) > 0) and g < 100:
    #     high = 2.0*high; g += 1
    # ------------------------------------------------------------------------
    # >>> FIX B (same as findSigmaHat) ---------------------------------------
    k = 0
    while (high - low > my_eps) and k < 200:
        mid = 0.5*(low+high); fm = FindSigma_obj(mid)
        if not np.isfinite(fm): break
        if fm >= 0: low = mid
        else: high = mid
        k += 1
    # k = 0
    # while (high - low > my_eps) and k < 200:
    #     mid = 0.5*(low+high); fm = FindSigma_obj(mid)
    #     if not np.isfinite(fm): break
    #     if fm >= 0: low = mid
    #     else: high = mid
    #     k += 1
    # ------------------------------------------------------------------------
    return 0.5 * (low + high)

def Interpolate(K,V):
    N = len(K)
    nu = []
    kappa = []
    omega = []
    A = V[0]
    delta = 0.5
    B = delta*(V[1]-V[0])/(K[1]-K[0])
    for i in range(N-2):
        K1t = K[i]
        K2t = K[i+1]
        V1t = V[i]
        V2t = V[i+1]
        B1 = delta*(V[i+2]-V2t)/(K[i+2]-K2t) + (1.0-delta)*(V2t-V1t)/(K2t-K1t)
        w = (V2t + B*K1t - A - B1*K2t)/(B-B1)
        sigma_h = findSigmaHat(A,B,w,K1t,K2t,V2t)
        sigma = FindSigma(A,B,w,K1t,K2t,V2t,B1,sigma_h)
        sigma_t = SigmaTilde(sigma,A,B,w,K1t,K2t,V2t)
        At = Atilde(sigma,A,B,w,K1t)
        Bt = Btilde(sigma,A,B,w,K1t)
        nu.append(K1t)
        nu.append(w)
        omega.append(1.0/sigma)
        omega.append(1.0/sigma_t)
        kappa.append([0.5*(A+sigma*B),0.5*(A-sigma*B)])
        kappa.append([0.5*(At+sigma_t*Bt),0.5*(At-sigma_t*Bt)])
        A = V2t
        B = B1
    return nu, omega, kappa, B


def build_theta(bid_ask, S0, R1=None, R2=None,
                target_prices=None, target_strikes=None,
                moneyness=None, Kbar=None, verbose=True):
    """Calibrate LVG theta to arb-free call prices.  Active core == HW5b.

    HW5b hard-codes S, Kbar and reads K,C from a CSV; here the same algorithm
    runs on any Process_SPX dataset.  Only the [interface] glue (dataset unpack,
    list->tensor conversion) and the commented >>> FIX blocks are non-HW5b.
    """
    BUILD_THETA_LOG.clear()                       # [interface]
    # heal a leftover run-log tee from a previously-interrupted run_and_plot
    import sys as _bt_sys
    while getattr(_bt_sys.stdout, '_is_runlog_tee', False):
        _bt_sys.stdout = _bt_sys.stdout.ws[0]
    _blog("[build_theta] starting")
    # [interface] unpack dataset (HW5b: K,C come from ArbFree_calls_strikes.csv)
    strikes = bid_ask[:, 0].double()
    bids = bid_ask[:, 1].double()
    asks = bid_ask[:, 2].double()
    target = (0.5 * (bids + asks) if target_prices is None else target_prices.double())
    targetK = (strikes if target_strikes is None else target_strikes.double())
    K = targetK.cpu().numpy()
    C = target.cpu().numpy()
    if moneyness is not None:
        msk = (K >= moneyness[0] * S0) & (K <= moneyness[1] * S0)
        K, C = K[msk], C[msk]
    o = np.argsort(K)
    K, C = K[o], C[o]
    N = len(K)
    _blog(f"[build_theta] N={N} strikes after filter, S0={S0}")
    S = float(S0)
    if Kbar is None:
        # HW5b uses a fixed Kbar=2000 ABOVE all strikes.  K.max() alone makes
        # K2[1]=Kbar-K[-1]=0 -> zero-gap knot -> divide-by-zero -> nan.  Default
        # to extract's convention so Kbar always exceeds the last strike.
        Kbar = max(2000.0, float(K.max()) + 100.0)
    Kbar = float(Kbar)

    # ── HW5b cell 9: main algo (i0, Vtemp, K1/V1/K2/V2 setup) ─────────────────
    i0 = 0
    for i in range(N-1):
        if (K[i] < S) & (S < K[i+1]):
            i0 = i

    Lt1 = C[i0] + (S-K[i0])*(C[i0]-C[i0-1])/(K[i0]-K[i0-1])
    Lt2 = C[i0+1] + (S-K[i0+1])*(C[i0+2]-C[i0+1])/(K[i0+2]-K[i0+1])
    Lt = max([Lt1, Lt2])
    Ut = C[i0]*(K[i0+1]-S)/(K[i0+1]-K[i0]) + C[i0+1]*(S-K[i0])/(K[i0+1]-K[i0])
    delta = 0.5
    Vtemp = delta*Lt + (1-delta)*Ut

    # Boundary anchors: left wing at K=L=0 (C=S), right wing at K=Kbar (C=0).
    # These now match the arb-free LP (Process_SPX), which enforces convexity
    # out to L=0 and Kbar -- so HW5b's native anchoring is consistent and no
    # phantom offset is needed.
    K1 = [0]
    V1 = [0]
    for i in range(N):
        if (K[i] < S):
            K1.append(K[i])
            V1.append(C[i]-S+K[i])

    K2 = [0]
    V2 = [0]
    for i in range(N):
        if (K[N-1-i] > S):
            K2.append(Kbar - K[N-1-i])
            V2.append(C[N-1-i])

    K1.append(S)
    K2.append(Kbar-S)
    V1.append(Vtemp)
    V2.append(Vtemp)
    if BUILD_THETA_DEBUG:
        _g1 = np.diff(np.asarray(K1, float)); _g2 = np.diff(np.asarray(K2, float))
        _blog(f"[build_theta] Kbar={Kbar:.2f}  last strike={float(K.max()):.2f}  Kbar-K[-1]={Kbar-float(K.max()):.4g}")
        _blog(f"[build_theta] K1 {len(K1)} pts min gap={_g1.min():.4g} @idx {int(np.argmin(_g1))} | "
              f"K2 {len(K2)} pts min gap={_g2.min():.4g} @idx {int(np.argmin(_g2))}")
        if _g1.min() <= 0 or _g2.min() <= 0:
            _blog("[build_theta] ** ZERO/NEG KNOT GAP -> divide-by-zero -> nan.  "
                  "Likely Kbar <= last strike; pass Kbar=ch['Kbar'].")

    _blog(f"[build_theta] Interpolate LEFT: {len(K1)} pts")
    nu1, omega1, kappa1, B1_last = Interpolate(K1, V1)
    if BUILD_THETA_DEBUG:
        _nb = int(np.sum(~np.isfinite(np.asarray(omega1, float))))
        _blog(f"[build_theta] LEFT non-finite omega: {_nb}/{len(omega1)}")
    _blog(f"[build_theta] Interpolate LEFT done, nu1 len={len(nu1)}")
    _blog(f"[build_theta] Interpolate RIGHT: {len(K2)} pts")
    nu2, omega2, kappa2, B2_last = Interpolate(K2, V2)
    if BUILD_THETA_DEBUG:
        _nb = int(np.sum(~np.isfinite(np.asarray(omega2, float))))
        _blog(f"[build_theta] RIGHT non-finite omega: {_nb}/{len(omega2)}")
    _blog(f"[build_theta] Interpolate RIGHT done, nu2 len={len(nu2)}")

    # ── HW5b cells 13-15: final-touch extra S0 segments ──────────────────────
    _blog("[build_theta] computing extra S0 segments")
    delta1 = 0.5
    rho = delta1 + delta1*(V2[-2]-V2[-1])/(K2[-1]-K2[-2]) + (1.0-delta1)*(V1[-1]-V1[-2])/(K1[-1]-K1[-2])

    K1t = K1[-2]; K2t = K1[-1]; V1t = V1[-2]; V2t = V1[-1]
    A = V1t; B = B1_last; B1 = rho
    w = (V2t + B*K1t - A - B1*K2t)/(B-B1)
    sigma_h = findSigmaHat(A,B,w,K1t,K2t,V2t)
    sigma = FindSigma(A,B,w,K1t,K2t,V2t,B1,sigma_h)
    sigma_t = SigmaTilde(sigma,A,B,w,K1t,K2t,V2t)
    At = Atilde(sigma,A,B,w,K1t); Bt = Btilde(sigma,A,B,w,K1t)
    nu1.append(K1t); nu1.append(w)
    omega1.append(1.0/sigma); omega1.append(1.0/sigma_t)
    kappa1.append([0.5*(A+sigma*B),0.5*(A-sigma*B)])
    kappa1.append([0.5*(At+sigma_t*Bt),0.5*(At-sigma_t*Bt)])

    K1t = K2[-2]; K2t = K2[-1]; V1t = V2[-2]; V2t = V2[-1]
    A = V1t; B = B2_last; B1 = 1 - rho
    w = (V2t + B*K1t - A - B1*K2t)/(B-B1)
    sigma_h = findSigmaHat(A,B,w,K1t,K2t,V2t)
    sigma = FindSigma(A,B,w,K1t,K2t,V2t,B1,sigma_h)
    sigma_t = SigmaTilde(sigma,A,B,w,K1t,K2t,V2t)
    At = Atilde(sigma,A,B,w,K1t); Bt = Btilde(sigma,A,B,w,K1t)
    nu2.append(K1t); nu2.append(w)
    omega2.append(1.0/sigma); omega2.append(1.0/sigma_t)
    kappa2.append([0.5*(A+sigma*B),0.5*(A-sigma*B)])
    kappa2.append([0.5*(At+sigma_t*Bt),0.5*(At-sigma_t*Bt)])

    # ── [interface] HW5b lists → optimiser tensors ───────────────────────────
    # HW5b keeps nu2 in K2-space (Kbar-K).  Optimiser wants nus2 in K-space.
    nus1 = np.array(nu1, dtype=float)
    sig1 = 1.0 / np.array(omega1, dtype=float)
    nus2 = Kbar - np.array(nu2, dtype=float)
    sig2 = 1.0 / np.array(omega2, dtype=float)

    # [interface] nus2 = Kbar - nu2 comes out DESCENDING; the optimiser needs
    # nus strictly increasing.  Sort each side ascending (sigmas kept paired).
    # With convex-to-boundary arb-free inputs there are no degenerate/duplicate
    # knots, so no dedup or range filter is required.
    o1 = np.argsort(nus1); nus1, sig1 = nus1[o1], sig1[o1]
    o2 = np.argsort(nus2); nus2, sig2 = nus2[o2], sig2[o2]

    # >>> FIX G (global sigma floor: cap cumulative exp) ----------------------
    # generate_call_prices propagates kappa through every interval as a product
    # of exp(gap_i/sigma_i).  With no floor, sum_i gap_i/sigma_i can exceed ~709
    # and exp() overflows to inf -> nan prices -> nan J.  Floor each sigma so
    # the per-side cumulative exponent <= MAX_EXP.  To enable, uncomment:
    # MAX_EXP = np.log(np.finfo(float).max) - 1.0
    # sig1 = np.clip(sig1, float(S0) / MAX_EXP,        float(S0))
    # sig2 = np.clip(sig2, float(Kbar - S0) / MAX_EXP, float(Kbar - S0))
    # -------------------------------------------------------------------------

    sigs1 = torch.tensor(sig1, dtype=torch.float64)
    sigs2 = torch.tensor(sig2, dtype=torch.float64)
    nus1_t = torch.tensor(nus1, dtype=torch.float64)
    nus2_t = torch.tensor(nus2, dtype=torch.float64)

    _blog(f"[build_theta] done: R1={len(nus1)} R2={len(nus2)}")
    if verbose:
        try:
            th = obtain_theta(len(nus1), len(nus2), S0, sigs1, sigs2, nus1_t, nus2_t)
            _, _, _, c1, c2 = calculate_J(len(nus1), len(nus2), S0, th)
            Cw = generate_call_prices(torch.tensor(K), len(nus1), len(nus2), S0, th, c1, c2).detach().numpy()
            rmse = float(np.sqrt(np.mean((Cw - C) ** 2)))
            Cb = generate_call_prices(strikes, len(nus1), len(nus2), S0, th, c1, c2).detach().numpy()
            b = bids.cpu().numpy(); a = asks.cpu().numpy()
            mv = max(np.clip(b - Cb, 0, None).max(), np.clip(Cb - a, 0, None).max())
            mono_str = f"window {moneyness}" if moneyness else "all strikes"
            _blog(f"[build theta] {mono_str} | R1={len(nus1)} R2={len(nus2)} | "
                  f"RMSE to target = {rmse:.4g} | band violation = {mv:.3e}")
        except Exception as _e:
            _blog(f"[build theta] diagnostics skipped: {_e}")

    return sigs1.detach(), sigs2.detach(), nus1_t.detach(), nus2_t.detach()


In [ ]:
# ════════════════════════════════════════════════════════════════════════════
# SIGMA-ONLY HELPERS
# ════════════════════════════════════════════════════════════════════════════

def build_sigma_only_start(bid_ask, S0, sigs1_init, sigs2_init, nus1_init, nus2_init,
                            R1_new, R2_new, Kbar=None, verbose=True, uniform=False):
    """Densify the build_theta fit onto a finer FIXED-nu grid by INTERVAL
    SUBDIVISION.

    Each parent LVG interval is split into f evenly-spaced sub-intervals and the
    parent's sigma is copied to every sub-interval.  Because an LVG piece with a
    fixed sigma solves V'' = (1/sigma)^2 V, subdividing it and re-anchoring kappa
    reproduces the SAME analytic curve: the refined theta has identical C, C',
    C'' and J to build_theta, and the C'' jump at every inserted knot is exactly
    zero.  So this is an EXACT warm start for sigma-only optimization -- the AL
    starts on build_theta's curve and can only smooth it further.

    Conventions (must match _sigonly_gaps):
      Left  boundaries [nus1[0..R1-1], S0]; sigs1[j] governs [B[j], B[j+1]].
      Right boundaries [S0, nus2[0..R2-1]]; sigs2[j] governs [B[j], B[j+1]].
    f1 ~ R1_new/R1, f2 ~ R2_new/R2 (rounded, >=1); the returned counts are R1*f1
    and R2*f2 (exact when R1_new/R2_new are integer multiples, e.g. the 4x case).
    """
    def _to_np(x):
        return x.detach().cpu().numpy() if torch.is_tensor(x) else np.asarray(x, dtype=float)

    S0f = float(S0)
    nus1_old = _to_np(nus1_init); sigs1_old = _to_np(sigs1_init)
    nus2_old = _to_np(nus2_init); sigs2_old = _to_np(sigs2_init)
    R1 = len(sigs1_old); R2 = len(sigs2_old)

    BL = np.append(nus1_old, S0f)                      # R1+1 left boundaries
    BR = np.insert(nus2_old, 0, S0f)                   # R2+1 right boundaries

    def _refine(B, sig_old, R, counts):
        # counts[j] = # equal sub-intervals for parent j; copy parent sigma so the
        # refined curve is identical to build_theta (every inserted-knot C'' jump = 0).
        nus, sig = [], []
        for j in range(R):
            fj = max(1, int(counts[j]))
            nus.extend(np.linspace(B[j], B[j + 1], fj, endpoint=False).tolist())
            sig.extend([float(sig_old[j])] * fj)
        return nus, sig

    if uniform:
        # UNIFORM-WIDTH refinement: split each parent into pieces of ~equal target
        # width = span / R_new, so every interval ends up at ~that small width.  This
        # redistributes inserted nus toward the WIDE intervals (e.g. the wings, which
        # the proportional split under-resolves) instead of mirroring build_theta density.
        twL = (S0f - float(BL[0])) / max(R1_new, 1)
        twR = (float(BR[-1]) - S0f) / max(R2_new, 1)
        cL = [round((BL[j + 1] - BL[j]) / twL) for j in range(R1)]
        cR = [round((BR[j + 1] - BR[j]) / twR) for j in range(R2)]
        _mode = f"uniform-width  twL={twL:.3g} twR={twR:.3g}"
    else:
        f1 = max(1, int(round(R1_new / max(R1, 1))))
        f2 = max(1, int(round(R2_new / max(R2, 1))))
        cL = [f1] * R1; cR = [f2] * R2
        _mode = f"subdivision  f1={f1} f2={f2}"

    _n1, new_sig1 = _refine(BL, sigs1_old, R1, cL)
    new_nus1 = np.asarray(_n1); new_sig1 = np.asarray(new_sig1)   # knots < S0
    refined_R, new_sig2 = _refine(BR, sigs2_old, R2, cR)
    new_nus2 = np.asarray(refined_R[1:] + [float(nus2_old[-1])])  # knots > S0, ends at Kbar
    new_sig2 = np.asarray(new_sig2)

    dtype   = torch.float64
    sigs1_t = torch.tensor(new_sig1, dtype=dtype)
    sigs2_t = torch.tensor(new_sig2, dtype=dtype)
    nus1_t  = torch.tensor(new_nus1, dtype=dtype)
    nus2_t  = torch.tensor(new_nus2, dtype=dtype)

    if verbose:
        print(f'[build_sigma_only_start] {_mode}  '
              f'R1 {R1}->{len(new_nus1)}  R2 {R2}->{len(new_nus2)}')
        g1, g2 = _sigonly_gaps(nus1_t, nus2_t, S0)
        bl = float((g1 / sigs1_t).sum()); br = float((g2 / sigs2_t).sum())
        print(f'  budget sums (curve-preserving, == build_theta): '
              f'left={bl:.1f}  right={br:.1f}  cap={SIG_BUDGET_CAP:.0f}')
        if bl >= SIG_BUDGET_CAP or br >= SIG_BUDGET_CAP:
            print(f'  ** WARNING: a side exceeds SIG_BUDGET_CAP -> raw_sigonly_from_sigs '
                  f'will clamp and the warm start will NOT be exact.  Raise SIG_BUDGET_CAP '
                  f'above {max(bl, br):.0f} to keep the exact build_theta start.')
        print(f'  Left  nus: [{new_nus1[0]:.2f}, ..., {new_nus1[-1]:.2f}] (< S0={S0f:.2f})')
        print(f'  Right nus: [{new_nus2[0]:.2f}, ..., {new_nus2[-1]:.2f}] (> S0)')

    return sigs1_t, sigs2_t, nus1_t, nus2_t

def _diag_J(R1, R2, S0, theta):
    import io
    buf = io.StringIO()
    try:
        nus1  = theta[:R1];          sigs1 = theta[R1:2*R1]
        nus2  = theta[2*R1:2*R1+R2]; sigs2 = theta[2*R1+R2:]
        buf.write(f'  nus1  [{nus1.min():.2f}, {nus1.max():.2f}]  sigs1 [{sigs1.min():.3f}, {sigs1.max():.3f}]\n')
        buf.write(f'  nus2  [{nus2.min():.2f}, {nus2.max():.2f}]  sigs2 [{sigs2.min():.3f}, {sigs2.max():.3f}]\n')
        nus1_a = torch.cat([nus1, torch.tensor([float(S0)])])
        nus2_a = torch.cat([torch.tensor([float(S0)]), nus2])
        with torch.no_grad():
            dist0 = nus1_a[1] - nus1_a[0]
            c = [(-torch.exp(-dist0/sigs1[0]), torch.exp(dist0/sigs1[0]))]
            first_bad_L = None
            for j in range(R1 - 1):
                P  = c[j][0] + c[j][1]
                S_ = (1/sigs1[j]) * (-c[j][0] + c[j][1])
                d  = nus1_a[j+2] - nus1_a[j+1]
                n0 = 0.5*(P - sigs1[j+1]*S_)*torch.exp(-d/sigs1[j+1])
                n1 = 0.5*(P + sigs1[j+1]*S_)*torch.exp( d/sigs1[j+1])
                c.append((n0, n1))
                if first_bad_L is None and not (torch.isfinite(n0) and torch.isfinite(n1)):
                    first_bad_L = j+1
                    buf.write(f'  ** LEFT non-finite at j={j+1}: c=({n0.item():.3g},{n1.item():.3g}) '
                              f'sigma={sigs1[j+1].item():.3f} dist={d.item():.3f}\n')
            v1    = c[-1][0] + c[-1][1]
            Dk_v1 = (1/sigs1[-1]) * (-c[-1][0] + c[-1][1])
            buf.write(f'  v1={v1.item():.4g}  Dk_v1={Dk_v1.item():.4g}\n')
            c2 = [None]*R2
            d_u = nus2_a[-1] - nus2_a[-2]
            c2[-1] = (torch.exp(d_u/sigs2[-1]), -torch.exp(-d_u/sigs2[-1]))
            first_bad_R = None
            for j in range(R2-1, 0, -1):
                P  = c2[j][0] + c2[j][1]
                S_ = (1/sigs2[j]) * (-c2[j][0] + c2[j][1])
                d  = nus2_a[j] - nus2_a[j-1]
                n0 = 0.5*(P - sigs2[j-1]*S_)*torch.exp( d/sigs2[j-1])
                n1 = 0.5*(P + sigs2[j-1]*S_)*torch.exp(-d/sigs2[j-1])
                c2[j-1] = (n0, n1)
                if first_bad_R is None and not (torch.isfinite(n0) and torch.isfinite(n1)):
                    first_bad_R = j-1
                    buf.write(f'  ** RIGHT non-finite at j={j-1}: c=({n0.item():.3g},{n1.item():.3g}) '
                              f'sigma={sigs2[j-1].item():.3f} dist={d.item():.3f}\n')
            v2    = c2[0][0] + c2[0][1]
            Dk_v2 = (1/sigs2[0]) * (-c2[0][0] + c2[0][1])
            buf.write(f'  v2={v2.item():.4g}  Dk_v2={Dk_v2.item():.4g}\n')
            denom = Dk_v1*v2 - v1*Dk_v2
            buf.write(f'  denom={denom.item():.4g}  '
                      f'|Dk_v1*v2|={abs(Dk_v1.item()*v2.item()):.3g}  '
                      f'|v1*Dk_v2|={abs(v1.item()*Dk_v2.item()):.3g}\n')
    except Exception as e:
        buf.write(f'  _diag_J exception: {e}\n')
    return buf.getvalue()


def optimize_sigma_only(R1_new, R2_new, S0, sigs1_new, sigs2_new, nus1_new, nus2_new,
                        loss_fn, generate_call_prices, bid_ask, **kwargs):
    nus1_t = nus1_new.detach().clone()
    nus2_t = nus2_new.detach().clone()

    # BUDGET reparam: bound the cumulative sum_j gap_j/sigma_j per side at cap.
    # No raw value (init, L-BFGS step, or wild Phase-0 sample) can overflow, yet
    # individual sigmas stay free to get small for sharp local curvature.
    cap_left = cap_right = SIG_BUDGET_CAP
    gaps1, gaps2 = _sigonly_gaps(nus1_t, nus2_t, S0)

    reparam_fn = lambda r: build_theta_sigonly_from_raw(
        r, R1_new, R2_new, nus1_t, nus2_t, S0,
        cap_left=cap_left, cap_right=cap_right)
    raw_init   = raw_sigonly_from_sigs(
        sigs1_new.detach(), sigs2_new.detach(), gaps1, gaps2,
        cap_left=cap_left, cap_right=cap_right)

    verbose = kwargs.get('verbose', True)
    with torch.no_grad():
        theta_test = reparam_fn(raw_init)
        J_test, _, _, _, _ = loss_fn(R1_new, R2_new, S0, theta_test)
        Sl = float((gaps1 / theta_test[R1_new:2*R1_new]).sum())
        Sr = float((gaps2 / theta_test[2*R1_new+R2_new:]).sum())
        if verbose:
            print(f'\n[sigma-only pre-flight] J={J_test.item():.6g}  '
                  f'(budget sums: left={Sl:.1f}/{cap_left:.0f}, right={Sr:.1f}/{cap_right:.0f})')
        if not torch.isfinite(J_test):
            print('[sigma-only] WARNING: J=NaN/Inf at initial theta -- tracing recursion:')
            print(_diag_J(R1_new, R2_new, S0, theta_test))

    return optimize_augmented_lagrangian(
        R1_new, R2_new, S0, sigs1_new, sigs2_new, nus1_new, nus2_new,
        loss_fn, generate_call_prices, bid_ask,
        reparam_fn=reparam_fn, raw_init=raw_init,
        **kwargs)


def optimize_sigma_only_LKbar(R1_new, R2_new, S0, sigs1_new, sigs2_new,
                              nus1_new, nus2_new,
                              loss_fn, generate_call_prices, bid_ask,
                              kbar_up_factor=2.0, **kwargs):
    """Sigma-only optimization that ALSO frees the two extreme boundary knots
    L = nus1[0] and Kbar = nus2[-1] (all interior nus stay fixed).  Adds exactly
    2 DOF over plain sigma-only.  Warm-starts on the densified build_theta curve
    (L, Kbar at their initial values), so the AL begins on that curve and can
    only smooth it further."""
    nus1_t = nus1_new.detach().clone()
    nus2_t = nus2_new.detach().clone()
    cap_left = cap_right = SIG_BUDGET_CAP
    lo_L, hi_L, lo_K, hi_K = sigonly_LKbar_bounds(
        nus1_t, nus2_t, S0, kbar_up_factor=kbar_up_factor)
    gaps1, gaps2 = _sigonly_gaps(nus1_t, nus2_t, S0)

    reparam_fn = lambda r: build_theta_sigonly_LKbar_from_raw(
        r, R1_new, R2_new, nus1_t, nus2_t, S0, lo_L, hi_L, lo_K, hi_K,
        cap_left=cap_left, cap_right=cap_right)

    raw_sigs = raw_sigonly_from_sigs(sigs1_new.detach(), sigs2_new.detach(),
                                     gaps1, gaps2, cap_left=cap_left, cap_right=cap_right)
    raw_L    = _bounded_inv(float(nus1_t[0]),  lo_L, hi_L)
    raw_Kbar = _bounded_inv(float(nus2_t[-1]), lo_K, hi_K)
    raw_init = torch.cat([raw_sigs, raw_L.reshape(1), raw_Kbar.reshape(1)])

    verbose = kwargs.get('verbose', True)
    with torch.no_grad():
        theta_test = reparam_fn(raw_init)
        J_test, _, _, _, _ = loss_fn(R1_new, R2_new, S0, theta_test)
        if verbose:
            print(f'\n[sigma+L/Kbar pre-flight] J={J_test.item():.6g}  '
                  f'L: {float(nus1_t[0]):.3f} in ({lo_L:.3f}, {hi_L:.3f})  |  '
                  f'Kbar: {float(nus2_t[-1]):.3f} in ({lo_K:.3f}, {hi_K:.3f})')
        if not torch.isfinite(J_test):
            print('[sigma+L/Kbar] WARNING: J=NaN/Inf at initial theta')

    return optimize_augmented_lagrangian(
        R1_new, R2_new, S0, sigs1_new, sigs2_new, nus1_new, nus2_new,
        loss_fn, generate_call_prices, bid_ask,
        reparam_fn=reparam_fn, raw_init=raw_init, **kwargs)


In [ ]:
def compare_start_dates(
        expiry_date,           # int YYYYMMDD: the single expiry held fixed for every run
        start_dates,           # list[int YYYYMMDD]: trade dates to fit against that expiry
        csv_path=None,         # str or None: raw chain CSV; if set, missing per-date files are auto-built
        optimize_mode="sig_nu",# "sig_nu" | "sig_only" | "sig_only_LKbar" (see run_multi_expiry)
        R1_mult=4,             # int >=1: LEFT-wing knot densification factor (1 = none)
        R2_mult=4,             # int >=1: RIGHT-wing knot densification factor (1 = none)
        unif_nus=False,        # bool: True = uniform-width knot insertion; False = proportional
        max_outer=10,          # int: max outer AL iterations per date
        normalize=True,        # bool: True = mesh-invariant roughness; False = raw sum
        obj=1,                 # 1 | 2: objective kind (1 = 1/sigma^2 jumps; 2 = relative log(V/C'') jumps)
        n_grid=400,            # int: # K-grid points for the V/C'' before/after curves (plot resolution only)
        batch_size=10,         # int: dates per summary image
        sample=False,          # bool: True = uniformly subsample start_dates to sample_size
        sample_size=20,        # int: # start dates kept when sample=True
        out_dir=".",           # str: folder for all images/CSVs/logs from this call
        emit_overall=True,     # bool: also emit one ALL-SAMPLE summary across batches (when >1 batch)
        short_tenor_days=14.0, # float: tenor cutoff (in days) for the OPTIONAL blanket pre-override below
        short_tenor_mode=None, # None = no tenor pre-judging (recommended; use retry instead).  If set
                               #   ("sig_only"|...), dates with T<short_tenor_days START in this mode.
        retry_fallback=True,   # bool: if the PRIMARY run underperforms, retry in retry_mode and keep the
                               #   better of the two.  Data-driven -- only failures pay for a 2nd solve.
        retry_threshold=95.0,  # float %: a primary run with pct_improve below this counts as a "failure"
                               #   and triggers the retry (95 = retry unless the date nearly fully smoothed)
        retry_mode="sig_only", # "sig_only" | "sig_nu" | "sig_only_LKbar" | None: mode for the retry attempt
                               #   (None disables retry).  sig_only is the robust rescue for sig_nu failures.
        **run_kwargs):         # forwarded to run_and_plot -> optimize_augmented_lagrangian (tol_constraint,
                               #   rho_init/rho_scale/rho_max, feas_restore, verbose, ...)
    """
    expiry_date : int YYYYMMDD, held fixed for every run.
    start_dates : list of int YYYYMMDD trade dates.  Each needs its
                  Quotes_SPX_<date>.csv / ArbFree_SPX_<date>.csv (from process_all).
                  Pass csv_path (the raw chain CSV) to auto-run process_all for any
                  date whose per-date files are missing.

    Sampling:
      * sample=True : uniformly subsample sample_size dates from start_dates before
        processing (keeps the overlay plots legible and the run short).
    """
    import math, time, os
    os.makedirs(out_dir, exist_ok=True)   # all images for this call land here

    # Single objective: calculate_J (log roughness of 1/sigma^2).  Mirror run_and_plot's
    # normalize choice on the module flag so this function's J_init / J_fin and the bar
    # plot match each run's printed 'J IMPROVEMENT'.
    set_J_objective(obj)
    set_J_normalize(normalize)
    J_loss = calculate_J

    def _VC2_curve(R1, R2, S0, theta, kmin, kmax):
        # V(K)/C''(K) on a fine K grid -> (log-moneyness, ratio) with C''>0 masked
        K  = np.linspace(kmin, kmax, n_grid)
        Kt = torch.tensor(K, dtype=torch.float64)
        with torch.no_grad():
            _, _, _, c1, c2 = calculate_J(R1, R2, S0, theta)
            c2k = generate_C2K(Kt, R1, R2, S0, theta, c1, c2).cpu().numpy()
            Ck  = generate_call_prices(Kt, R1, R2, S0, theta, c1, c2).cpu().numpy()
        V = Ck - np.maximum(S0 - K, 0.0)
        m = c2k > 0
        return np.log(K[m] / S0), np.clip(V[m] / c2k[m], 1e-30, None)

    # optional uniform subsample of the start dates
    start_dates = list(start_dates)
    if sample and 0 < sample_size < len(start_dates):
        idx = np.linspace(0, len(start_dates) - 1, sample_size).round().astype(int)
        idx = sorted(set(int(i) for i in idx))
        start_dates = [start_dates[i] for i in idx]
        print(f"[compare] sample=True: using {len(start_dates)} uniformly-spaced "
              f"start dates of the original list")

    def _emit_batch_summary(batch_results, tag):
        # One combined 3-panel summary image (J% bar + V/C'' before/after + stats box)
        # for the given set of records.  `tag` labels the title/filename: it is the
        # batch number for a per-batch image, or "ALL-SAMPLE" for the across-batches one.
        if not batch_results:
            return
        labels = [str(r["start_date"]) for r in batch_results]
        cmap = plt.cm.viridis(np.linspace(0, 0.9, len(batch_results)))
        # Per-date mode: the short-tenor fallback may run some dates in a different mode
        # than the batch's requested optimize_mode.  Flag those so the fallback is VISIBLE
        # (the title shows the requested mode only).  fb_idx = indices that fell back.
        modes_pd = [r.get("mode", optimize_mode) for r in batch_results]
        fb_idx = [k for k, m in enumerate(modes_pd) if m != optimize_mode]
        fb_modes = sorted(set(modes_pd[k] for k in fb_idx))
        def _mark_fallback(bars):
            # red hatch on bars whose date used a fallback mode
            for k in fb_idx:
                bars[k].set_hatch("//"); bars[k].set_edgecolor("red"); bars[k].set_linewidth(1.3)
        fig, axes = plt.subplots(2, 3, figsize=(24, 13))
        ax0, ax1, ax2 = axes[0]
        axA, axB, axT = axes[1]   # axT (was the blank panel) now holds the TRUE-% bar

        # (a) J improvement % bar
        _mark_fallback(ax0.bar(labels, [r["pct_improve"] for r in batch_results], color=cmap))
        ax0.set_ylabel("J improvement (%)"); ax0.set_xlabel("start date")
        ax0.set_title(f"J improvement ({tag})")
        ax0.grid(alpha=0.3, axis="y")
        plt.setp(ax0.get_xticklabels(), rotation=90, fontsize=7)
        for i, r in enumerate(batch_results):
            ax0.text(i, r["pct_improve"], f"{r['pct_improve']:.0f}",
                     ha="center", va="bottom", fontsize=6)

        # (b) V/C'' before
        for r, c in zip(batch_results, cmap):
            ax1.semilogy(r["x_init"], r["vc2_init"], "-", color=c, lw=1.2,
                         label=str(r["start_date"]))
        ax1.axvline(0.0, color="gray", ls="--", lw=0.8)
        ax1.set_xlabel("log(K / S0)"); ax1.set_ylabel("V / C''  (log)")
        ax1.set_title("V/C'' BEFORE optimization")
        ax1.grid(alpha=0.3, which="both"); ax1.legend(fontsize=6, ncol=2)

        # (c) V/C'' after
        for r, c in zip(batch_results, cmap):
            ax2.semilogy(r["x_final"], r["vc2_final"], "-", color=c, lw=1.2,
                         label=str(r["start_date"]))
        ax2.axvline(0.0, color="gray", ls="--", lw=0.8)
        ax2.set_xlabel("log(K / S0)"); ax2.set_ylabel("V / C''  (log)")
        ax2.set_title(f"V/C'' AFTER optimization (mode={optimize_mode})")
        ax2.grid(alpha=0.3, which="both"); ax2.legend(fontsize=6, ncol=2)

        # (A) fit quality vs time-to-expiry T — one marker per start date.  Reveals
        # whether the optimiser degrades at short maturities (tighter band, sparser
        # strikes).  Left axis = TRUE roughness reduction %, right axis = S_final (log).
        Ts = [float(r.get("T", np.nan)) for r in batch_results]
        axA.scatter(Ts, [r["true_reduce"] for r in batch_results], c=cmap, s=70, zorder=3)
        for r, t in zip(batch_results, Ts):
            axA.annotate(str(r["start_date"]), (t, r["true_reduce"]), fontsize=5,
                         textcoords="offset points", xytext=(3, 3))
        axA.set_xlabel("time to expiry  T  (years)")
        axA.set_ylabel("TRUE roughness reduction (%)")
        axA.set_title("(A) fit quality vs time-to-expiry")
        axA.grid(alpha=0.3)
        axA2 = axA.twinx()
        axA2.scatter(Ts, [r["S_final"] for r in batch_results], facecolors="none",
                     edgecolors="crimson", marker="s", s=45, zorder=2)
        axA2.set_yscale("log")
        axA2.set_ylabel("S_final  (log scale, □)", color="crimson")
        axA2.tick_params(axis="y", colors="crimson")

        # (B) ATM local-variance term structure: sigma^2(S0) = V/C'' at K=S0 (x=0),
        # before vs after, against T.  V/C''=sigma^2 exactly, so this is the model-
        # implied ATM variance term structure (read from each date's own V/C'' curve).
        def _atm(xs, ys):
            xs = np.asarray(xs); ys = np.asarray(ys)
            if xs.size < 2:
                return np.nan
            o = np.argsort(xs)
            return float(np.interp(0.0, xs[o], ys[o]))      # value at log(K/S0)=0
        atm_b = np.array([_atm(r["x_init"],  r["vc2_init"])  for r in batch_results])
        atm_a = np.array([_atm(r["x_final"], r["vc2_final"]) for r in batch_results])
        oT = np.argsort(Ts); Ts_s = np.asarray(Ts)[oT]
        axB.semilogy(Ts_s, atm_b[oT], "o--", color="steelblue", lw=1.2, label="before")
        axB.semilogy(Ts_s, atm_a[oT], "o-",  color="tomato",    lw=1.6, label="after")
        axB.set_xlabel("time to expiry  T  (years)")
        axB.set_ylabel("ATM  V/C''  = σ²(S0)   (log)")
        axB.set_title("(B) ATM local-variance term structure")
        axB.grid(alpha=0.3, which="both"); axB.legend(fontsize=8)

        # (T) TRUE roughness reduction % per start date — model-space analogue of the
        # log-J bar in (a).  Same start-date axis + viridis colours so the two bars line up
        # one-to-one: (a) is the optimiser's log-J %, this is the true exp-space reduction.
        _mark_fallback(axT.bar(labels, [r["true_reduce"] for r in batch_results], color=cmap))
        axT.set_ylabel("TRUE roughness reduction (%)"); axT.set_xlabel("start date")
        axT.set_title(f"TRUE % improvement ({tag})")
        axT.grid(alpha=0.3, axis="y")
        plt.setp(axT.get_xticklabels(), rotation=90, fontsize=7)
        for i, r in enumerate(batch_results):
            axT.text(i, r["true_reduce"], f"{r['true_reduce']:.1f}",
                     ha="center", va="bottom", fontsize=6)

        # suptitle + mode subtitle
        fig.suptitle(
            f"Single-mode summary | fixed expiry {expiry_date} | {tag} "
            f"({labels[0]}..{labels[-1]})",
            fontsize=14, fontweight="bold", y=0.98)
        _batch_rt = float(sum(r.get("run_time", 0.0) for r in batch_results))
        fig.text(
            0.5, 0.95,
            f"Mode:  {optimize_mode}   |   obj={obj}   |   mult={R1_mult}x{R2_mult}   |   "
            f"batch run time: {_batch_rt:.1f}s ({_batch_rt/60.0:.1f} min)"
            + (f"   |   short-tenor fallback: {len(fb_idx)} date(s) -> {','.join(fb_modes)} "
               f"(red-hatched bars)" if fb_idx else ""),
            ha="center", fontsize=14, style="italic", color="#333333")

        # grey stats box at the bottom
        pcts = [r["pct_improve"] for r in batch_results]
        mean_pct = float(np.mean(pcts)); med_pct = float(np.median(pcts))
        min_pct  = float(np.min(pcts));  max_pct  = float(np.max(pcts))
        std_pct  = float(np.std(pcts))
        trs = [r["true_reduce"] for r in batch_results]
        stats_text = (
            f"log-J improvement (%) — "
            f"Mean: {mean_pct:6.2f}%  |  Median: {med_pct:6.2f}%  |  "
            f"Min: {min_pct:6.2f}%  |  Max: {max_pct:6.2f}%  |  "
            f"Std: {std_pct:6.2f}%\n"
            f"TRUE roughness reduction (%) — "
            f"Mean: {float(np.mean(trs)):6.2f}%  |  Median: {float(np.median(trs)):6.2f}%  |  "
            f"Min: {float(np.min(trs)):6.2f}%  |  Max: {float(np.max(trs)):6.2f}%  |  "
            f"Std: {float(np.std(trs)):6.2f}%\n"
            f"ABSOLUTE initial roughness — "
            f"S_init median: {float(np.median([np.exp(r['J_init']) for r in batch_results])):.4g}  |  "
            f"sqrt(S_init) median: {float(np.median([np.exp(r['J_init'] / 2.0) for r in batch_results])):.4g}\n"
            f"ABSOLUTE final roughness — "
            f"S_final median: {float(np.median([r['S_final'] for r in batch_results])):.4g}  |  "
            f"sqrt(S_final) median: {float(np.median([r['sqrt_S_final'] for r in batch_results])):.4g}")
        fig.text(0.5, 0.005, stats_text, ha="center", fontsize=13,
                 bbox=dict(boxstyle="round,pad=0.5", facecolor="lightgray", alpha=0.8))

        fig.tight_layout(rect=[0, 0.06, 1, 0.94])
        stamp = _dt.datetime.now().strftime("%Y%m%d_%H%M%S")
        _tagstr = str(tag).replace(" ", "")
        fname = os.path.join(out_dir,
                f"compare_summary_{stamp}_{optimize_mode}_{expiry_date}_{_tagstr}.png")
        fig.savefig(fname, dpi=140, bbox_inches="tight")
        print(f"[compare] saved {tag} summary -> {fname}")
        plt.show()
        plt.close('all')
    results = []
    n_batches = max(1, math.ceil(len(start_dates) / batch_size))
    for b in range(n_batches):
        batch_dates = start_dates[b * batch_size:(b + 1) * batch_size]
        batch_results = []
        print(f"\n========== BATCH {b+1}/{n_batches}  ({len(batch_dates)} dates, "
              f"mode={optimize_mode}) ==========")
        for sd in batch_dates:
            # ensure per-date Quotes/ArbFree files exist (optionally build them)
            if csv_path is not None and not (os.path.exists(f"Quotes_csv/Quotes_SPX_{sd}.csv")
                                             and os.path.exists(f"ArbFree_csv/ArbFree_SPX_{sd}.csv")):
                try:
                    process_all(csv_path, sd, quotes_out=f"Quotes_csv/Quotes_SPX_{sd}.csv",
                                arbfree_out=f"ArbFree_csv/ArbFree_SPX_{sd}.csv", verbose=False)
                except Exception as e:
                    print(f"[compare] process_all({sd}) failed: {e}; skipping date")
                    continue
            try:
                ch = extract(sd, expiry_date)
            except Exception as e:
                print(f"[compare] extract({sd}, {expiry_date}) failed: {e}; skipping date")
                continue

            bid_ask, arb_p, arb_K = ch["bid_ask"], ch["arb_p"], ch["arb_K"]
            S0 = ch["S0"]
            s1, s2, n1, n2 = build_theta(bid_ask, S0, target_prices=arb_p, target_strikes=arb_K)
            R1, R2 = len(n1), len(n2)
            strikes = bid_ask[:, 0].cpu().numpy()
            kmin, kmax = float(strikes.min()), float(strikes.max())

            # pre-optimization theta on the build_theta grid (the "before" curve)
            theta_init = obtain_theta(R1, R2, S0, s1, s2, n1, n2).detach()
            with torch.no_grad():
                J_init = J_loss(R1, R2, S0, theta_init)[0].item()
                _c1, _c2 = calculate_J(R1, R2, S0, theta_init)[3:5]
                _C = generate_call_prices(bid_ask[:, 0], R1, R2, S0, theta_init, _c1, _c2)
            # SKIP unfittable dates: build_theta may return a non-finite / collapsed-sigma
            # fit (nan RMSE, positive J) for hard/degenerate-band expiries; that theta yields
            # nan prices and poisons the optimizer, so skip it like a missing expiry.
            if not (torch.isfinite(theta_init).all() and torch.isfinite(_C).all()
                    and np.isfinite(J_init)):
                print(f"[compare] {sd}: build_theta fit non-finite/collapsed "
                      f"(J_init={J_init:.4g}); skipping unfittable date")
                plt.close('all'); continue
            x_i, vc2_i = _VC2_curve(R1, R2, S0, theta_init, kmin, kmax)

            # PER-DATE MODE.  Optional blanket tenor pre-override (off by default): if
            # short_tenor_mode is set, near-expiry dates START in that mode.  The preferred
            # robustness mechanism is the RETRY below -- it doesn't pre-judge dates by tenor.
            _T_days = float(ch["T"]) * 365.25
            _mode_sd = optimize_mode
            if (short_tenor_mode is not None and _T_days < short_tenor_days
                    and optimize_mode != short_tenor_mode):
                _mode_sd = short_tenor_mode
                print(f"[short-tenor pre-override] {sd}: T={_T_days:.1f}d < {short_tenor_days:.0f}d "
                      f"-> mode {optimize_mode} -> {_mode_sd}")

            def _run_one_mode(_run_mode):
                # one full optimisation in _run_mode (also emits this date's diagnostics
                # image); returns everything needed to SCORE the run so we can compare modes.
                _t0 = time.time()
                _theta = run_and_plot(
                    R1, R2, S0, s1, s2, n1, n2,
                    calculate_J, generate_call_prices, bid_ask,
                    arb_prices=arb_p, arb_strikes=arb_K,
                    optimize_mode=_run_mode,
                    R1_new=R1 * R1_mult, R2_new=R2 * R2_mult,
                    max_outer=max_outer, T=ch["T"], r=-np.log(ch["D"]) / ch["T"],
                    start_date=sd, expiry_date=expiry_date,
                    normalize=normalize, obj=obj,
                    R1_mult=R1_mult, R2_mult=R2_mult, out_dir=out_dir,
                    uniform_nus=unif_nus, **run_kwargs)
                _rt = time.time() - _t0
                # recover the ACTUAL densified knot counts so J_fin / V-C'' use the same grid
                # run_and_plot returned (uniform insertion makes the count vary).
                _s1d, _s2d, _n1d, _n2d = build_sigma_only_start(
                    bid_ask, S0, s1, s2, n1, n2, R1 * R1_mult, R2 * R2_mult,
                    Kbar=float(bid_ask[:, 0].max().item()), uniform=unif_nus, verbose=False)
                _R1f, _R2f = len(_s1d), len(_s2d)
                with torch.no_grad():
                    _Jf = J_loss(_R1f, _R2f, S0, _theta)[0].item()
                _xf, _vc2f = _VC2_curve(_R1f, _R2f, S0, _theta, kmin, kmax)
                _pct = (J_init - _Jf) / abs(J_init) * 100.0 if J_init != 0 else float("nan")
                return dict(mode=_run_mode, theta=_theta, J_final=_Jf, pct=_pct,
                            true_pct=true_reduction_pct(J_init, _Jf),
                            x_f=_xf, vc2_f=_vc2f, rt=_rt)

            # PRIMARY run in the per-date mode.
            _best = _run_one_mode(_mode_sd)

            # RETRY-ON-FAILURE: only when the primary UNDERPERFORMS (pct < retry_threshold)
            # do we spend a second solve in retry_mode and KEEP the better of the two.  This
            # rescues sig_nu's near-expiry failures WITHOUT second-guessing healthy dates and
            # WITHOUT pre-judging by tenor -- the data (pct) decides, per date.
            if (retry_fallback and retry_mode is not None
                    and _best["true_pct"] < retry_threshold):
                print(f"[retry-fallback] {sd}: {_best['mode']} gave {_best['pct']:.2f}% "
                      f"(< {retry_threshold:.0f}%) -> retrying in {retry_mode}")
                _alt = _run_one_mode(retry_mode)
                _won = _alt if _alt["true_pct"] > _best["true_pct"] else _best
                print(f"[retry-fallback] {sd}: {retry_mode} gave {_alt['true_pct']:.2f}% -> "
                      f"keeping {_won['mode']} ({_won['true_pct']:.2f}%)")
                _best = _won

            # unpack the WINNER (retry runs overwrite the per-date image with the last mode
            # tried; the recorded rec/summary below always reflect the kept winner).
            _mode_sd = _best["mode"]
            theta_fin = _best["theta"]; J_fin = _best["J_final"]
            pct = _best["pct"]; true_pct = _best["true_pct"]
            x_f, vc2_f = _best["x_f"], _best["vc2_f"]; _date_rt = _best["rt"]
            S_fin = float(np.exp(J_fin)); rms_fin = float(np.exp(J_fin / 2.0))

            rec = dict(start_date=sd, S0=S0, T=ch["T"], mode=_mode_sd,
                       J_init=J_init, J_final=J_fin, pct_improve=pct, true_reduce=true_pct,
                       S_final=S_fin, sqrt_S_final=rms_fin, run_time=_date_rt,
                       x_init=x_i, vc2_init=vc2_i, x_final=x_f, vc2_final=vc2_f)
            results.append(rec); batch_results.append(rec)
            print(f"[compare] {sd} -> {expiry_date}: J {J_init:.4g} -> {J_fin:.4g}  "
                  f"(log-J {pct:.2f}%  |  true {true_pct:.2f}%  |  S_final {S_fin:.4g}  sqrt {rms_fin:.4g})")
            plt.close('all')      # per-date: run_and_plot already saved+displayed from disk

        # end of batch: emit this batch's summary image and free memory
        _emit_batch_summary(batch_results, f"batch{b + 1:02d}")

    # Across-ALL-batches summary for the whole sample of this expiry (one extra image).
    if emit_overall and n_batches > 1:
        _emit_batch_summary(results, "ALL-SAMPLE")

    if not results:
        print("[compare] no successful runs -- nothing produced")
    else:
        print(f"\n[compare] done: {len(results)} dates over {n_batches} batch(es) "
              f"in mode={optimize_mode}; summary images saved to {out_dir!r}.")
    return results


In [ ]:
def summarize_fit_results(expiry_date, csv_source="fit_results_*.csv",
                          optimize_mode=None, n_grid=400, save=True, show=True):
    """Overview summary graphs for ONE fixed expiry from saved fit_results CSVs,
    for a SINGLE-mode (non-staged) optimization run.

    expiry_date   : int YYYYMMDD (must match dataset_expiry_date in the CSV meta).
    csv_source    : glob string (default 'fit_results_*.csv') OR a list of CSV paths.
    optimize_mode : if given, only records with this optimize_mode are used; if None
                    the mode is auto-detected (a 'mixed' label is shown and every
                    mode kept if more than one is present).
    Returns a list of per-date summary dicts (start_date, J_init, J_final, pct, ...).
    """
    import glob, pandas as pd, numpy as np, torch, datetime as _dt
    import matplotlib.pyplot as plt

    exp = int(expiry_date)
    files = sorted(glob.glob(csv_source)) if isinstance(csv_source, str) else list(csv_source)

    def _theta_from_params(df_p, S0):
        L = df_p[df_p["side"] == "L"].sort_values("idx")
        R = df_p[df_p["side"] == "R"].sort_values("idx")
        nus1  = torch.tensor(L["nu"].astype(float).values,    dtype=torch.float64)
        sigs1 = torch.tensor(L["sigma"].astype(float).values, dtype=torch.float64)
        nus2  = torch.tensor(R["nu"].astype(float).values,    dtype=torch.float64)
        sigs2 = torch.tensor(R["sigma"].astype(float).values, dtype=torch.float64)
        R1, R2 = len(nus1), len(nus2)
        theta = obtain_theta(R1, R2, S0, sigs1, sigs2, nus1, nus2).detach()
        return theta, R1, R2

    def _vc2(theta, R1, R2, S0, kmin, kmax):
        K  = np.linspace(kmin, kmax, n_grid)
        Kt = torch.tensor(K, dtype=torch.float64)
        with torch.no_grad():
            _, _, _, c1, c2 = calculate_J(R1, R2, S0, theta)
            c2k = generate_C2K(Kt, R1, R2, S0, theta, c1, c2).cpu().numpy()
            Ck  = generate_call_prices(Kt, R1, R2, S0, theta, c1, c2).cpu().numpy()
        V = Ck - np.maximum(S0 - K, 0.0)
        m = (c2k > 0) & np.isfinite(c2k) & np.isfinite(V)
        return np.log(K[m]), np.clip(V[m] / c2k[m], 1e-30, None)

    # ── gather records for this expiry, filtered to the requested mode ──
    bydate = {}
    modes_seen = set()
    for f in files:
        try:
            df = pd.read_csv(f)
        except Exception:
            continue
        if "record_type" not in df.columns:
            continue
        meta = {row["key"]: row["value"] for _, row in df[df["record_type"] == "meta"].iterrows()}
        if "dataset_expiry_date" not in meta or "dataset_start_date" not in meta:
            continue
        try:
            if int(float(meta["dataset_expiry_date"])) != exp:
                continue
            sd = int(float(meta["dataset_start_date"]))
            S0 = float(meta["S0"]); J_i = float(meta["J_initial"]); J_f = float(meta["J_final"])
            rt = float(meta.get("run_time_sec", float("nan")))
        except Exception:
            continue
        md = str(meta.get("optimize_mode", "?"))
        # filter to the requested single mode (if any)
        if optimize_mode is not None and md != optimize_mode:
            continue
        modes_seen.add(md)
        mk = df[df["record_type"] == "market"]
        if len(mk) == 0:
            continue
        inb = mk["in_band"].astype(str).str.strip().str.lower().isin(["true", "1", "1.0"])
        feasible = bool(inb.all())
        strikes = mk["strike"].astype(float).values
        bydate.setdefault(sd, []).append(dict(
            mode=md, J_init=J_i, J_final=J_f, run_time=rt,
            feasible=feasible, S0=S0, kmin=float(strikes.min()), kmax=float(strikes.max()),
            pi=df[df["record_type"] == "param_init"], pf=df[df["record_type"] == "param_final"]))

    if not bydate:
        print(f"[summary] no fit_results CSVs found for expiry {exp} in '{csv_source}'"
              + (f" with mode={optimize_mode}" if optimize_mode else ""))
        return []

    # label shown on the image: the single mode, or 'mixed: a,b' if several present
    if optimize_mode is not None:
        mode_label = optimize_mode
    elif len(modes_seen) == 1:
        mode_label = next(iter(modes_seen))
    else:
        mode_label = "mixed: " + ",".join(sorted(modes_seen))
        print(f"[summary] WARNING: multiple modes present {sorted(modes_seen)}; "
              f"pass optimize_mode=... to restrict to one")

    # ── per date: pick best-feasible record (min J_final); J_init = build_theta J ──
    summary = []
    for sd in sorted(bydate):
        recs = bydate[sd]
        feas = [r for r in recs if r["feasible"]]
        if not feas:
            print(f"[summary] {sd}: no feasible record; skipping")
            continue
        best = min(feas, key=lambda r: r["J_final"])
        J_init = best["J_init"]; J_fin = best["J_final"]; S0 = best["S0"]
        pct = (J_init - J_fin) / abs(J_init) * 100.0 if J_init != 0 else float("nan")
        true_pct = true_reduction_pct(J_init, J_fin)
        S_fin = float(np.exp(J_fin)); rms_fin = float(np.exp(J_fin / 2.0))
        try:
            th_i, R1i, R2i = _theta_from_params(best["pi"], S0)
            x_i, vc2_i = _vc2(th_i, R1i, R2i, S0, best["kmin"], best["kmax"])
            th_f, R1f, R2f = _theta_from_params(best["pf"], S0)
            x_f, vc2_f = _vc2(th_f, R1f, R2f, S0, best["kmin"], best["kmax"])
        except Exception as e:
            print(f"[summary] {sd}: curve reconstruction failed ({e}); J%% only")
            x_i = vc2_i = x_f = vc2_f = np.array([])
        total_rt = float(np.nansum([r["run_time"] for r in recs]))
        summary.append(dict(start_date=sd, S0=S0, J_init=J_init, J_final=J_fin,
                            pct_improve=pct, true_reduce=true_pct, S_final=S_fin, sqrt_S_final=rms_fin,
                            mode=best["mode"], run_time=total_rt,
                            x_init=x_i, vc2_init=vc2_i, x_final=x_f, vc2_final=vc2_f))
        print(f"[summary] {sd}: mode={best['mode']:14s} J {J_init:.4g} -> {J_fin:.4g}  "
              f"(log-J {pct:.2f}%  |  true {true_pct:.2f}%)")

    if not summary:
        print(f"[summary] no feasible dates for expiry {exp}")
        return summary

    # ── J improvement stats ──
    pcts    = [s["pct_improve"] for s in summary]
    mean_pct = float(np.mean(pcts));   med_pct = float(np.median(pcts))
    min_pct  = float(np.min(pcts));    max_pct = float(np.max(pcts))
    std_pct  = float(np.std(pcts))
    print("")
    trs = [s_["true_reduce"] for s_ in summary]
    print(f"[summary] mode={mode_label} | log-J improvement statistics across {len(summary)} dates:")
    print(f"  Mean:   {mean_pct:7.2f}%")
    print(f"  Median: {med_pct:7.2f}%")
    print(f"  Min:    {min_pct:7.2f}%  |  Max: {max_pct:7.2f}%")
    print(f"  Std:    {std_pct:7.2f}%")
    print(f"[summary] TRUE roughness reduction: Mean {float(np.mean(trs)):7.2f}%  |  "
          f"Median {float(np.median(trs)):7.2f}%  |  Min {float(np.min(trs)):7.2f}%  |  Max {float(np.max(trs)):7.2f}%")
    sfs = [s_["S_final"] for s_ in summary]; rfs = [s_["sqrt_S_final"] for s_ in summary]
    sis = [float(np.exp(s_["J_init"]))       for s_ in summary]   # S_init  = exp(J_init)
    rfi = [float(np.exp(s_["J_init"] / 2.0)) for s_ in summary]   # sqrt(S_init)
    print(f"[summary] ABSOLUTE initial roughness: S_init median {float(np.median(sis)):.4g}  |  "
          f"sqrt(S_init) median {float(np.median(rfi)):.4g}")
    print(f"[summary] ABSOLUTE final roughness: S_final median {float(np.median(sfs)):.4g}  |  "
          f"sqrt(S_final) median {float(np.median(rfs)):.4g}")
    print("")

    # ── run-time stats ──
    rts     = [s["run_time"] for s in summary]
    mean_rt = float(np.nanmean(rts));  med_rt  = float(np.nanmedian(rts))
    min_rt  = float(np.nanmin(rts));   max_rt  = float(np.nanmax(rts))
    std_rt  = float(np.nanstd(rts));   tot_rt  = float(np.nansum(rts))
    print(f"[summary] run-time statistics across {len(summary)} dates (single-mode):")
    print(f"  Mean:   {mean_rt:8.2f} s  ({mean_rt/60:6.2f} min)")
    print(f"  Median: {med_rt:8.2f} s  ({med_rt/60:6.2f} min)")
    print(f"  Min:    {min_rt:8.2f} s  |  Max: {max_rt:8.2f} s")
    print(f"  Std:    {std_rt:8.2f} s")
    print(f"  Total:  {tot_rt:8.2f} s  ({tot_rt/60:6.2f} min)")
    print("")

    # ── overview figure: 3 panels (J% bar, V/C'' before, V/C'' after) ──
    labels = [str(s["start_date"]) for s in summary]
    cmap   = plt.cm.viridis(np.linspace(0, 0.9, len(summary)))
    # 2x2 layout: top row = the two per-start-date bars (log-J % and TRUE %), bottom row
    # = V/C'' before / after.  The TRUE-% bar mirrors the one in the compare_summary image
    # so both summary images carry the same information.
    fig, axes = plt.subplots(2, 2, figsize=(20, 13))
    axJ, axT = axes[0]
    axBefore, axAfter = axes[1]

    # (1) log-J improvement % per date
    axJ.bar(labels, [s["pct_improve"] for s in summary], color=cmap)
    axJ.set_ylabel("J improvement (%)"); axJ.set_xlabel("start date")
    axJ.set_title(f"J improvement by start date (expiry {exp})")
    axJ.grid(alpha=0.3, axis="y")
    plt.setp(axJ.get_xticklabels(), rotation=90, fontsize=6)
    for j, s in enumerate(summary):
        axJ.text(j, s["pct_improve"], f"{s['pct_improve']:.0f}",
                 ha="center", va="bottom", fontsize=5)

    # (2) TRUE roughness reduction % per date — same start-date axis/colours as (1); the
    # exp-space reduction, model-space analogue of the optimiser's log-J % bar.
    axT.bar(labels, [s["true_reduce"] for s in summary], color=cmap)
    axT.set_ylabel("TRUE roughness reduction (%)"); axT.set_xlabel("start date")
    axT.set_title(f"TRUE % improvement by start date (expiry {exp})")
    axT.grid(alpha=0.3, axis="y")
    plt.setp(axT.get_xticklabels(), rotation=90, fontsize=6)
    for j, s in enumerate(summary):
        axT.text(j, s["true_reduce"], f"{s['true_reduce']:.1f}",
                 ha="center", va="bottom", fontsize=5)

    # (3,4) V/C'' before / after, vs log(K)
    for s, c in zip(summary, cmap):
        if len(s["x_init"]):
            axBefore.semilogy(s["x_init"], s["vc2_init"], "-", color=c, lw=1.0,
                              label=str(s["start_date"]))
        if len(s["x_final"]):
            axAfter.semilogy(s["x_final"], s["vc2_final"], "-", color=c, lw=1.0,
                             label=str(s["start_date"]))
    axBefore.set_xlabel("log(K)"); axBefore.set_ylabel("V / C''  (log)"); axBefore.set_title("V/C'' BEFORE")
    axAfter.set_xlabel("log(K)"); axAfter.set_ylabel("V / C''  (log)")
    axAfter.set_title(f"V/C'' AFTER (mode={mode_label})")
    axBefore.grid(alpha=0.3, which="both"); axAfter.grid(alpha=0.3, which="both")
    if len(summary) <= 30:
        axBefore.legend(fontsize=5, ncol=2); axAfter.legend(fontsize=5, ncol=2)

    fig.suptitle(
        f"Overview summary from fit_results CSVs | fixed expiry {exp} | "
        f"{len(summary)} feasible dates",
        fontsize=14, fontweight="bold", y=0.99)
    # single-mode tag one line below the suptitle (vs the staged 'Pipeline: ...' line)
    fig.text(
        0.5, 0.945,
        f"Optimization mode:  {mode_label}   (single-stage, non-staged)",
        ha="center", fontsize=11, style="italic", color="#333333")

    # ── stats text box at the very bottom (J improvement + run time, mode tagged) ──
    stats_text = (
        f"mode = {mode_label}\n"
        f"J improvement (%) — Mean: {mean_pct:6.2f}%  |  Median: {med_pct:6.2f}%  |  "
        f"Min: {min_pct:6.2f}%  |  Max: {max_pct:6.2f}%  |  Std: {std_pct:6.2f}%\n"
        f"ABSOLUTE initial roughness — S_init median: {float(np.median(sis)):.4g}  |  "
        f"sqrt(S_init) median: {float(np.median(rfi)):.4g}\n"
        f"ABSOLUTE final roughness — S_final median: {float(np.median(sfs)):.4g}  |  "
        f"sqrt(S_final) median: {float(np.median(rfs)):.4g}\n"
        f"Run time per date (s) — Mean: {mean_rt:6.1f}  |  Median: {med_rt:6.1f}  |  "
        f"Min: {min_rt:6.1f}  |  Max: {max_rt:6.1f}  |  Std: {std_rt:6.1f}  |  "
        f"Total: {tot_rt:7.1f}")
    fig.text(0.5, 0.01, stats_text, ha="center", fontsize=11,
             bbox=dict(boxstyle="round,pad=0.5", facecolor="lightgray", alpha=0.8))

    fig.tight_layout(rect=[0, 0.10, 1, 0.92])
    if save:
        stamp = _dt.datetime.now().strftime("%Y%m%d_%H%M%S")
        fname = f"overview_summary_{mode_label.replace(':', '').replace(' ', '_').replace(',', '-')}_{exp}_{stamp}.png"
        fig.savefig(fname, dpi=140, bbox_inches="tight")
        print(f"[summary] saved -> {fname}")
    if show:
        plt.show()
    plt.close('all')
    return summary


In [ ]:
def start_dates_for_expiry(
        csv_path,              # str: path to the raw SPX chain CSV
        expiry_date,           # int YYYYMMDD: the single expiry whose trade dates you want
        min_strikes=0):        # int: keep only dates with >= this many two-sided (C+P bid>0) strikes
                               #      (0 = every date carrying the expiry; compare_* skips unfittable ones)
    """All trade (start) dates in a raw SPX CSV that carry a given expiry.

    csv_path     : raw chain CSV (cols: date, exdate, cp_flag, strike_price, best_bid, ...)
    expiry_date  : int YYYYMMDD
    min_strikes  : if >0, keep only dates with >= this many strikes that have a positive
                   bid on BOTH a call and a put for the expiry (a cheap liquidity screen).
                   Default 0 = every date carrying the expiry; the compare_* sweep will
                   skip any that can't actually be fit.
    Returns a sorted list of int trade dates strictly before the expiry (so T > 0).
    """
    import pandas as pd
    exp = int(expiry_date)
    cols = ["date", "exdate"] + (["cp_flag", "strike_price", "best_bid"] if min_strikes > 0 else [])
    df = pd.read_csv(csv_path, usecols=cols)
    df = df[(df["exdate"] == exp) & (df["date"] < exp)]          # date < exdate => T > 0
    if df.empty:
        print(f"[start_dates_for_expiry] no trade dates carry expiry {exp} in {csv_path}")
        return []
    if min_strikes > 0:
        keep = []
        for d, g in df.groupby("date"):
            c = set(g.loc[(g.cp_flag == "C") & (g.best_bid > 0), "strike_price"])
            p = set(g.loc[(g.cp_flag == "P") & (g.best_bid > 0), "strike_price"])
            if len(c & p) >= min_strikes:        # strikes with two-sided C+P quotes
                keep.append(int(d))
        dates = sorted(keep)
    else:
        dates = sorted(int(d) for d in df["date"].unique())
    print(f"[start_dates_for_expiry] {len(dates)} start date(s) for expiry {exp}"
          + (f"  (>= {min_strikes} two-sided strikes)" if min_strikes > 0 else ""))
    return dates

def expiry_dates_from_csv(
        csv_path,              # str: path to the raw SPX chain CSV (needs 'date','exdate' cols)
        sample=False,          # bool: True = uniformly subsample to sample_size expiries; False = keep ALL
        sample_size=20,        # int: how many expiries to keep when sample=True (ignored if sample=False)
        min_start_dates=0,     # int: keep only expiries with >= this many trade dates (0 = no screen)
        date_min=None,         # int YYYYMMDD or None: earliest expiry to include (None = no lower bound)
        date_max=None):        # int YYYYMMDD or None: latest expiry to include (None = no upper bound)
    """All DISTINCT expiry dates (exdate) in a raw SPX chain CSV, sorted ascending.

    The return value is ready to pass STRAIGHT into run_multi_expiry(expiry_dates=...).

    csv_path        : raw chain CSV (cols include 'date', 'exdate').
    sample          : if True, uniformly subsample sample_size expiries from the full
                      sorted list (evenly spaced, endpoints kept) -- mirrors the
                      compare_start_dates / run_multi_expiry sampling so a sweep stays
                      short.  If False, every distinct expiry is returned.
    sample_size     : number of expiries to keep when sample=True (ignored otherwise).
    min_start_dates : if >0, keep only expiries that carry at least this many distinct
                      trade dates strictly before the expiry (T>0) -- drops thinly-quoted
                      expiries not worth a sweep.  Same spirit as start_dates_for_expiry's
                      min_strikes screen (compare_* will still skip any unfittable date).
    date_min,date_max : optional inclusive int-YYYYMMDD bounds on the EXPIRY date, to
                      restrict the sweep to a window (e.g. date_min=20110101,
                      date_max=20111231).  None = no bound on that side.
    Returns a sorted list of int expiry dates (possibly empty).
    """
    import pandas as pd, numpy as np
    df = pd.read_csv(csv_path, usecols=["date", "exdate"])
    df = df[df["date"] < df["exdate"]]                      # keep only T>0 (date < exdate)
    if date_min is not None:
        df = df[df["exdate"] >= int(date_min)]
    if date_max is not None:
        df = df[df["exdate"] <= int(date_max)]
    if df.empty:
        print(f"[expiry_dates_from_csv] no expiries found in {csv_path}"
              + ("" if (date_min is None and date_max is None) else " within the given date window"))
        return []

    if min_start_dates > 0:
        counts = df.groupby("exdate")["date"].nunique()     # distinct start dates per expiry
        dates = sorted(int(e) for e in counts[counts >= min_start_dates].index)
    else:
        dates = sorted(int(e) for e in df["exdate"].unique())

    if not dates:
        print(f"[expiry_dates_from_csv] no expiries with >= {min_start_dates} start dates")
        return []

    n_full = len(dates)
    if sample and 0 < sample_size < n_full:
        idx = np.linspace(0, n_full - 1, sample_size).round().astype(int)
        idx = sorted(set(int(i) for i in idx))              # de-dup after rounding
        dates = [dates[i] for i in idx]
        print(f"[expiry_dates_from_csv] sample=True: using {len(dates)} of {n_full} distinct "
              f"expiries (uniformly spaced)"
              + (f", >= {min_start_dates} start dates" if min_start_dates > 0 else ""))
    else:
        print(f"[expiry_dates_from_csv] {len(dates)} distinct expiry date(s)"
              + (f"  (>= {min_start_dates} start dates)" if min_start_dates > 0 else ""))
    return dates


In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# run_multi_expiry — sweep the single-mode optimiser over MANY expiry dates, with
# every output filed neatly per expiry so results are easy to drop into a report.
# ══════════════════════════════════════════════════════════════════════════════
def run_multi_expiry(
        expiry_dates,          # list[int YYYYMMDD]: expiries to sweep (e.g. from expiry_dates_from_csv)
        csv_path,              # str: raw SPX chain CSV (per-date Quotes/ArbFree files auto-built if missing)
        optimize_mode="sig_nu",# "sig_nu" | "sig_only" | "sig_only_LKbar": which params the optimiser moves
                               #   sig_nu        = sigmas AND knot positions (highest ceiling, default)
                               #   sig_only      = sigmas only, knots frozen (robust, exact feasible start)
                               #   sig_only_LKbar= sig_only but the 2 boundary knots (L, Kbar) are also free
        R1_mult=1,             # int >=1: densify LEFT-wing knots by this factor before optimising (1 = no densify)
        R2_mult=1,             # int >=1: densify RIGHT-wing knots by this factor (1 = no densify)
        max_outer=100,         # int: max outer AL iterations per date (higher = more thorough, slower)
        normalize=True,        # bool: True = mesh-INVARIANT roughness (recommended); False = raw mesh-dependent sum
        obj=2,                 # 1 | 2: smoothness objective. 1 = jumps in 1/sigma^2; 2 = relative log(V/C'') jumps
                               #        (2 = scale-invariant, robust across datasets — recommended)
        unif_nus=False,        # bool: True = uniform-width knot insertion; False = proportional (mirror build_theta)
        batch_size=10,         # int: dates per summary image (one combined plot is emitted per batch)
        sample=False,          # bool: True = uniformly subsample start dates to sample_size before running
        sample_size=20,        # int: how many start dates to keep when sample=True
        min_strikes=0,         # int: liquidity screen passed to start_dates_for_expiry (0 = keep all)
        testing_root="testing",# str: root folder; each expiry gets testing_root/<expiry>_<timestamp>/
        **run_kwargs):         # forwarded to compare_start_dates -> run_and_plot -> optimize_augmented_lagrangian.
                               #   Common ones you may set: short_tenor_days (float, default 14), short_tenor_mode
                               #   ("sig_only"|None), tol_constraint (float), rho_init/rho_scale/rho_max, verbose (bool).
    """Optimise a whole LIST of expiries in one call.

    For EACH expiry date, independently:
      1. make a dedicated folder  testing_root/<expiry>_<timestamp>/  — EVERYTHING
         produced for that expiry (per-date diagnostics PNGs, fit_results CSVs,
         run logs, the per-batch summary images, the all-sample summary image, and
         a manifest CSV) is written INTO this folder via out_dir.  So each expiry is
         fully self-contained and trivial to grab for a report.
      2. pull that expiry's trade (start) dates from the raw chain
         (start_dates_for_expiry); `min_strikes` applies the same liquidity screen.
      3. call compare_start_dates with the given batch_size / sample / sample_size.
         compare_start_dates emits ONE summary image PER BATCH and — when there is
         more than one batch — ONE extra "ALL-SAMPLE" summary across every batch.
      4. write manifest_<expiry>.csv (one tidy row per fitted date) for easy tabling.

    Then it moves on to the next expiry and repeats.

    Parameters mirror compare_start_dates; `obj=2` (relative log-jump) + normalize=True
    are the robust defaults.  Extra **run_kwargs flow through to run_and_plot.

    Returns {expiry_date: (folder_path, results_list)} for downstream use.
    """
    import os, datetime
    os.makedirs(testing_root, exist_ok=True)
    all_out = {}

    for exp in expiry_dates:
        stamp  = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
        folder = os.path.join(testing_root, f"{exp}_{stamp}")
        os.makedirs(folder, exist_ok=True)
        print("=" * 80)
        print(f"###  EXPIRY {exp}   ->   {folder}")
        print("=" * 80)

        sds = start_dates_for_expiry(csv_path, exp, min_strikes=min_strikes)
        if not sds:
            print(f"[multi] no start dates found for expiry {exp}; skipping.")
            all_out[exp] = (folder, [])
            continue

        # All images/CSVs/logs for this expiry land in `folder` via out_dir.
        results = compare_start_dates(
            expiry_date=exp, start_dates=sds, csv_path=csv_path,
            optimize_mode=optimize_mode, R1_mult=R1_mult, R2_mult=R2_mult,
            unif_nus=unif_nus, max_outer=max_outer, normalize=normalize, obj=obj,
            batch_size=batch_size, sample=sample, sample_size=sample_size,
            out_dir=folder, emit_overall=True, **run_kwargs)

        # A compact manifest (one row per fitted date) for quick tabling in the report.
        try:
            keep = ["start_date", "S0", "T", "J_init", "J_final", "pct_improve",
                    "true_reduce", "S_final", "sqrt_S_final", "run_time"]
            pd.DataFrame([{k: r.get(k) for k in keep} for r in results]).to_csv(
                os.path.join(folder, f"manifest_{exp}.csv"), index=False)
        except Exception as _e:
            print(f"[multi] manifest write skipped for {exp}: {_e}")

        all_out[exp] = (folder, results)
        print(f"[multi] expiry {exp} done ({len(results)} dates) -> {folder}")

    # Final index of where everything went.
    print("\n[multi] ===== ALL EXPIRIES DONE =====")
    for exp, (folder, res) in all_out.items():
        print(f"  expiry {exp}: {len(res)} dates  ->  {folder}")
    return all_out

In [ ]:
# exps = expiry_dates_from_csv("SPX_opt_2011_2012.csv", sample=True, sample_size=30, min_start_dates=10)
# out = run_multi_expiry(
#     expiry_dates=exps,
#     csv_path="SPX_opt_2011_2012.csv",
#     optimize_mode="sig_nu", R1_mult=1, R2_mult=1, max_outer=100,
#     normalize=True, obj=2, unif_nus=False,
#     batch_size=10, sample=True, sample_size=10)

In [ ]:
# exps = expiry_dates_from_csv("SPX_opt_2024_2025.csv", sample=False, sample_size=30, min_start_dates=10)
# print(len(exps))

In [ ]:
exps = expiry_dates_from_csv("SPX_opt_2011_2012.csv", sample=False, sample_size=30, min_start_dates=10)
out = run_multi_expiry(
    expiry_dates=exps,
    csv_path="SPX_opt_2011_2012.csv",
    optimize_mode="sig_only", retry_mode="sig_nu", R1_mult=1, R2_mult=1, max_outer=100,
    normalize=True, obj=2, unif_nus=False,
    batch_size=10, sample=True, sample_size=10)

In [ ]:
# exps = expiry_dates_from_csv("SPX_opt_2011_2012.csv", sample=True, sample_size=30, min_start_dates=10)
# out = run_multi_expiry(
#     expiry_dates=[20120121, 20120622, 20120907],
#     csv_path="SPX_opt_2011_2012.csv",
#     optimize_mode="sig_only_LKbar", retry_mode="sig_only", R1_mult=1, R2_mult=1, max_outer=100,
#     normalize=True, obj=2, unif_nus=False,
#     batch_size=10, sample=True, sample_size=10)

In [ ]:
# out = run_multi_expiry(
#     expiry_dates=[20241025, 20241206, 20250728],
#     csv_path="SPX_opt_2024_2025.csv",
#     optimize_mode="sig_only_LKbar", retry_mode="sig_only", R1_mult=1, R2_mult=1, max_outer=100,
#     normalize=True, obj=2, unif_nus=False,
#     batch_size=10, sample=True, sample_size=10)

In [ ]:
# out = run_multi_expiry(
#     expiry_dates=[20251128],
#     csv_path="SPX_opt_2024_2025.csv",
#     optimize_mode="sig_only_LKbar", retry_mode="sig_only", R1_mult=1, R2_mult=1, max_outer=100,
#     normalize=True, obj=2, unif_nus=False,
#     batch_size=10, sample=True, sample_size=10)

In [ ]:
# out = run_multi_expiry(
#     expiry_dates=[20251128],
#     csv_path="SPX_opt_2024_2025.csv",
#     optimize_mode="sig_only_LKbar", retry_mode="sig_nu", R1_mult=1, R2_mult=1, max_outer=100,
#     normalize=True, obj=2, unif_nus=False,
#     batch_size=10, sample=True, sample_size=10)

In [ ]:
# out = run_multi_expiry(
#     expiry_dates=[20241025, 20241206, 20250728,20251128],
#     csv_path="SPX_opt_2024_2025.csv",
#     optimize_mode="sig_only_LKbar", retry_mode="sig_nu", R1_mult=1, R2_mult=1, max_outer=100,
#     normalize=True, obj=2, unif_nus=False,
#     batch_size=10, sample=True, sample_size=10)

In [ ]:
# exps = expiry_dates_from_csv("SPX_opt_2024_2025.csv", sample=True, sample_size=30, min_start_dates=10)
# out = run_multi_expiry(
#     expiry_dates=[20301220],
#     csv_path="SPX_opt_2024_2025.csv",
#     optimize_mode="sig_only", retry_mode="sig_nu", R1_mult=1, R2_mult=1, max_outer=100,
#     normalize=True, obj=2, unif_nus=False,
#     batch_size=10, sample=True, sample_size=10)

In [ ]:
# exps = expiry_dates_from_csv("SPX_opt_2011_2012.csv", sample=False, sample_size=30, min_start_dates=10)
# out = run_multi_expiry(
#     expiry_dates=exps,
#     csv_path="SPX_opt_2011_2012.csv",
#     optimize_mode="sig_only_LKbar", retry_mode="sig_nu", R1_mult=1, R2_mult=1, max_outer=100,
#     normalize=True, obj=2, unif_nus=False,
#     batch_size=10, sample=True, sample_size=10)

In [ ]:
# common = dict(
#     expiry_dates=[20110122],
#     csv_path="SPX_opt_2011_2012.csv",
#     R1_mult=1, R2_mult=1, max_outer=100,
#     normalize=True, obj=2, unif_nus=False,
#     batch_size=10, sample=False,
#     min_strikes=0,
#     max_knots_short=None,        # <-- keep ORIGINAL dense grid (no coarsening)
# )

# out_signu    = run_multi_expiry(optimize_mode="sig_nu",   **common)
# out_sigonly  = run_multi_expiry(optimize_mode="sig_only", **common)

In [ ]:
# run_and_plot redirects sys.stdout to a _Tee wrapper and only restores it in a
# late `finally`. A KeyboardInterrupt during the optimize phase aborts before that
# restore, leaving stdout stuck as the Tee (which silently swallows output).
# Each _Tee holds the previous stream at .ws[0], so walk ws[0] to unwrap them all.
_n = 0
while getattr(sys.stdout, "_is_runlog_tee", False):
    sys.stdout = sys.stdout.ws[0]
    _n += 1
while getattr(sys.stderr, "_is_runlog_tee", False):   # in case stderr ever got wrapped
    sys.stderr = sys.stderr.ws[0]

print(f"stdout restored after unwrapping {_n} tee layer(s): {type(sys.stdout).__name__}")